In [1]:
#!/usr/bin/env python3
"""
RSA Lag Analysis — All HC + LTC Regions (LHP / RHP / LLTC / RLTC), Band x Pair (Long Format)
----------------------------------------------------------------------------------------------
Phases:
  encoding   — enc_osc_f00..f17   (   0 to +1500 ms relative to word onset)
  pre_vocal  — ret1_osc_f00..f17  (-1500 to    0 ms relative to vocal onset)
  peri_vocal — ret2_osc_f00..f17  ( -500 to +1000 ms relative to vocal onset)
  post_vocal — ret3_osc_f00..f17  (    0 to +1500 ms relative to vocal onset)

For every trial, for every pair of clean recalls (i, j) with i < j,
for every band, for every phase:
  - Slice the 18-freq oscillatory IRASA vector to the band freq indices
  - Flatten each (N_channels x N_band_freqs) sub-matrix to a 1-D vector
  - RSA_r = Pearson r between the two flattened vectors

Output: LONG format, one row per pair x band x phase.
"""

import warnings
import traceback
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from scipy.spatial.distance import euclidean

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

EXPERIMENTS = ['DBOY1', 'EFRCourierReadOnly', 'EFRCourierOpenLoop']

INPUT_DIRS = {
    'DBOY1':               Path('./subject_results_HP_LTC_DBOY1'),
    'EFRCourierReadOnly':  Path('./subject_results_HP_LTC_EFRCourierReadOnly'),
    'EFRCourierOpenLoop':  Path('./subject_results_HP_LTC_EFRCourierOpenLoop'),
}

OUTPUT_DIR = Path('./rsa_lag_all_regions')
OUTPUT_DIR.mkdir(exist_ok=True)

REGIONS    = ['LHP', 'RHP', 'LLTC', 'RLTC']
HEMISPHERE = {'LHP': 'left', 'RHP': 'right', 'LLTC': 'left', 'RLTC': 'right'}

N_FREQS = 18

BANDS = {
    'theta': list(range(0, 4)),
    'alpha': list(range(4, 7)),
    'beta':  list(range(7, 12)),
    'gamma': list(range(12, 18)),
}
BAND_ORDER = ['theta', 'alpha', 'beta', 'gamma']

PHASES = {
    'encoding':   [f'enc_osc_f{i:02d}'    for i in range(N_FREQS)],
    'pre_vocal':  [f'ret1_osc_f{i:02d}'   for i in range(N_FREQS)],
    'peri_vocal': [f'ret2_osc_f{i:02d}'   for i in range(N_FREQS)],
    'post_vocal': [f'ret3_osc_f{i:02d}'   for i in range(N_FREQS)],
    'oldret':     [f'oldret_osc_f{i:02d}' for i in range(N_FREQS)],   # -750 to 0 ms
    'new':        [f'new_osc_f{i:02d}'    for i in range(N_FREQS)],   # -1000 to +500 ms
}
PHASE_ORDER = ['encoding', 'pre_vocal', 'peri_vocal', 'post_vocal', 'oldret', 'new']

WORD2VEC_PATH = Path('/home1/noaherz/word2vec/GoogleNews-vectors-negative300.bin.gz')

OUTPUT_COLS = [
    'subject', 'session', 'experiment', 'trial',
    'region', 'hemisphere',
    'phase', 'band', 'band_freq_indices',
    'output_pos_i', 'output_pos_j', 'output_lag',
    'word_i', 'word_j',
    'serial_pos_i', 'serial_pos_j', 'T_lag', 'SP_lag',
    'RSA_r', 'n_channels', 'semantic_sim',
]

# ============================================================================
# WORD2VEC LOADER + SIMILARITY
# ============================================================================

def load_word2vec(path: Path):
    if path is None or not path.exists():
        print(f"  Warning: Word2Vec model not found at {path}. semantic_sim will be NaN.")
        return None
    try:
        import gensim.models as gensim_models
        print(f"  Loading Word2Vec from {path} ...")
        model = gensim_models.KeyedVectors.load_word2vec_format(str(path), binary=True)
        try:
            vs = len(model)
        except TypeError:
            vs = len(model.vocab)
        print(f"  Word2Vec loaded - vocab: {vs:,}")
        return model
    except Exception as e:
        print(f"  Word2Vec load failed: {e}. semantic_sim will be NaN.")
        return None


def case_insensitive_similarity(word1: str, word2: str, model) -> Optional[float]:
    cases = [
        (word1.lower(), word2.lower()),
        (word1.lower(), word2.upper()),
        (word1.upper(), word2.lower()),
        (word1.upper(), word2.upper()),
    ]
    sims = []
    for w1, w2 in cases:
        try:
            sims.append(model.similarity(w1, w2))
        except KeyError:
            continue
    return float(max(sims)) if sims else None


def build_similarity_cache(words: set, model) -> dict:
    if model is None:
        return {}
    unique_words = sorted(w for w in words if isinstance(w, str))
    n = len(unique_words)
    print(f"    Building semsim cache: {n} words -> {n*(n-1)//2} pairs ...")
    cache = {}
    for i, w1 in enumerate(unique_words):
        for w2 in unique_words[i:]:
            key = frozenset({w1, w2})
            if key not in cache:
                sim = case_insensitive_similarity(w1, w2, model)
                cache[key] = sim if sim is not None else np.nan
    return cache


# ============================================================================
# VECTOR + STATISTICS HELPERS
# ============================================================================

def build_band_vector(recall_rows: pd.DataFrame,
                      channel_index,
                      phase_cols: list,
                      band_freq_indices: list) -> np.ndarray:
    ch_df = (
        recall_rows
        .drop_duplicates(subset='pair_idx')
        .set_index('pair_idx')
        .reindex(channel_index)
    )
    band_cols = [phase_cols[i] for i in band_freq_indices]
    mat = ch_df.reindex(columns=band_cols).values
    return mat.flatten()


def safe_pearsonr(v1: np.ndarray, v2: np.ndarray) -> float:
    if len(v1) != len(v2):
        return np.nan
    mask = np.isfinite(v1) & np.isfinite(v2)
    if mask.sum() < 3:
        return np.nan
    r, _ = pearsonr(v1[mask], v2[mask])
    return float(r)


def safe_euclidean(x1, z1, x2, z2) -> float:
    if any(not np.isfinite(v) for v in (x1, z1, x2, z2)):
        return np.nan
    return float(euclidean([x1, z1], [x2, z2]))


def extract_scalar(series: pd.Series, field: str, context: str):
    unique_vals = series.dropna().unique()
    if len(unique_vals) > 1:
        warnings.warn(
            f"[{context}] '{field}' has {len(unique_vals)} distinct values "
            f"({unique_vals[:3]}...). Taking first.")
    return series.iloc[0]


# ============================================================================
# TRIAL-LEVEL PROCESSING
# ============================================================================

def process_trial_region(trial_df: pd.DataFrame,
                         region: str,
                         sim_cache: dict) -> List[Dict]:
    rows = []

    output_positions = sorted(
        trial_df['recall_output_position'].unique(),
        key=lambda x: int(x),
    )
    if len(output_positions) < 2:
        return rows

    channel_index = sorted(trial_df['pair_idx'].unique(), key=int)
    sample_row    = trial_df.iloc[0]

    pos_data: Dict[int, Dict] = {}

    for op in output_positions:
        op_rows = trial_df[trial_df['recall_output_position'] == op]
        ctx = (f"subj={sample_row['subject']} sess={sample_row['session']} "
               f"trial={sample_row['trial']} region={region} op={op}")

        word       = extract_scalar(op_rows['recalled_word'],   'recalled_word',   ctx)
        serial_pos = extract_scalar(op_rows['serial_position'], 'serial_position', ctx)
        store_x    = extract_scalar(op_rows['store_x'],         'store_x',         ctx)
        store_z    = extract_scalar(op_rows['store_z'],         'store_z',         ctx)
        n_ch       = op_rows['pair_idx'].nunique()

        phase_band_vectors = {
            phase_name: {
                band_name: build_band_vector(
                    op_rows, channel_index, phase_cols, freq_idx
                )
                for band_name, freq_idx in BANDS.items()
            }
            for phase_name, phase_cols in PHASES.items()
        }

        pos_data[op] = {
            'phase_band_vectors': phase_band_vectors,
            'word':               word,
            'serial_pos':         serial_pos,
            'store_x':            store_x,
            'store_z':            store_z,
            'n_channels':         n_ch,
        }

    subject    = sample_row['subject']
    session    = sample_row['session']
    experiment = sample_row['experiment']
    trial      = sample_row['trial']

    for idx_i, op_i in enumerate(output_positions):
        for op_j in output_positions[idx_i + 1:]:
            d_i = pos_data[op_i]
            d_j = pos_data[op_j]

            output_lag = int(op_j) - int(op_i)
            T_lag      = abs(int(d_i['serial_pos']) - int(d_j['serial_pos']))
            SP_lag     = safe_euclidean(d_i['store_x'], d_i['store_z'],
                                        d_j['store_x'], d_j['store_z'])
            n_channels = min(d_i['n_channels'], d_j['n_channels'])

            w_i = str(d_i['word']).lower() if pd.notna(d_i['word']) else None
            w_j = str(d_j['word']).lower() if pd.notna(d_j['word']) else None
            sem_sim = (
                sim_cache.get(frozenset({w_i, w_j}), np.nan)
                if (w_i and w_j and sim_cache)
                else np.nan
            )

            for phase_name in PHASE_ORDER:
                for band_name in BAND_ORDER:
                    freq_idx = BANDS[band_name]
                    v_i      = d_i['phase_band_vectors'][phase_name][band_name]
                    v_j      = d_j['phase_band_vectors'][phase_name][band_name]
                    rsa_r    = safe_pearsonr(v_i, v_j)

                    rows.append({
                        'subject':           subject,
                        'session':           session,
                        'experiment':        experiment,
                        'trial':             trial,
                        'region':            region,
                        'hemisphere':        HEMISPHERE[region],
                        'phase':             phase_name,
                        'band':              band_name,
                        'band_freq_indices': str(freq_idx),
                        'output_pos_i':      op_i,
                        'output_pos_j':      op_j,
                        'output_lag':        output_lag,
                        'word_i':            d_i['word'],
                        'word_j':            d_j['word'],
                        'serial_pos_i':      d_i['serial_pos'],
                        'serial_pos_j':      d_j['serial_pos'],
                        'T_lag':             T_lag,
                        'SP_lag':            SP_lag,
                        'RSA_r':             rsa_r,
                        'n_channels':        n_channels,
                        'semantic_sim':      sem_sim,
                    })

    return rows


# ============================================================================
# PER-EXPERIMENT RUNNER
# ============================================================================

def run_experiment(exp: str, w2v_model):
    print(f"\n{'='*70}")
    print(f"EXPERIMENT: {exp}")
    print(f"{'='*70}")

    input_dir = INPUT_DIRS.get(exp)
    if input_dir is None:
        print(f"  No INPUT_DIR configured for '{exp}'.")
        return

    input_path = input_dir / f"ALL_SUBJECTS_{exp}_irasa_HP_LTC_wide.csv"
    if not input_path.exists():
        print(f"  Master CSV not found: {input_path}")
        return

    print(f"  Loading {input_path} ...")
    df = pd.read_csv(input_path)
    print(f"  Loaded {len(df):,} rows | "
          f"{df['subject'].nunique()} subjects | "
          f"{df['session'].nunique()} sessions")

    for phase_name, phase_cols in PHASES.items():
        missing = [c for c in phase_cols if c not in df.columns]
        if missing:
            print(f"  Warning - Phase '{phase_name}': {len(missing)}/18 columns absent "
                  f"(e.g. {missing[0]}). RSA_r will be NaN for this phase.")

    df = df[df['region'].isin(REGIONS)].copy()
    print(f"  After region filter: {len(df):,} rows")
    if df.empty:
        print("  No data found for target regions.")
        return

    df['pair_idx'] = df['pair_idx'].astype(int)

    all_words_lower = set(
        df['recalled_word'].dropna().astype(str).str.lower().unique()
    )
    sim_cache = build_similarity_cache(all_words_lower, w2v_model)

    all_region_dfs = []

    for region in REGIONS:
        print(f"\n  {'─'*60}")
        print(f"  Region: {region}")

        region_df = df[df['region'] == region].copy()
        if region_df.empty:
            print(f"  No rows for region {region} - skipping")
            continue

        print(f"  Rows: {len(region_df):,}")

        all_rows = []
        groups   = region_df.groupby(['subject', 'session', 'trial'])
        n_groups = len(groups)

        for g_idx, ((subj, sess, trial), trial_df) in enumerate(groups):
            if g_idx % 200 == 0:
                print(f"    Processing trial group {g_idx}/{n_groups} ...")
            try:
                rows = process_trial_region(trial_df, region, sim_cache)
                all_rows.extend(rows)
            except Exception as e:
                print(f"    FAILED [{subj} sess={sess} trial={trial}]: {e}")
                traceback.print_exc()
                continue

        if not all_rows:
            print(f"  No pairs generated for region {region}")
            continue

        result_df = pd.DataFrame(all_rows, columns=OUTPUT_COLS)
        all_region_dfs.append(result_df)

        for phase_name in PHASE_ORDER:
            phase_df = result_df[result_df['phase'] == phase_name]
            for band_name in BAND_ORDER:
                band_df  = phase_df[phase_df['band'] == band_name]
                out_path = (OUTPUT_DIR /
                            f"ALL_SUBJECTS_{exp}_{region}_{phase_name}_{band_name}_rsa_lag.csv")
                band_df.to_csv(out_path, index=False)
                print(f"    {region}/{phase_name}/{band_name}: {len(band_df):,} pairs | "
                      f"RSA NaN={band_df['RSA_r'].isna().mean()*100:.1f}% | "
                      f"-> {out_path.name}")

        for phase_name in PHASE_ORDER:
            phase_df = result_df[result_df['phase'] == phase_name]
            reg_path = (OUTPUT_DIR /
                        f"ALL_SUBJECTS_{exp}_{region}_{phase_name}_allbands_rsa_lag.csv")
            phase_df.to_csv(reg_path, index=False)
            print(f"  {region}/{phase_name} all-bands -> {reg_path.name}")

    if all_region_dfs:
        combined  = pd.concat(all_region_dfs, ignore_index=True)
        comb_path = OUTPUT_DIR / f"ALL_SUBJECTS_{exp}_allregions_allbands_rsa_lag.csv"
        combined.to_csv(comb_path, index=False)
        print(f"\n  Combined all-regions all-bands -> {comb_path.name}")
        print(f"    Total rows : {len(combined):,}")
        print(f"    Regions    : {sorted(combined['region'].unique().tolist())}")
        print(f"    Phases     : {combined['phase'].unique().tolist()}")
        print(f"    Bands      : {combined['band'].unique().tolist()}")


# ============================================================================
# MAIN
# ============================================================================

if __name__ == '__main__':
    w2v_model = load_word2vec(WORD2VEC_PATH)

    for exp in EXPERIMENTS:
        run_experiment(exp, w2v_model)

    print(f"\n{'='*70}")
    print("ALL EXPERIMENTS COMPLETE")
    print(f"{'='*70}")

  Loading Word2Vec from /home1/noaherz/word2vec/GoogleNews-vectors-negative300.bin.gz ...
  Word2Vec loaded - vocab: 3,000,000

EXPERIMENT: DBOY1
  Loading subject_results_HP_LTC_DBOY1/ALL_SUBJECTS_DBOY1_irasa_HP_LTC_wide.csv ...
  Loaded 29,088 rows | 40 subjects | 8 sessions
  After region filter: 29,088 rows
    Building semsim cache: 234 words -> 27261 pairs ...

  ────────────────────────────────────────────────────────────
  Region: LHP
  Rows: 3,609
    Processing trial group 0/179 ...
    LHP/encoding/theta: 1,814 pairs | RSA NaN=0.0% | -> ALL_SUBJECTS_DBOY1_LHP_encoding_theta_rsa_lag.csv
    LHP/encoding/alpha: 1,814 pairs | RSA NaN=0.0% | -> ALL_SUBJECTS_DBOY1_LHP_encoding_alpha_rsa_lag.csv
    LHP/encoding/beta: 1,814 pairs | RSA NaN=0.0% | -> ALL_SUBJECTS_DBOY1_LHP_encoding_beta_rsa_lag.csv
    LHP/encoding/gamma: 1,814 pairs | RSA NaN=0.0% | -> ALL_SUBJECTS_DBOY1_LHP_encoding_gamma_rsa_lag.csv
    LHP/pre_vocal/theta: 1,814 pairs | RSA NaN=0.0% | -> ALL_SUBJECTS_DBOY1_LHP_

In [ ]:
#!/usr/bin/env python3
"""
LMM Analysis: T_lag / SP_lag → RSA_r
All 4 regions (LHP / RHP / LLTC / RLTC), separate model sets per region
Retrieval phase only.  Single experiment — no experiment factor.
----------------------------------------------------------------------
RSA_r is the OUTCOME.  T_lag and SP_lag are the PREDICTORS OF INTEREST,
tested in separate model sets.

Input : ./rsa_lag_all_regions/ALL_SUBJECTS_{exp}_allregions_allbands_rsa_lag.csv
        (output of rsa_lag_all_regions.py)

For each REGION in [LHP, RHP, LLTC, RLTC]:
  For each PREDICTOR in [T_lag, SP_lag]:
    For each BAND in [theta, alpha, beta, gamma]:

      Model 1 (bare):
          RSA_r ~ predictor                              + (1|subj/sess)

      Model 2 (controlled):
          RSA_r ~ predictor + cross_lag + output_lag
                            + [semantic_sim if available]
                                                         + (1|subj/sess)

      (No hemisphere factor — hemisphere is fixed within each region run.)

Outputs (per region):
  ./rsa_lag_all_regions/LMM_{region}_{predictor}_{band}_results.csv
  ./rsa_lag_all_regions/LMM_{region}_{predictor}_{band}_results.txt

Master:
  ./rsa_lag_all_regions/LMM_ALL_results.csv

Plots:
  ./rsa_lag_all_regions/plots/forest_{region}_{predictor}_{band}.png
  ./rsa_lag_all_regions/plots/interaction_{region}_{predictor}_{band}.png
  ./rsa_lag_all_regions/plots/summary_heatmap_{region}.png
  ./rsa_lag_all_regions/plots/beta_bars_{region}.png
"""

import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import pearsonr, ttest_1samp
from statsmodels.regression.mixed_linear_model import MixedLM
from statsmodels.stats.multitest import fdrcorrection

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION  — edit these to switch experiment / paths
# ============================================================================

EXPERIMENT  = 'DBOY1'            # single experiment — no experiment factor
PHASE       = 'oldret'        # only retrieval rows are kept PHASE = 'pre_vocal'   # -1500 to 0 ms, vocal-locked PHASE = 'peri_vocal'  # -500 to +1000 ms, vocal-locked PHASE = 'post_vocal'  # 0 to +1500 ms, vocal-locked

INPUT_DIR   = Path('./rsa_lag_all_regions')
OUTPUT_DIR  = Path('./rsa_lag_all_regions')
PLOT_DIR    = OUTPUT_DIR / 'plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

OUTCOME     = 'RSA_r'
PREDICTORS  = ['SP_lag','T_lag']
BANDS       = ['alpha','gamma'] #'alpha', 'beta', 'gamma']
REGIONS     = ['LHP','RHP','LLTC','RLTC'] #  'LLTC', 'RLTC'

CROSS_COVARIATE = {
    'T_lag':  'SP_lag',
    'SP_lag': 'T_lag',
}
PRED_LABELS = {
    'T_lag':  'Temporal Lag (T_lag)',
    'SP_lag': 'Spatial Lag (SP_lag)',
}
MODEL_LABELS = {
    'Model1': 'Bare',
    'Model2': 'Controlled',
}

# ---- Palette (light background) ---------------------------------------------
BG_COLOR    = 'white'
AX_COLOR    = '#F7F7F7'
GRID_COLOR  = '#DDDDDD'
TEXT_COLOR  = '#222222'
SPINE_COLOR = '#AAAAAA'

REGION_COLORS = {
    'LHP':  '#1A3A6B',
    'RHP':  '#8B1A1A',
    'LLTC': '#1A6B3A',
    'RLTC': '#6B5A1A',
}
BAND_COLORS = {
    'theta': '#4575B4',
    'alpha': '#74ADD1',
    'beta':  '#F46D43',
    'gamma': '#D73027',
}

# ============================================================================
# LMM FITTING
# ============================================================================

def fit_lmm(df: pd.DataFrame,
            pred_cols: List[str],
            label: str,
            formula_rhs: Optional[str] = None) -> Tuple[Optional[object], int]:
    """
    Fit RSA_r ~ formula_rhs with nested random effects (1|subject/session).
    No experiment dummy — single experiment only.
    """
    df = df.copy()
    df['subj_sess'] = (df['subject'].astype(str)
                       + '_' + df['session'].astype(str))

    real_cols = [c for c in pred_cols if c in df.columns]
    keep      = [OUTCOME] + real_cols + ['subject', 'subj_sess']
    df        = df[keep].dropna()

    if len(df) < 20:
        print(f"    [{label}] Too few rows ({len(df)}) — skipping")
        return None, 0

    rhs     = formula_rhs if formula_rhs else ' + '.join(pred_cols)
    formula = f"{OUTCOME} ~ {rhs}"
    print(f"    [{label}] {formula}  |  N={len(df):,}")

    model = MixedLM.from_formula(
        formula,
        data       = df,
        groups     = df['subject'],
        vc_formula = {'subj_sess': '0 + C(subj_sess)'},
    )

    result = None
    for method in ['lbfgs', 'nm', 'powell']:
        try:
            result = model.fit(reml=True, method=method)
            if np.isfinite(result.llf):
                print(f"    [{label}] optimizer={method}  "
                      f"converged={getattr(result, 'converged', None)}  "
                      f"llf={result.llf:.4f}  AIC={result.aic:.4f}")
                break
            else:
                print(f"    [{label}] llf=NaN with {method}, trying next …")
        except Exception as e:
            print(f"    [{label}] {method} failed: {e}")
            result = None

    if result is None or not np.isfinite(result.llf):
        print(f"    [{label}] WARNING: fit unsuccessful.")
    return result, len(df)


# ============================================================================
# RESULT EXTRACTION
# ============================================================================

def extract_rows(result,
                 pred_display: Dict[str, str],
                 model_label: str,
                 predictor: str,
                 band: str,
                 region: str) -> pd.DataFrame:
    if result is None:
        return pd.DataFrame()
    rows = []
    for col, name in pred_display.items():
        matched = col if col in result.params.index else None
        if matched is None:
            hits    = [k for k in result.params.index
                       if col.lower() in k.lower()]
            matched = hits[0] if hits else None
        if matched is None:
            print(f"    WARNING: '{col}' not found in params — skipping")
            continue
        rows.append({
            'experiment':            EXPERIMENT,
            'phase':                 PHASE,
            'region':                region,
            'predictor_of_interest': predictor,
            'band':                  band,
            'model':                 model_label,
            'term':                  name,
            'col':                   col,
            'beta':                  result.params[matched],
            'se':                    result.bse[matched],
            'z':                     result.tvalues[matched],
            'p_raw':                 result.pvalues[matched],
            'llf':                   result.llf,
            'aic':                   result.aic,
            'nobs':                  int(result.nobs),
        })
    return pd.DataFrame(rows)


def apply_fdr_within_model(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    _, df['p_fdr'] = fdrcorrection(df['p_raw'].values)
    return df


# ============================================================================
# TEXT FORMATTING
# ============================================================================

def sig_stars(p: float) -> str:
    return ('***' if p < 0.001 else
            '**'  if p < 0.01  else
            '*'   if p < 0.05  else
            '†'   if p < 0.10  else '')


def format_block(title: str, rows_df: pd.DataFrame) -> str:
    sep  = '=' * 92
    sep2 = '-' * 92
    hdr  = (f"{'Term':<40} {'β':>8} {'SE':>8} {'z':>8} "
            f"{'p_raw':>10} {'p_fdr':>10} {'AIC':>10} {'N':>8} {'sig':>5}")
    lines = [sep, title, sep2, hdr, sep2]
    for _, row in rows_df.iterrows():
        aic_s = (f"{row['aic']:>10.2f}"
                 if pd.notna(row.get('aic')) else '       NaN')
        p_fdr = row.get('p_fdr', np.nan)
        lines.append(
            f"{row['term']:<40} {row['beta']:>8.4f} {row['se']:>8.4f} "
            f"{row['z']:>8.3f} {row['p_raw']:>10.4f} "
            f"{p_fdr:>10.4f} {aic_s} {int(row['nobs']):>8,} "
            f"{sig_stars(p_fdr):>5}"
        )
    lines += [sep2,
              'FDR: BH correction within each model across terms.',
              '† p<.10  * p<.05  ** p<.01  *** p<.001',
              f'Outcome = RSA_r.  Phase = {PHASE}.  Experiment = {EXPERIMENT}.',
              'All continuous predictors on raw scale.',
              sep]
    return '\n'.join(lines)


# ============================================================================
# PLOTTING
# ============================================================================

def _style_ax(ax):
    ax.set_facecolor(AX_COLOR)
    ax.tick_params(colors=TEXT_COLOR, labelsize=8.5)
    ax.xaxis.label.set_color(TEXT_COLOR)
    ax.yaxis.label.set_color(TEXT_COLOR)
    ax.title.set_color(TEXT_COLOR)
    ax.grid(True, color=GRID_COLOR, lw=0.6, zorder=0)
    for spine in ax.spines.values():
        spine.set_edgecolor(SPINE_COLOR)
        spine.set_linewidth(0.8)


def plot_forest(all_results: pd.DataFrame,
                region: str,
                predictor: str,
                band: str,
                save_path: Path):
    """
    Forest plot: beta of the predictor of interest ± 95%CI across
    models (Model1 / Model2) for one region × band.
    """
    df = all_results[
        (all_results['region']                == region) &
        (all_results['predictor_of_interest'] == predictor) &
        (all_results['band']                  == band) &
        (all_results['col']                   == predictor)
    ].copy()

    if df.empty:
        print(f"    No rows for forest plot {region} {predictor} {band}")
        return

    df['ci_lo'] = df['beta'] - 1.96 * df['se']
    df['ci_hi'] = df['beta'] + 1.96 * df['se']

    models = [m for m in ['Model1', 'Model2'] if m in df['model'].values]
    color  = REGION_COLORS.get(region, '#444444')

    fig, ax = plt.subplots(figsize=(9, max(3, len(models) * 1.4 + 1)))
    fig.patch.set_facecolor(BG_COLOR)
    _style_ax(ax)

    y_pos, yticks, ylabels = 0, [], []

    for model in models:
        row = df[df['model'] == model]
        if row.empty:
            continue
        row  = row.iloc[0]
        xerr = [[row['beta'] - row['ci_lo']], [row['ci_hi'] - row['beta']]]
        ax.errorbar(row['beta'], y_pos, xerr=xerr,
                    fmt='o', color=color, ecolor=color,
                    elinewidth=1.5, capsize=4, capthick=1.5,
                    markersize=7, zorder=3)
        p_show = row.get('p_fdr', row['p_raw'])
        stars  = sig_stars(p_show)
        if stars:
            offset = abs(row['ci_hi'] - row['beta']) * 0.15 + 0.001
            ax.text(row['ci_hi'] + offset, y_pos, stars,
                    color=color, va='center', fontsize=9, fontweight='bold')
        yticks.append(y_pos)
        ylabels.append(MODEL_LABELS.get(model, model))
        y_pos -= 1

    ax.axvline(0, color=SPINE_COLOR, lw=1.2, ls='--', zorder=1)
    ax.set_yticks(yticks)
    ax.set_yticklabels(ylabels, fontsize=9)
    ax.set_xlabel(f'β  ({PRED_LABELS[predictor]})', fontsize=10)
    ax.set_title(
        f"{region}  |  RSA_r ~ {PRED_LABELS[predictor]}  |  "
        f"{band.capitalize()} band  [{EXPERIMENT} / {PHASE}]",
        fontsize=11, fontweight='bold', pad=8)

    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"    ✓ Forest → {save_path.name}")


def _subject_slopes(hd: pd.DataFrame, predictor: str) -> pd.DataFrame:
    rows = []
    for subj, sg in hd.groupby('subject'):
        sg = sg.dropna(subset=[predictor, OUTCOME])
        if len(sg) < 3:
            continue
        x = sg[predictor].values.astype(float)
        y = sg[OUTCOME].values.astype(float)
        if x.std() == 0:
            continue
        m, b = np.polyfit(x, y, 1)
        rows.append({'subject': subj, 'slope': m,
                     'intercept': b, 'n_pairs': len(sg)})
    return pd.DataFrame(rows)


def _subject_zscore_rsa(hd: pd.DataFrame) -> pd.DataFrame:
    hd = hd.copy()
    def zscore(x):
        s = x.std()
        return (x - x.mean()) / s if s > 0 else x - x.mean()
    hd['RSA_r_z'] = hd.groupby('subject')[OUTCOME].transform(zscore)
    return hd


def plot_interaction(region: str,
                     predictor: str,
                     band: str,
                     raw_df: pd.DataFrame,
                     save_path: Path):
    """
    Two-panel figure for one region:
    TOP    — spaghetti + group mean (z-scored RSA_r within subject)
    BOTTOM — subject slope dot plot (OLS slope per subject)
    """
    sub_df = raw_df[
        raw_df['band'] == band
    ].copy().dropna(subset=[predictor, OUTCOME])

    if sub_df.empty:
        return

    color       = REGION_COLORS.get(region, '#444444')
    is_discrete = (predictor == 'T_lag')

    fig, (ax_sp, ax_sl) = plt.subplots(2, 1, figsize=(10, 10))
    fig.patch.set_facecolor(BG_COLOR)
    _style_ax(ax_sp)
    _style_ax(ax_sl)

    n_subj  = sub_df['subject'].nunique()
    n_pairs = len(sub_df)

    x_all = sub_df[predictor].values.astype(float)
    y_all = sub_df[OUTCOME].values.astype(float)
    r_val, p_val = pearsonr(x_all, y_all)
    p_str = f'p={p_val:.3f}' if p_val >= 0.001 else 'p<0.001'

    hd = _subject_zscore_rsa(sub_df)

    if is_discrete:
        MAX_VALS = 20
        top_vals = sorted(
            hd[predictor].value_counts().nlargest(MAX_VALS).index.tolist())
        x_grid = np.array(top_vals, dtype=float)
    else:
        N_BINS     = 12
        hd['_bin'] = pd.cut(hd[predictor], bins=N_BINS)
        bin_mids   = (hd.groupby('_bin', observed=True)[predictor]
                       .mean().dropna())
        x_grid     = bin_mids.values

    # ── TOP: Spaghetti ────────────────────────────────────────────────────────
    subj_lines = []
    for subj, sg in hd.groupby('subject'):
        sg = sg.dropna(subset=[predictor, 'RSA_r_z'])
        if len(sg) < 3:
            continue
        if is_discrete:
            pts = (sg.groupby(predictor)['RSA_r_z']
                     .mean().reindex(top_vals))
        else:
            pts = (sg.groupby('_bin', observed=True)['RSA_r_z'].mean())
            pts.index = pts.index.map(
                lambda b: b.mid if hasattr(b, 'mid') else np.nan)
            pts = pts.dropna()
        pts = pts.dropna()
        if len(pts) < 2:
            continue
        subj_lines.append(pts.values)
        ax_sp.plot(x_grid[:len(pts.values)], pts.values,
                   color=color, alpha=0.15, lw=0.9, zorder=2)

    if subj_lines:
        max_len  = max(len(s) for s in subj_lines)
        padded   = np.array([
            np.pad(s.astype(float), (0, max_len - len(s)),
                   constant_values=np.nan)
            for s in subj_lines
        ])
        grp_mean = np.nanmean(padded, axis=0)
        grp_sem  = (np.nanstd(padded, axis=0)
                    / np.sqrt((~np.isnan(padded)).sum(axis=0)))
        xg = x_grid[:max_len]

        ax_sp.fill_between(xg, grp_mean - grp_sem, grp_mean + grp_sem,
                           color=color, alpha=0.20, zorder=3)
        ax_sp.plot(xg, grp_mean, color=color, lw=2.5, zorder=4,
                   label=f'Group mean ± SEM  (n={n_subj} subj)')

        valid = np.isfinite(grp_mean) & np.isfinite(xg)
        if valid.sum() >= 2:
            m_z, b_z = np.polyfit(xg[valid], grp_mean[valid], 1)
            ax_sp.plot(xg[valid], m_z * xg[valid] + b_z,
                       color=TEXT_COLOR, lw=1.5, ls='--', alpha=0.6,
                       zorder=5, label=f'OLS  r={r_val:.3f} {p_str}')

    ax_sp.axhline(0, color=SPINE_COLOR, lw=1, ls=':', zorder=1)
    if is_discrete:
        ax_sp.set_xticks(x_grid)
        ax_sp.set_xticklabels(
            [str(int(v)) for v in x_grid], fontsize=7,
            rotation=45 if len(x_grid) > 12 else 0)
    ax_sp.set_xlabel(PRED_LABELS[predictor], fontsize=9)
    ax_sp.set_ylabel('RSA_r  (z-scored within subject)', fontsize=9)
    ax_sp.set_title(
        f'{region}  |  {band.capitalize()} band  |  '
        f'{EXPERIMENT} / {PHASE}\n'
        f'Spaghetti: each line = 1 subject  (n={n_subj}, {n_pairs:,} pairs)',
        fontsize=10, fontweight='bold')
    ax_sp.legend(facecolor='white', edgecolor=SPINE_COLOR, fontsize=8)

    # ── BOTTOM: Subject slope dot plot ────────────────────────────────────────
    slopes_df = _subject_slopes(hd, predictor)

    if slopes_df.empty:
        ax_sl.set_visible(False)
    else:
        slopes     = slopes_df['slope'].values
        n_sl       = len(slopes)
        mean_slope = slopes.mean()
        sem_slope  = slopes.std() / np.sqrt(n_sl)
        ci95       = 1.96 * sem_slope
        n_neg      = (slopes < 0).sum()
        n_pos      = (slopes > 0).sum()

        order      = np.argsort(slopes)
        y_pos_sl   = np.arange(n_sl)
        dot_colors = [REGION_COLORS.get(region, '#444444')
                      if s >= 0 else '#888888'
                      for s in slopes[order]]

        ax_sl.scatter(slopes[order], y_pos_sl,
                      c=dot_colors, s=40, zorder=3,
                      edgecolors='white', linewidths=0.4)
        for yi, si, dc in zip(y_pos_sl, slopes[order], dot_colors):
            ax_sl.plot([0, si], [yi, yi],
                       color=dc, alpha=0.35, lw=0.8, zorder=2)

        ax_sl.axvline(0, color=SPINE_COLOR, lw=1.2, ls='--', zorder=1)
        ax_sl.axvspan(mean_slope - ci95, mean_slope + ci95,
                      color=color, alpha=0.12, zorder=0)
        ax_sl.axvline(mean_slope, color=color, lw=2.5, zorder=4,
                      label=f'Mean β={mean_slope:.4f} ± {ci95:.4f} (95%CI)')

        t_stat, p_1samp = ttest_1samp(slopes, 0)
        p1_str = f'p={p_1samp:.3f}' if p_1samp >= 0.001 else 'p<0.001'
        stars  = sig_stars(p_1samp)
        ax_sl.text(mean_slope, n_sl + 0.3,
                   f'{stars}  {p1_str}' if stars else p1_str,
                   ha='center', va='bottom',
                   color=color, fontsize=9, fontweight='bold')

        ax_sl.set_yticks([])
        ax_sl.set_xlabel(
            f'Subject-level OLS slope  (RSA_r ~ {predictor})', fontsize=9)
        ax_sl.set_title(
            f'{region}  |  {band.capitalize()} band\n'
            f'Each dot = 1 subject slope  '
            f'({n_neg}/{n_sl} negative, {n_pos}/{n_sl} positive)',
            fontsize=10, fontweight='bold')
        ax_sl.legend(facecolor='white', edgecolor=SPINE_COLOR, fontsize=8)
        pct_neg = n_neg / n_sl * 100
        ax_sl.text(0.02, 0.04,
                   f'{pct_neg:.0f}% of subjects show negative slope',
                   transform=ax_sl.transAxes,
                   fontsize=8, color=TEXT_COLOR, style='italic')

    fig.suptitle(
        f"RSA_r ~ {PRED_LABELS[predictor]}  |  {region}  |  "
        f"{band.capitalize()} band\n"
        f"Top: z-scored spaghetti + group mean  |  "
        f"Bottom: per-subject slopes",
        fontsize=12, fontweight='bold', y=1.01, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"    ✓ Interaction plot → {save_path.name}")


def plot_summary_heatmap(all_results: pd.DataFrame,
                         region: str,
                         save_path: Path):
    """
    Heatmap of predictor beta (Model1) across predictor × band for one region.
    """
    pivot_rows = []
    for pred in PREDICTORS:
        for band in BANDS:
            sub = all_results[
                (all_results['region']                == region) &
                (all_results['predictor_of_interest'] == pred) &
                (all_results['band']                  == band) &
                (all_results['model']                 == 'Model1') &
                (all_results['col']                   == pred)
            ]
            if sub.empty:
                beta, stars = np.nan, ''
            else:
                r     = sub.iloc[0]
                beta  = r['beta']
                stars = sig_stars(r.get('p_fdr', r['p_raw']))
            pivot_rows.append({'predictor': PRED_LABELS[pred],
                                'band': band, 'beta': beta, 'stars': stars})

    piv       = pd.DataFrame(pivot_rows)
    beta_mat  = piv.pivot(index='predictor', columns='band', values='beta')[BANDS]
    stars_mat = piv.pivot(index='predictor', columns='band', values='stars')[BANDS]

    fig, ax = plt.subplots(figsize=(9, 3.0))
    fig.patch.set_facecolor(BG_COLOR)
    vmax = np.nanmax(np.abs(beta_mat.values)) or 1.0
    im   = ax.imshow(beta_mat.values.astype(float),
                     aspect='auto', cmap='RdBu_r',
                     vmin=-vmax, vmax=vmax)

    for i in range(beta_mat.shape[0]):
        for j in range(beta_mat.shape[1]):
            val  = beta_mat.values[i, j]
            star = stars_mat.values[i, j]
            if not np.isnan(val):
                cell_norm = (val + vmax) / (2 * vmax)
                txt_color = ('white'
                             if (cell_norm < 0.35 or cell_norm > 0.65)
                             else TEXT_COLOR)
                ax.text(j, i, f"{val:.3f}\n{star}",
                        ha='center', va='center',
                        color=txt_color, fontsize=9, fontweight='bold')

    ax.set_xticks(range(len(BANDS)))
    ax.set_xticklabels([b.capitalize() for b in BANDS], fontsize=10)
    ax.set_yticks(range(len(beta_mat.index)))
    ax.set_yticklabels(beta_mat.index.tolist(), fontsize=9)
    ax.tick_params(colors=TEXT_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(SPINE_COLOR)

    cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label('β  (predictor → RSA_r)', fontsize=9, color=TEXT_COLOR)
    cbar.ax.yaxis.set_tick_params(color=TEXT_COLOR)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COLOR)

    ax.set_title(
        f'{region}  |  RSA_r ~ predictor  |  β (Model 1)  '
        f'across Predictors × Bands  [{EXPERIMENT} / {PHASE}]',
        fontsize=10, fontweight='bold', pad=8, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"  ✓ Heatmap → {save_path.name}")


def plot_beta_bars(all_results: pd.DataFrame,
                   region: str,
                   save_path: Path):
    """
    Two-row bar chart for one region:
    Row 1 — Model 1 (bare): one bar per band × predictor
    Row 2 — Model 2 (controlled): one bar per band × predictor
    """
    color  = REGION_COLORS.get(region, '#444444')
    n_cols = len(PREDICTORS)
    fig, axes = plt.subplots(2, n_cols, figsize=(7 * n_cols, 8))
    fig.patch.set_facecolor(BG_COLOR)
    if n_cols == 1:
        axes = axes[:, np.newaxis]

    x     = np.arange(len(BANDS))
    width = 0.50

    for col_idx, pred in enumerate(PREDICTORS):
        for row_idx, model_key in enumerate(['Model1', 'Model2']):
            ax = axes[row_idx, col_idx]
            _style_ax(ax)

            betas, errors, pvals = [], [], []
            for band in BANDS:
                sub = all_results[
                    (all_results['region']                == region) &
                    (all_results['predictor_of_interest'] == pred) &
                    (all_results['model']                 == model_key) &
                    (all_results['band']                  == band) &
                    (all_results['col']                   == pred)
                ]
                if sub.empty:
                    betas.append(np.nan); errors.append(0); pvals.append(1.0)
                else:
                    r = sub.iloc[0]
                    betas.append(r['beta'])
                    errors.append(r['se'])
                    pvals.append(r.get('p_fdr', r['p_raw']))

            plot_betas  = [b if np.isfinite(b) else 0 for b in betas]
            plot_errors = [e if np.isfinite(betas[i]) else 0
                           for i, e in enumerate(errors)]

            bars = ax.bar(x, plot_betas, width,
                          color=[BAND_COLORS[b] for b in BANDS],
                          yerr=plot_errors,
                          error_kw=dict(ecolor=TEXT_COLOR, capsize=3,
                                        elinewidth=1),
                          alpha=0.82)

            for bar, beta, p in zip(bars, betas, pvals):
                if not np.isfinite(beta):
                    continue
                stars = sig_stars(p)
                if stars:
                    h    = bar.get_height()
                    sign = 1 if h >= 0 else -1
                    ax.text(bar.get_x() + bar.get_width() / 2,
                            h + sign * max(abs(h) * 0.05, 0.005),
                            stars, ha='center', va='bottom',
                            color=TEXT_COLOR, fontsize=9, fontweight='bold')

            ax.axhline(0, color=SPINE_COLOR, lw=1.0, ls='--')
            ax.set_xticks(x)
            ax.set_xticklabels([b.capitalize() for b in BANDS], fontsize=9)
            ax.set_xlabel('Frequency Band', fontsize=9)
            ax.set_ylabel('β', fontsize=9)
            ax.set_title(
                f"{region}  |  RSA_r ~ {PRED_LABELS[pred]}\n"
                f"[{MODEL_LABELS[model_key]}]",
                fontsize=9, fontweight='bold')

    fig.suptitle(
        f'{region}  |  RSA_r ~ T_lag / SP_lag  |  '
        f'β by Frequency Band  [{EXPERIMENT} / {PHASE}]',
        fontsize=12, fontweight='bold', y=1.01, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"  ✓ Beta bars → {save_path.name}")


# ============================================================================
# DATA LOADING
# ============================================================================

def load_data() -> Optional[pd.DataFrame]:
    fpath = (INPUT_DIR
             / f"ALL_SUBJECTS_{EXPERIMENT}_allregions_allbands_rsa_lag.csv")

    if not fpath.exists():
        print(f"  ✗ Master CSV not found: {fpath}")
        print("  Trying per-region fallback …")
        dfs = []
        for region in REGIONS:
            for band in BANDS:
                p = (INPUT_DIR /
                     f"ALL_SUBJECTS_{EXPERIMENT}_{region}_retrieval_{band}_rsa_lag.csv")
                if p.exists():
                    tmp = pd.read_csv(p)
                    dfs.append(tmp)
        if not dfs:
            print("  ✗ No fallback files found either.")
            return None
        df = pd.concat(dfs, ignore_index=True)
    else:
        print(f"  Loading {fpath.name} …")
        df = pd.read_csv(fpath)
        print(f"  Loaded {len(df):,} rows")

    # ── Filter to retrieval phase only ────────────────────────────────────────
    if 'phase' in df.columns:
        before = len(df)
        df     = df[df['phase'] == PHASE].copy()
        print(f"  Phase filter ({PHASE}): {before:,} → {len(df):,} rows")
    else:
        print("  WARNING: 'phase' column not found — assuming all rows are retrieval")

    # ── Filter to target regions ──────────────────────────────────────────────
    if 'region' in df.columns:
        df = df[df['region'].isin(REGIONS)].copy()
        print(f"  Region filter {REGIONS}: {len(df):,} rows")

    if df.empty:
        print("  ✗ No data after filtering.")
        return None

    # Prefix subjects with experiment to future-proof merges
    df['subject'] = (EXPERIMENT + '_'
                     + df['subject'].astype(str))

    print(f"\n  Total rows   : {len(df):,}")
    print(f"  Subjects     : {df['subject'].nunique()}")
    print(f"  Regions      : {sorted(df['region'].unique().tolist())}")
    print(f"  Bands        : {sorted(df['band'].unique().tolist())}")
    return df


# ============================================================================
# ANALYSIS LOOP  (one region × predictor × band)
# ============================================================================

def run_analysis_for_region_band(df: pd.DataFrame,
                                  region: str,
                                  predictor: str,
                                  band: str,
                                  has_semsim: bool
                                  ) -> Tuple[pd.DataFrame, str]:
    """
    Fits Model 1 & Model 2 for RSA_r ~ predictor within one
    region × frequency band (retrieval phase, single experiment).
    """
    sub = df[
        (df['region'] == region) &
        (df['band']   == band)
    ].copy()

    if sub.empty:
        return pd.DataFrame(), ""

    print(f"\n  ── {region}  |  RSA_r ~ {predictor}  |  {band.capitalize()} ──")
    print(f"     Rows: {len(sub):,}  |  Subjects: {sub['subject'].nunique()}")

    all_rows    = []
    text_blocks = [
        f"EXPERIMENT: {EXPERIMENT}  PHASE: {PHASE}  "
        f"REGION: {region}  PREDICTOR: {predictor}  BAND: {band}",
        f"N rows: {len(sub):,}  |  subjects: {sub['subject'].nunique()}\n",
    ]

    cross_cov   = CROSS_COVARIATE[predictor]
    cross_label = PRED_LABELS[cross_cov]

    def run_and_collect(real_cols, model_key, title, pred_display,
                        formula_rhs=None):
        res, _ = fit_lmm(sub, real_cols, model_key,
                         formula_rhs=formula_rhs)
        rows   = extract_rows(res, pred_display, model_key,
                              predictor, band, region)
        rows   = apply_fdr_within_model(rows)
        if not rows.empty:
            all_rows.append(rows)
            block = format_block(title, rows)
            text_blocks.append(block)
            print('\n' + block)
        return res

    # ── Model 1: bare — no covariates, no experiment dummy ───────────────────
    run_and_collect(
        [predictor],
        'Model1',
        f'Model 1 — RSA_r ~ {predictor}  [{region} / {band}]',
        {predictor: PRED_LABELS[predictor]},
    )

    # ── Model 2: controlled — cross-lag + output_lag [+ semantic_sim] ────────
    ctrl_cols = [predictor, cross_cov, 'output_lag']
    ctrl_disp = {
        predictor:   PRED_LABELS[predictor],
        cross_cov:   f'{cross_label}  [covariate]',
        'output_lag': 'output_lag',
    }
    if has_semsim:
        ctrl_cols = [predictor, cross_cov, 'output_lag', 'semantic_sim']
        ctrl_disp['semantic_sim'] = 'semantic_sim'

    run_and_collect(
        ctrl_cols,
        'Model2',
        f'Model 2 — RSA_r ~ {predictor} + {cross_cov} + controls  '
        f'[{region} / {band}]',
        ctrl_disp,
    )

    result_df = (pd.concat(all_rows, ignore_index=True)
                 if all_rows else pd.DataFrame())
    return result_df, '\n\n'.join(text_blocks)


# ============================================================================
# ENTRY POINT
# ============================================================================

def main():
    print(f"\n{'='*80}")
    print(f"RSA_r ~ T_lag / SP_lag  |  All 4 regions  |  "
          f"{EXPERIMENT}  |  {PHASE} phase")
    print(f"{'='*80}")

    df = load_data()
    if df is None or df.empty:
        print("No data loaded. Exiting.")
        return

    has_semsim = ('semantic_sim' in df.columns
                  and df['semantic_sim'].notna().any())
    print(f"\n  Outcome        : RSA_r")
    print(f"  Predictors     : {PREDICTORS}")
    print(f"  Bands          : {BANDS}")
    print(f"  Regions        : {REGIONS}")
    print(f"  semantic_sim   : {has_semsim}")

    ALL_RESULTS = []

    for region in REGIONS:
        region_results = []
        print(f"\n{'─'*80}")
        print(f"REGION: {region}")
        print(f"{'─'*80}")

        for predictor in PREDICTORS:
            for band in BANDS:
                res_df, text = run_analysis_for_region_band(
                    df, region, predictor, band, has_semsim)

                if not res_df.empty:
                    region_results.append(res_df)
                    ALL_RESULTS.append(res_df)

                tag      = f"{region}_{predictor}_{band}"
                csv_path = OUTPUT_DIR / f"LMM_{tag}_results.csv"
                txt_path = OUTPUT_DIR / f"LMM_{tag}_results.txt"
                if not res_df.empty:
                    res_df.to_csv(csv_path, index=False)
                    print(f"    ✓ {csv_path.name}")
                if text:
                    with open(txt_path, 'w') as f:
                        f.write(text)
                    print(f"    ✓ {txt_path.name}")

        # ── Per-region plots ──────────────────────────────────────────────────
        if region_results:
            reg_df = pd.concat(region_results, ignore_index=True)

            plot_summary_heatmap(
                reg_df, region,
                PLOT_DIR / f"summary_heatmap_{region}.png")
            plot_beta_bars(
                reg_df, region,
                PLOT_DIR / f"beta_bars_{region}.png")

            for predictor in PREDICTORS:
                for band in BANDS:
                    region_sub = df[
                        (df['region'] == region) &
                        (df['band']   == band)
                    ].copy()

                    plot_forest(
                        reg_df, region, predictor, band,
                        PLOT_DIR / f"forest_{region}_{predictor}_{band}.png")
                    plot_interaction(
                        region, predictor, band, region_sub,
                        PLOT_DIR / f"interaction_{region}_{predictor}_{band}.png")

    # ── Master CSV ────────────────────────────────────────────────────────────
    if ALL_RESULTS:
        master_df = pd.concat(ALL_RESULTS, ignore_index=True)
        master_path = OUTPUT_DIR / 'LMM_ALL_results.csv'
        master_df.to_csv(master_path, index=False)
        print(f"\n  ✓ Master results → {master_path.name}  "
              f"({len(master_df):,} rows)")

    print(f"\n{'='*80}")
    print(f"✓ ANALYSIS COMPLETE  [{EXPERIMENT} / {PHASE}]")
    print(f"  Results : {OUTPUT_DIR}")
    print(f"  Plots   : {PLOT_DIR}")
    print(f"{'='*80}")


if __name__ == '__main__':
    main()


RSA_r ~ T_lag / SP_lag  |  All 4 regions  |  DBOY1  |  oldret phase
  Loading ALL_SUBJECTS_DBOY1_allregions_allbands_rsa_lag.csv …
  Loaded 175,968 rows
  Phase filter (oldret): 175,968 → 29,328 rows
  Region filter ['LHP', 'RHP', 'LLTC', 'RLTC']: 29,328 rows

  Total rows   : 29,328
  Subjects     : 39
  Regions      : ['LHP', 'LLTC', 'RHP', 'RLTC']
  Bands        : ['alpha', 'beta', 'gamma', 'theta']

  Outcome        : RSA_r
  Predictors     : ['SP_lag', 'T_lag']
  Bands          : ['alpha', 'gamma']
  Regions        : ['LHP', 'RHP', 'LLTC', 'RLTC']
  semantic_sim   : True

────────────────────────────────────────────────────────────────────────────────
REGION: LHP
────────────────────────────────────────────────────────────────────────────────

  ── LHP  |  RSA_r ~ SP_lag  |  Alpha ──
     Rows: 1,814  |  Subjects: 27
    [Model1] RSA_r ~ SP_lag  |  N=1,814
    [Model1] optimizer=lbfgs  converged=True  llf=-1194.1903  AIC=nan

Model 1 — RSA_r ~ SP_lag  [LHP / alpha]
--------------

In [58]:
#!/usr/bin/env python3
"""
LMM Analysis: T_lag / SP_lag → RSA_r
All 4 regions (LHP / RHP / LLTC / RLTC), separate model sets per region
Retrieval phase only.  Single experiment — no experiment factor.
----------------------------------------------------------------------
RSA_r is the OUTCOME.  T_lag and SP_lag are the PREDICTORS OF INTEREST,
tested in separate model sets.

Input : ./rsa_lag_all_regions/ALL_SUBJECTS_{exp}_allregions_allbands_rsa_lag.csv
        (output of rsa_lag_all_regions.py)

For each REGION in [LHP, RHP, LLTC, RLTC]:
  For each PREDICTOR in [T_lag, SP_lag]:
    For each BAND in [theta, alpha, beta, gamma]:

      Model 1 (bare):
          RSA_r ~ predictor                              + (1|subj/sess)

      Model 2 (controlled):
          RSA_r ~ predictor + cross_lag + output_lag
                            + [semantic_sim if available]
                                                         + (1|subj/sess)

      (No hemisphere factor — hemisphere is fixed within each region run.)

Outputs (per region):
  ./rsa_lag_all_regions/LMM_{region}_{predictor}_{band}_results.csv
  ./rsa_lag_all_regions/LMM_{region}_{predictor}_{band}_results.txt

Master:
  ./rsa_lag_all_regions/LMM_ALL_results.csv

Plots:
  ./rsa_lag_all_regions/plots/forest_{region}_{predictor}_{band}.png
  ./rsa_lag_all_regions/plots/interaction_{region}_{predictor}_{band}.png
  ./rsa_lag_all_regions/plots/summary_heatmap_{region}.png
  ./rsa_lag_all_regions/plots/beta_bars_{region}.png
"""

import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import pearsonr, ttest_1samp
from statsmodels.regression.mixed_linear_model import MixedLM
from statsmodels.stats.multitest import fdrcorrection

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION  — edit these to switch experiment / paths
# ============================================================================

EXPERIMENT  = 'DBOY1'            # single experiment — no experiment factor
PHASE       = 'pre_vocal'        # only retrieval rows are kept PHASE = 'pre_vocal'   # -1500 to 0 ms, vocal-locked PHASE = 'peri_vocal'  # -500 to +1000 ms, vocal-locked PHASE = 'post_vocal'  # 0 to +1500 ms, vocal-locked

INPUT_DIR   = Path('./rsa_lag_all_regions')
OUTPUT_DIR  = Path('./rsa_lag_all_regions')
PLOT_DIR    = OUTPUT_DIR / 'plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

OUTCOME     = 'RSA_r'
PREDICTORS  = ['T_lag']
BANDS       = ['theta','alpha','gamma'] #'alpha', 'beta', 'gamma']
REGIONS     = ['LHP','RHP','LLTC','RLTC'] #  'LLTC', 'RLTC'

CROSS_COVARIATE = {
    'T_lag':  'SP_lag',
    'SP_lag': 'T_lag',
}
PRED_LABELS = {
    'T_lag':  'Temporal Lag (T_lag)',
    'SP_lag': 'Spatial Lag (SP_lag)',
}
MODEL_LABELS = {
    'Model1': 'Bare',
    'Model2': 'Controlled',
}

# ---- Palette (light background) ---------------------------------------------
BG_COLOR    = 'white'
AX_COLOR    = '#F7F7F7'
GRID_COLOR  = '#DDDDDD'
TEXT_COLOR  = '#222222'
SPINE_COLOR = '#AAAAAA'

REGION_COLORS = {
    'LHP':  '#1A3A6B',
    'RHP':  '#8B1A1A',
    'LLTC': '#1A6B3A',
    'RLTC': '#6B5A1A',
}
BAND_COLORS = {
    'theta': '#4575B4',
    'alpha': '#74ADD1',
    'beta':  '#F46D43',
    'gamma': '#D73027',
}

# ============================================================================
# LMM FITTING
# ============================================================================

def fit_lmm(df: pd.DataFrame,
            pred_cols: List[str],
            label: str,
            formula_rhs: Optional[str] = None) -> Tuple[Optional[object], int]:
    """
    Fit RSA_r ~ formula_rhs with nested random effects (1|subject/session).
    No experiment dummy — single experiment only.
    """
    df = df.copy()
    df['subj_sess'] = (df['subject'].astype(str)
                       + '_' + df['session'].astype(str))

    real_cols = [c for c in pred_cols if c in df.columns]
    keep      = [OUTCOME] + real_cols + ['subject', 'subj_sess']
    df        = df[keep].dropna()

    if len(df) < 20:
        print(f"    [{label}] Too few rows ({len(df)}) — skipping")
        return None, 0

    rhs     = formula_rhs if formula_rhs else ' + '.join(pred_cols)
    formula = f"{OUTCOME} ~ {rhs}"
    print(f"    [{label}] {formula}  |  N={len(df):,}")

    model = MixedLM.from_formula(
        formula,
        data       = df,
        groups     = df['subject'],
        vc_formula = {'subj_sess': '0 + C(subj_sess)'},
    )

    result = None
    for method in ['lbfgs', 'nm', 'powell']:
        try:
            result = model.fit(reml=True, method=method)
            if np.isfinite(result.llf):
                print(f"    [{label}] optimizer={method}  "
                      f"converged={getattr(result, 'converged', None)}  "
                      f"llf={result.llf:.4f}  AIC={result.aic:.4f}")
                break
            else:
                print(f"    [{label}] llf=NaN with {method}, trying next …")
        except Exception as e:
            print(f"    [{label}] {method} failed: {e}")
            result = None

    if result is None or not np.isfinite(result.llf):
        print(f"    [{label}] WARNING: fit unsuccessful.")
    return result, len(df)


# ============================================================================
# RESULT EXTRACTION
# ============================================================================

def extract_rows(result,
                 pred_display: Dict[str, str],
                 model_label: str,
                 predictor: str,
                 band: str,
                 region: str) -> pd.DataFrame:
    if result is None:
        return pd.DataFrame()
    rows = []
    for col, name in pred_display.items():
        matched = col if col in result.params.index else None
        if matched is None:
            hits    = [k for k in result.params.index
                       if col.lower() in k.lower()]
            matched = hits[0] if hits else None
        if matched is None:
            print(f"    WARNING: '{col}' not found in params — skipping")
            continue
        rows.append({
            'experiment':            EXPERIMENT,
            'phase':                 PHASE,
            'region':                region,
            'predictor_of_interest': predictor,
            'band':                  band,
            'model':                 model_label,
            'term':                  name,
            'col':                   col,
            'beta':                  result.params[matched],
            'se':                    result.bse[matched],
            'z':                     result.tvalues[matched],
            'p_raw':                 result.pvalues[matched],
            'llf':                   result.llf,
            'aic':                   result.aic,
            'nobs':                  int(result.nobs),
        })
    return pd.DataFrame(rows)


def apply_fdr_within_model(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    _, df['p_fdr'] = fdrcorrection(df['p_raw'].values)
    return df


# ============================================================================
# TEXT FORMATTING
# ============================================================================

def sig_stars(p: float) -> str:
    return ('***' if p < 0.001 else
            '**'  if p < 0.01  else
            '*'   if p < 0.05  else
            '†'   if p < 0.10  else '')


def format_block(title: str, rows_df: pd.DataFrame) -> str:
    sep  = '=' * 92
    sep2 = '-' * 92
    hdr  = (f"{'Term':<40} {'β':>8} {'SE':>8} {'z':>8} "
            f"{'p_raw':>10} {'p_fdr':>10} {'AIC':>10} {'N':>8} {'sig':>5}")
    lines = [sep, title, sep2, hdr, sep2]
    for _, row in rows_df.iterrows():
        aic_s = (f"{row['aic']:>10.2f}"
                 if pd.notna(row.get('aic')) else '       NaN')
        p_fdr = row.get('p_fdr', np.nan)
        lines.append(
            f"{row['term']:<40} {row['beta']:>8.4f} {row['se']:>8.4f} "
            f"{row['z']:>8.3f} {row['p_raw']:>10.4f} "
            f"{p_fdr:>10.4f} {aic_s} {int(row['nobs']):>8,} "
            f"{sig_stars(p_fdr):>5}"
        )
    lines += [sep2,
              'FDR: BH correction within each model across terms.',
              '† p<.10  * p<.05  ** p<.01  *** p<.001',
              f'Outcome = RSA_r.  Phase = {PHASE}.  Experiment = {EXPERIMENT}.',
              'All continuous predictors on raw scale.',
              sep]
    return '\n'.join(lines)


# ============================================================================
# PLOTTING
# ============================================================================

def _style_ax(ax):
    ax.set_facecolor(AX_COLOR)
    ax.tick_params(colors=TEXT_COLOR, labelsize=8.5)
    ax.xaxis.label.set_color(TEXT_COLOR)
    ax.yaxis.label.set_color(TEXT_COLOR)
    ax.title.set_color(TEXT_COLOR)
    ax.grid(True, color=GRID_COLOR, lw=0.6, zorder=0)
    for spine in ax.spines.values():
        spine.set_edgecolor(SPINE_COLOR)
        spine.set_linewidth(0.8)


def plot_forest(all_results: pd.DataFrame,
                region: str,
                predictor: str,
                band: str,
                save_path: Path):
    """
    Forest plot: beta of the predictor of interest ± 95%CI across
    models (Model1 / Model2) for one region × band.
    """
    df = all_results[
        (all_results['region']                == region) &
        (all_results['predictor_of_interest'] == predictor) &
        (all_results['band']                  == band) &
        (all_results['col']                   == predictor)
    ].copy()

    if df.empty:
        print(f"    No rows for forest plot {region} {predictor} {band}")
        return

    df['ci_lo'] = df['beta'] - 1.96 * df['se']
    df['ci_hi'] = df['beta'] + 1.96 * df['se']

    models = [m for m in ['Model1', 'Model2'] if m in df['model'].values]
    color  = REGION_COLORS.get(region, '#444444')

    fig, ax = plt.subplots(figsize=(9, max(3, len(models) * 1.4 + 1)))
    fig.patch.set_facecolor(BG_COLOR)
    _style_ax(ax)

    y_pos, yticks, ylabels = 0, [], []

    for model in models:
        row = df[df['model'] == model]
        if row.empty:
            continue
        row  = row.iloc[0]
        xerr = [[row['beta'] - row['ci_lo']], [row['ci_hi'] - row['beta']]]
        ax.errorbar(row['beta'], y_pos, xerr=xerr,
                    fmt='o', color=color, ecolor=color,
                    elinewidth=1.5, capsize=4, capthick=1.5,
                    markersize=7, zorder=3)
        p_show = row.get('p_fdr', row['p_raw'])
        stars  = sig_stars(p_show)
        if stars:
            offset = abs(row['ci_hi'] - row['beta']) * 0.15 + 0.001
            ax.text(row['ci_hi'] + offset, y_pos, stars,
                    color=color, va='center', fontsize=9, fontweight='bold')
        yticks.append(y_pos)
        ylabels.append(MODEL_LABELS.get(model, model))
        y_pos -= 1

    ax.axvline(0, color=SPINE_COLOR, lw=1.2, ls='--', zorder=1)
    ax.set_yticks(yticks)
    ax.set_yticklabels(ylabels, fontsize=9)
    ax.set_xlabel(f'β  ({PRED_LABELS[predictor]})', fontsize=10)
    ax.set_title(
        f"{region}  |  RSA_r ~ {PRED_LABELS[predictor]}  |  "
        f"{band.capitalize()} band  [{EXPERIMENT} / {PHASE}]",
        fontsize=11, fontweight='bold', pad=8)

    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"    ✓ Forest → {save_path.name}")


def _subject_slopes(hd: pd.DataFrame, predictor: str) -> pd.DataFrame:
    rows = []
    for subj, sg in hd.groupby('subject'):
        sg = sg.dropna(subset=[predictor, OUTCOME])
        if len(sg) < 3:
            continue
        x = sg[predictor].values.astype(float)
        y = sg[OUTCOME].values.astype(float)
        if x.std() == 0:
            continue
        m, b = np.polyfit(x, y, 1)
        rows.append({'subject': subj, 'slope': m,
                     'intercept': b, 'n_pairs': len(sg)})
    return pd.DataFrame(rows)


def _subject_zscore_rsa(hd: pd.DataFrame) -> pd.DataFrame:
    hd = hd.copy()
    def zscore(x):
        s = x.std()
        return (x - x.mean()) / s if s > 0 else x - x.mean()
    hd['RSA_r_z'] = hd.groupby('subject')[OUTCOME].transform(zscore)
    return hd


def plot_interaction(region: str,
                     predictor: str,
                     band: str,
                     raw_df: pd.DataFrame,
                     save_path: Path):
    """
    Two-panel figure for one region:
    TOP    — spaghetti + group mean (z-scored RSA_r within subject)
    BOTTOM — subject slope dot plot (OLS slope per subject)
    """
    sub_df = raw_df[
        raw_df['band'] == band
    ].copy().dropna(subset=[predictor, OUTCOME])

    if sub_df.empty:
        return

    color       = REGION_COLORS.get(region, '#444444')
    is_discrete = (predictor == 'T_lag')

    fig, (ax_sp, ax_sl) = plt.subplots(2, 1, figsize=(10, 10))
    fig.patch.set_facecolor(BG_COLOR)
    _style_ax(ax_sp)
    _style_ax(ax_sl)

    n_subj  = sub_df['subject'].nunique()
    n_pairs = len(sub_df)

    x_all = sub_df[predictor].values.astype(float)
    y_all = sub_df[OUTCOME].values.astype(float)
    r_val, p_val = pearsonr(x_all, y_all)
    p_str = f'p={p_val:.3f}' if p_val >= 0.001 else 'p<0.001'

    hd = _subject_zscore_rsa(sub_df)

    if is_discrete:
        MAX_VALS = 20
        top_vals = sorted(
            hd[predictor].value_counts().nlargest(MAX_VALS).index.tolist())
        x_grid = np.array(top_vals, dtype=float)
    else:
        N_BINS     = 12
        hd['_bin'] = pd.cut(hd[predictor], bins=N_BINS)
        bin_mids   = (hd.groupby('_bin', observed=True)[predictor]
                       .mean().dropna())
        x_grid     = bin_mids.values

    # ── TOP: Spaghetti ────────────────────────────────────────────────────────
    subj_lines = []
    for subj, sg in hd.groupby('subject'):
        sg = sg.dropna(subset=[predictor, 'RSA_r_z'])
        if len(sg) < 3:
            continue
        if is_discrete:
            pts = (sg.groupby(predictor)['RSA_r_z']
                     .mean().reindex(top_vals))
        else:
            pts = (sg.groupby('_bin', observed=True)['RSA_r_z'].mean())
            pts.index = pts.index.map(
                lambda b: b.mid if hasattr(b, 'mid') else np.nan)
            pts = pts.dropna()
        pts = pts.dropna()
        if len(pts) < 2:
            continue
        subj_lines.append(pts.values)
        ax_sp.plot(x_grid[:len(pts.values)], pts.values,
                   color=color, alpha=0.15, lw=0.9, zorder=2)

    if subj_lines:
        max_len  = max(len(s) for s in subj_lines)
        padded   = np.array([
            np.pad(s.astype(float), (0, max_len - len(s)),
                   constant_values=np.nan)
            for s in subj_lines
        ])
        grp_mean = np.nanmean(padded, axis=0)
        grp_sem  = (np.nanstd(padded, axis=0)
                    / np.sqrt((~np.isnan(padded)).sum(axis=0)))
        xg = x_grid[:max_len]

        ax_sp.fill_between(xg, grp_mean - grp_sem, grp_mean + grp_sem,
                           color=color, alpha=0.20, zorder=3)
        ax_sp.plot(xg, grp_mean, color=color, lw=2.5, zorder=4,
                   label=f'Group mean ± SEM  (n={n_subj} subj)')

        valid = np.isfinite(grp_mean) & np.isfinite(xg)
        if valid.sum() >= 2:
            m_z, b_z = np.polyfit(xg[valid], grp_mean[valid], 1)
            ax_sp.plot(xg[valid], m_z * xg[valid] + b_z,
                       color=TEXT_COLOR, lw=1.5, ls='--', alpha=0.6,
                       zorder=5, label=f'OLS  r={r_val:.3f} {p_str}')

    ax_sp.axhline(0, color=SPINE_COLOR, lw=1, ls=':', zorder=1)
    if is_discrete:
        ax_sp.set_xticks(x_grid)
        ax_sp.set_xticklabels(
            [str(int(v)) for v in x_grid], fontsize=7,
            rotation=45 if len(x_grid) > 12 else 0)
    ax_sp.set_xlabel(PRED_LABELS[predictor], fontsize=9)
    ax_sp.set_ylabel('RSA_r  (z-scored within subject)', fontsize=9)
    ax_sp.set_title(
        f'{region}  |  {band.capitalize()} band  |  '
        f'{EXPERIMENT} / {PHASE}\n'
        f'Spaghetti: each line = 1 subject  (n={n_subj}, {n_pairs:,} pairs)',
        fontsize=10, fontweight='bold')
    ax_sp.legend(facecolor='white', edgecolor=SPINE_COLOR, fontsize=8)

    # ── BOTTOM: Subject slope dot plot ────────────────────────────────────────
    slopes_df = _subject_slopes(hd, predictor)

    if slopes_df.empty:
        ax_sl.set_visible(False)
    else:
        slopes     = slopes_df['slope'].values
        n_sl       = len(slopes)
        mean_slope = slopes.mean()
        sem_slope  = slopes.std() / np.sqrt(n_sl)
        ci95       = 1.96 * sem_slope
        n_neg      = (slopes < 0).sum()
        n_pos      = (slopes > 0).sum()

        order      = np.argsort(slopes)
        y_pos_sl   = np.arange(n_sl)
        dot_colors = [REGION_COLORS.get(region, '#444444')
                      if s >= 0 else '#888888'
                      for s in slopes[order]]

        ax_sl.scatter(slopes[order], y_pos_sl,
                      c=dot_colors, s=40, zorder=3,
                      edgecolors='white', linewidths=0.4)
        for yi, si, dc in zip(y_pos_sl, slopes[order], dot_colors):
            ax_sl.plot([0, si], [yi, yi],
                       color=dc, alpha=0.35, lw=0.8, zorder=2)

        ax_sl.axvline(0, color=SPINE_COLOR, lw=1.2, ls='--', zorder=1)
        ax_sl.axvspan(mean_slope - ci95, mean_slope + ci95,
                      color=color, alpha=0.12, zorder=0)
        ax_sl.axvline(mean_slope, color=color, lw=2.5, zorder=4,
                      label=f'Mean β={mean_slope:.4f} ± {ci95:.4f} (95%CI)')

        t_stat, p_1samp = ttest_1samp(slopes, 0)
        p1_str = f'p={p_1samp:.3f}' if p_1samp >= 0.001 else 'p<0.001'
        stars  = sig_stars(p_1samp)
        ax_sl.text(mean_slope, n_sl + 0.3,
                   f'{stars}  {p1_str}' if stars else p1_str,
                   ha='center', va='bottom',
                   color=color, fontsize=9, fontweight='bold')

        ax_sl.set_yticks([])
        ax_sl.set_xlabel(
            f'Subject-level OLS slope  (RSA_r ~ {predictor})', fontsize=9)
        ax_sl.set_title(
            f'{region}  |  {band.capitalize()} band\n'
            f'Each dot = 1 subject slope  '
            f'({n_neg}/{n_sl} negative, {n_pos}/{n_sl} positive)',
            fontsize=10, fontweight='bold')
        ax_sl.legend(facecolor='white', edgecolor=SPINE_COLOR, fontsize=8)
        pct_neg = n_neg / n_sl * 100
        ax_sl.text(0.02, 0.04,
                   f'{pct_neg:.0f}% of subjects show negative slope',
                   transform=ax_sl.transAxes,
                   fontsize=8, color=TEXT_COLOR, style='italic')

    fig.suptitle(
        f"RSA_r ~ {PRED_LABELS[predictor]}  |  {region}  |  "
        f"{band.capitalize()} band\n"
        f"Top: z-scored spaghetti + group mean  |  "
        f"Bottom: per-subject slopes",
        fontsize=12, fontweight='bold', y=1.01, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"    ✓ Interaction plot → {save_path.name}")


def plot_summary_heatmap(all_results: pd.DataFrame,
                         region: str,
                         save_path: Path):
    """
    Heatmap of predictor beta (Model1) across predictor × band for one region.
    """
    pivot_rows = []
    for pred in PREDICTORS:
        for band in BANDS:
            sub = all_results[
                (all_results['region']                == region) &
                (all_results['predictor_of_interest'] == pred) &
                (all_results['band']                  == band) &
                (all_results['model']                 == 'Model1') &
                (all_results['col']                   == pred)
            ]
            if sub.empty:
                beta, stars = np.nan, ''
            else:
                r     = sub.iloc[0]
                beta  = r['beta']
                stars = sig_stars(r.get('p_fdr', r['p_raw']))
            pivot_rows.append({'predictor': PRED_LABELS[pred],
                                'band': band, 'beta': beta, 'stars': stars})

    piv       = pd.DataFrame(pivot_rows)
    beta_mat  = piv.pivot(index='predictor', columns='band', values='beta')[BANDS]
    stars_mat = piv.pivot(index='predictor', columns='band', values='stars')[BANDS]

    fig, ax = plt.subplots(figsize=(9, 3.0))
    fig.patch.set_facecolor(BG_COLOR)
    vmax = np.nanmax(np.abs(beta_mat.values)) or 1.0
    im   = ax.imshow(beta_mat.values.astype(float),
                     aspect='auto', cmap='RdBu_r',
                     vmin=-vmax, vmax=vmax)

    for i in range(beta_mat.shape[0]):
        for j in range(beta_mat.shape[1]):
            val  = beta_mat.values[i, j]
            star = stars_mat.values[i, j]
            if not np.isnan(val):
                cell_norm = (val + vmax) / (2 * vmax)
                txt_color = ('white'
                             if (cell_norm < 0.35 or cell_norm > 0.65)
                             else TEXT_COLOR)
                ax.text(j, i, f"{val:.3f}\n{star}",
                        ha='center', va='center',
                        color=txt_color, fontsize=9, fontweight='bold')

    ax.set_xticks(range(len(BANDS)))
    ax.set_xticklabels([b.capitalize() for b in BANDS], fontsize=10)
    ax.set_yticks(range(len(beta_mat.index)))
    ax.set_yticklabels(beta_mat.index.tolist(), fontsize=9)
    ax.tick_params(colors=TEXT_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(SPINE_COLOR)

    cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label('β  (predictor → RSA_r)', fontsize=9, color=TEXT_COLOR)
    cbar.ax.yaxis.set_tick_params(color=TEXT_COLOR)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COLOR)

    ax.set_title(
        f'{region}  |  RSA_r ~ predictor  |  β (Model 1)  '
        f'across Predictors × Bands  [{EXPERIMENT} / {PHASE}]',
        fontsize=10, fontweight='bold', pad=8, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"  ✓ Heatmap → {save_path.name}")


def plot_beta_bars(all_results: pd.DataFrame,
                   region: str,
                   save_path: Path):
    """
    Two-row bar chart for one region:
    Row 1 — Model 1 (bare): one bar per band × predictor
    Row 2 — Model 2 (controlled): one bar per band × predictor
    """
    color  = REGION_COLORS.get(region, '#444444')
    n_cols = len(PREDICTORS)
    fig, axes = plt.subplots(2, n_cols, figsize=(7 * n_cols, 8))
    fig.patch.set_facecolor(BG_COLOR)
    if n_cols == 1:
        axes = axes[:, np.newaxis]

    x     = np.arange(len(BANDS))
    width = 0.50

    for col_idx, pred in enumerate(PREDICTORS):
        for row_idx, model_key in enumerate(['Model1', 'Model2']):
            ax = axes[row_idx, col_idx]
            _style_ax(ax)

            betas, errors, pvals = [], [], []
            for band in BANDS:
                sub = all_results[
                    (all_results['region']                == region) &
                    (all_results['predictor_of_interest'] == pred) &
                    (all_results['model']                 == model_key) &
                    (all_results['band']                  == band) &
                    (all_results['col']                   == pred)
                ]
                if sub.empty:
                    betas.append(np.nan); errors.append(0); pvals.append(1.0)
                else:
                    r = sub.iloc[0]
                    betas.append(r['beta'])
                    errors.append(r['se'])
                    pvals.append(r.get('p_fdr', r['p_raw']))

            plot_betas  = [b if np.isfinite(b) else 0 for b in betas]
            plot_errors = [e if np.isfinite(betas[i]) else 0
                           for i, e in enumerate(errors)]

            bars = ax.bar(x, plot_betas, width,
                          color=[BAND_COLORS[b] for b in BANDS],
                          yerr=plot_errors,
                          error_kw=dict(ecolor=TEXT_COLOR, capsize=3,
                                        elinewidth=1),
                          alpha=0.82)

            for bar, beta, p in zip(bars, betas, pvals):
                if not np.isfinite(beta):
                    continue
                stars = sig_stars(p)
                if stars:
                    h    = bar.get_height()
                    sign = 1 if h >= 0 else -1
                    ax.text(bar.get_x() + bar.get_width() / 2,
                            h + sign * max(abs(h) * 0.05, 0.005),
                            stars, ha='center', va='bottom',
                            color=TEXT_COLOR, fontsize=9, fontweight='bold')

            ax.axhline(0, color=SPINE_COLOR, lw=1.0, ls='--')
            ax.set_xticks(x)
            ax.set_xticklabels([b.capitalize() for b in BANDS], fontsize=9)
            ax.set_xlabel('Frequency Band', fontsize=9)
            ax.set_ylabel('β', fontsize=9)
            ax.set_title(
                f"{region}  |  RSA_r ~ {PRED_LABELS[pred]}\n"
                f"[{MODEL_LABELS[model_key]}]",
                fontsize=9, fontweight='bold')

    fig.suptitle(
        f'{region}  |  RSA_r ~ T_lag / SP_lag  |  '
        f'β by Frequency Band  [{EXPERIMENT} / {PHASE}]',
        fontsize=12, fontweight='bold', y=1.01, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"  ✓ Beta bars → {save_path.name}")


# ============================================================================
# DATA LOADING
# ============================================================================

def load_data() -> Optional[pd.DataFrame]:
    fpath = (INPUT_DIR
             / f"ALL_SUBJECTS_{EXPERIMENT}_allregions_allbands_rsa_lag.csv")

    if not fpath.exists():
        print(f"  ✗ Master CSV not found: {fpath}")
        print("  Trying per-region fallback …")
        dfs = []
        for region in REGIONS:
            for band in BANDS:
                p = (INPUT_DIR /
                     f"ALL_SUBJECTS_{EXPERIMENT}_{region}_retrieval_{band}_rsa_lag.csv")
                if p.exists():
                    tmp = pd.read_csv(p)
                    dfs.append(tmp)
        if not dfs:
            print("  ✗ No fallback files found either.")
            return None
        df = pd.concat(dfs, ignore_index=True)
    else:
        print(f"  Loading {fpath.name} …")
        df = pd.read_csv(fpath)
        print(f"  Loaded {len(df):,} rows")

    # ── Filter to retrieval phase only ────────────────────────────────────────
    if 'phase' in df.columns:
        before = len(df)
        df     = df[df['phase'] == PHASE].copy()
        print(f"  Phase filter ({PHASE}): {before:,} → {len(df):,} rows")
    else:
        print("  WARNING: 'phase' column not found — assuming all rows are retrieval")

    # ── Filter to target regions ──────────────────────────────────────────────
    if 'region' in df.columns:
        df = df[df['region'].isin(REGIONS)].copy()
        print(f"  Region filter {REGIONS}: {len(df):,} rows")

    if df.empty:
        print("  ✗ No data after filtering.")
        return None

    # Prefix subjects with experiment to future-proof merges
    df['subject'] = (EXPERIMENT + '_'
                     + df['subject'].astype(str))

    print(f"\n  Total rows   : {len(df):,}")
    print(f"  Subjects     : {df['subject'].nunique()}")
    print(f"  Regions      : {sorted(df['region'].unique().tolist())}")
    print(f"  Bands        : {sorted(df['band'].unique().tolist())}")
    return df


# ============================================================================
# ANALYSIS LOOP  (one region × predictor × band)
# ============================================================================

def run_analysis_for_region_band(df: pd.DataFrame,
                                  region: str,
                                  predictor: str,
                                  band: str,
                                  has_semsim: bool
                                  ) -> Tuple[pd.DataFrame, str]:
    """
    Fits Model 1 & Model 2 for RSA_r ~ predictor within one
    region × frequency band (retrieval phase, single experiment).
    """
    sub = df[
        (df['region'] == region) &
        (df['band']   == band)
    ].copy()

    if sub.empty:
        return pd.DataFrame(), ""

    print(f"\n  ── {region}  |  RSA_r ~ {predictor}  |  {band.capitalize()} ──")
    print(f"     Rows: {len(sub):,}  |  Subjects: {sub['subject'].nunique()}")

    all_rows    = []
    text_blocks = [
        f"EXPERIMENT: {EXPERIMENT}  PHASE: {PHASE}  "
        f"REGION: {region}  PREDICTOR: {predictor}  BAND: {band}",
        f"N rows: {len(sub):,}  |  subjects: {sub['subject'].nunique()}\n",
    ]

    cross_cov   = CROSS_COVARIATE[predictor]
    cross_label = PRED_LABELS[cross_cov]

    def run_and_collect(real_cols, model_key, title, pred_display,
                        formula_rhs=None):
        res, _ = fit_lmm(sub, real_cols, model_key,
                         formula_rhs=formula_rhs)
        rows   = extract_rows(res, pred_display, model_key,
                              predictor, band, region)
        rows   = apply_fdr_within_model(rows)
        if not rows.empty:
            all_rows.append(rows)
            block = format_block(title, rows)
            text_blocks.append(block)
            print('\n' + block)
        return res

    # ── Model 1: bare — no covariates, no experiment dummy ───────────────────
    run_and_collect(
        [predictor],
        'Model1',
        f'Model 1 — RSA_r ~ {predictor}  [{region} / {band}]',
        {predictor: PRED_LABELS[predictor]},
    )

    # ── Model 2: controlled — cross-lag + output_lag [+ semantic_sim] ────────
    ctrl_cols = [predictor, cross_cov, 'output_lag']
    ctrl_disp = {
        predictor:   PRED_LABELS[predictor],
        cross_cov:   f'{cross_label}  [covariate]',
        'output_lag': 'output_lag',
    }
    if has_semsim:
        ctrl_cols = [predictor, cross_cov, 'output_lag', 'semantic_sim']
        ctrl_disp['semantic_sim'] = 'semantic_sim'

    run_and_collect(
        ctrl_cols,
        'Model2',
        f'Model 2 — RSA_r ~ {predictor} + {cross_cov} + controls  '
        f'[{region} / {band}]',
        ctrl_disp,
    )

    result_df = (pd.concat(all_rows, ignore_index=True)
                 if all_rows else pd.DataFrame())
    return result_df, '\n\n'.join(text_blocks)


# ============================================================================
# ENTRY POINT
# ============================================================================

def main():
    print(f"\n{'='*80}")
    print(f"RSA_r ~ T_lag / SP_lag  |  All 4 regions  |  "
          f"{EXPERIMENT}  |  {PHASE} phase")
    print(f"{'='*80}")

    df = load_data()
    if df is None or df.empty:
        print("No data loaded. Exiting.")
        return

    has_semsim = ('semantic_sim' in df.columns
                  and df['semantic_sim'].notna().any())
    print(f"\n  Outcome        : RSA_r")
    print(f"  Predictors     : {PREDICTORS}")
    print(f"  Bands          : {BANDS}")
    print(f"  Regions        : {REGIONS}")
    print(f"  semantic_sim   : {has_semsim}")

    ALL_RESULTS = []

    for region in REGIONS:
        region_results = []
        print(f"\n{'─'*80}")
        print(f"REGION: {region}")
        print(f"{'─'*80}")

        for predictor in PREDICTORS:
            for band in BANDS:
                res_df, text = run_analysis_for_region_band(
                    df, region, predictor, band, has_semsim)

                if not res_df.empty:
                    region_results.append(res_df)
                    ALL_RESULTS.append(res_df)

                tag      = f"{region}_{predictor}_{band}"
                csv_path = OUTPUT_DIR / f"LMM_{tag}_results.csv"
                txt_path = OUTPUT_DIR / f"LMM_{tag}_results.txt"
                if not res_df.empty:
                    res_df.to_csv(csv_path, index=False)
                    print(f"    ✓ {csv_path.name}")
                if text:
                    with open(txt_path, 'w') as f:
                        f.write(text)
                    print(f"    ✓ {txt_path.name}")

        # ── Per-region plots ──────────────────────────────────────────────────
        if region_results:
            reg_df = pd.concat(region_results, ignore_index=True)

            plot_summary_heatmap(
                reg_df, region,
                PLOT_DIR / f"summary_heatmap_{region}.png")
            plot_beta_bars(
                reg_df, region,
                PLOT_DIR / f"beta_bars_{region}.png")

            for predictor in PREDICTORS:
                for band in BANDS:
                    region_sub = df[
                        (df['region'] == region) &
                        (df['band']   == band)
                    ].copy()

                    plot_forest(
                        reg_df, region, predictor, band,
                        PLOT_DIR / f"forest_{region}_{predictor}_{band}.png")
                    plot_interaction(
                        region, predictor, band, region_sub,
                        PLOT_DIR / f"interaction_{region}_{predictor}_{band}.png")

    # ── Master CSV ────────────────────────────────────────────────────────────
    if ALL_RESULTS:
        master_df = pd.concat(ALL_RESULTS, ignore_index=True)
        master_path = OUTPUT_DIR / 'LMM_ALL_results.csv'
        master_df.to_csv(master_path, index=False)
        print(f"\n  ✓ Master results → {master_path.name}  "
              f"({len(master_df):,} rows)")

    print(f"\n{'='*80}")
    print(f"✓ ANALYSIS COMPLETE  [{EXPERIMENT} / {PHASE}]")
    print(f"  Results : {OUTPUT_DIR}")
    print(f"  Plots   : {PLOT_DIR}")
    print(f"{'='*80}")


if __name__ == '__main__':
    main()


RSA_r ~ T_lag / SP_lag  |  All 4 regions  |  DBOY1  |  pre_vocal phase
  Loading ALL_SUBJECTS_DBOY1_allregions_allbands_rsa_lag.csv …
  Loaded 117,312 rows
  Phase filter (pre_vocal): 117,312 → 29,328 rows
  Region filter ['LHP', 'RHP', 'LLTC', 'RLTC']: 29,328 rows

  Total rows   : 29,328
  Subjects     : 39
  Regions      : ['LHP', 'LLTC', 'RHP', 'RLTC']
  Bands        : ['alpha', 'beta', 'gamma', 'theta']

  Outcome        : RSA_r
  Predictors     : ['T_lag']
  Bands          : ['theta', 'alpha', 'gamma']
  Regions        : ['LHP', 'RHP', 'LLTC', 'RLTC']
  semantic_sim   : True

────────────────────────────────────────────────────────────────────────────────
REGION: LHP
────────────────────────────────────────────────────────────────────────────────

  ── LHP  |  RSA_r ~ T_lag  |  Theta ──
     Rows: 1,814  |  Subjects: 27
    [Model1] RSA_r ~ T_lag  |  N=1,814
    [Model1] optimizer=lbfgs  converged=True  llf=-913.5854  AIC=nan

Model 1 — RSA_r ~ T_lag  [LHP / theta]
-------------

In [57]:
#!/usr/bin/env python3
"""
LMM Analysis: T_lag / SP_lag → RSA_r
All 4 regions (LHP / RHP / LLTC / RLTC), separate model sets per region
Retrieval phase only.  Single experiment — no experiment factor.
----------------------------------------------------------------------
RSA_r is the OUTCOME.  T_lag and SP_lag are the PREDICTORS OF INTEREST,
tested in separate model sets.

Input : ./rsa_lag_all_regions/ALL_SUBJECTS_{exp}_allregions_allbands_rsa_lag.csv
        (output of rsa_lag_all_regions.py)

For each REGION in [LHP, RHP, LLTC, RLTC]:
  For each PREDICTOR in [T_lag, SP_lag]:
    For each BAND in [theta, alpha, beta, gamma]:

      Model 1 (bare):
          RSA_r ~ predictor                              + (1|subj/sess)

      Model 2 (controlled):
          RSA_r ~ predictor + cross_lag + output_lag
                            + [semantic_sim if available]
                                                         + (1|subj/sess)

      (No hemisphere factor — hemisphere is fixed within each region run.)

Outputs (per region):
  ./rsa_lag_all_regions/LMM_{region}_{predictor}_{band}_results.csv
  ./rsa_lag_all_regions/LMM_{region}_{predictor}_{band}_results.txt

Master:
  ./rsa_lag_all_regions/LMM_ALL_results.csv

Plots:
  ./rsa_lag_all_regions/plots/forest_{region}_{predictor}_{band}.png
  ./rsa_lag_all_regions/plots/interaction_{region}_{predictor}_{band}.png
  ./rsa_lag_all_regions/plots/summary_heatmap_{region}.png
  ./rsa_lag_all_regions/plots/beta_bars_{region}.png
"""

import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import pearsonr, ttest_1samp
from statsmodels.regression.mixed_linear_model import MixedLM
from statsmodels.stats.multitest import fdrcorrection

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION  — edit these to switch experiment / paths
# ============================================================================

EXPERIMENT  = 'DBOY1'            # single experiment — no experiment factor
PHASE       = 'encoding'        # only retrieval rows are kept PHASE = 'pre_vocal'   # -1500 to 0 ms, vocal-locked PHASE = 'peri_vocal'  # -500 to +1000 ms, vocal-locked PHASE = 'post_vocal'  # 0 to +1500 ms, vocal-locked

INPUT_DIR   = Path('./rsa_lag_all_regions')
OUTPUT_DIR  = Path('./rsa_lag_all_regions')
PLOT_DIR    = OUTPUT_DIR / 'plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

OUTCOME     = 'RSA_r'
PREDICTORS  = ['SP_lag']
BANDS       = ['theta','alpha','gamma'] #'alpha', 'beta', 'gamma']
REGIONS     = ['LHP','RHP','LLTC','RLTC'] #  'LLTC', 'RLTC'

CROSS_COVARIATE = {
    'T_lag':  'SP_lag',
    'SP_lag': 'T_lag',
}
PRED_LABELS = {
    'T_lag':  'Temporal Lag (T_lag)',
    'SP_lag': 'Spatial Lag (SP_lag)',
}
MODEL_LABELS = {
    'Model1': 'Bare',
    'Model2': 'Controlled',
}

# ---- Palette (light background) ---------------------------------------------
BG_COLOR    = 'white'
AX_COLOR    = '#F7F7F7'
GRID_COLOR  = '#DDDDDD'
TEXT_COLOR  = '#222222'
SPINE_COLOR = '#AAAAAA'

REGION_COLORS = {
    'LHP':  '#1A3A6B',
    'RHP':  '#8B1A1A',
    'LLTC': '#1A6B3A',
    'RLTC': '#6B5A1A',
}
BAND_COLORS = {
    'theta': '#4575B4',
    'alpha': '#74ADD1',
    'beta':  '#F46D43',
    'gamma': '#D73027',
}

# ============================================================================
# LMM FITTING
# ============================================================================

def fit_lmm(df: pd.DataFrame,
            pred_cols: List[str],
            label: str,
            formula_rhs: Optional[str] = None) -> Tuple[Optional[object], int]:
    """
    Fit RSA_r ~ formula_rhs with nested random effects (1|subject/session).
    No experiment dummy — single experiment only.
    """
    df = df.copy()
    df['subj_sess'] = (df['subject'].astype(str)
                       + '_' + df['session'].astype(str))

    real_cols = [c for c in pred_cols if c in df.columns]
    keep      = [OUTCOME] + real_cols + ['subject', 'subj_sess']
    df        = df[keep].dropna()

    if len(df) < 20:
        print(f"    [{label}] Too few rows ({len(df)}) — skipping")
        return None, 0

    rhs     = formula_rhs if formula_rhs else ' + '.join(pred_cols)
    formula = f"{OUTCOME} ~ {rhs}"
    print(f"    [{label}] {formula}  |  N={len(df):,}")

    model = MixedLM.from_formula(
        formula,
        data       = df,
        groups     = df['subject'],
        vc_formula = {'subj_sess': '0 + C(subj_sess)'},
    )

    result = None
    for method in ['lbfgs', 'nm', 'powell']:
        try:
            result = model.fit(reml=True, method=method)
            if np.isfinite(result.llf):
                print(f"    [{label}] optimizer={method}  "
                      f"converged={getattr(result, 'converged', None)}  "
                      f"llf={result.llf:.4f}  AIC={result.aic:.4f}")
                break
            else:
                print(f"    [{label}] llf=NaN with {method}, trying next …")
        except Exception as e:
            print(f"    [{label}] {method} failed: {e}")
            result = None

    if result is None or not np.isfinite(result.llf):
        print(f"    [{label}] WARNING: fit unsuccessful.")
    return result, len(df)


# ============================================================================
# RESULT EXTRACTION
# ============================================================================

def extract_rows(result,
                 pred_display: Dict[str, str],
                 model_label: str,
                 predictor: str,
                 band: str,
                 region: str) -> pd.DataFrame:
    if result is None:
        return pd.DataFrame()
    rows = []
    for col, name in pred_display.items():
        matched = col if col in result.params.index else None
        if matched is None:
            hits    = [k for k in result.params.index
                       if col.lower() in k.lower()]
            matched = hits[0] if hits else None
        if matched is None:
            print(f"    WARNING: '{col}' not found in params — skipping")
            continue
        rows.append({
            'experiment':            EXPERIMENT,
            'phase':                 PHASE,
            'region':                region,
            'predictor_of_interest': predictor,
            'band':                  band,
            'model':                 model_label,
            'term':                  name,
            'col':                   col,
            'beta':                  result.params[matched],
            'se':                    result.bse[matched],
            'z':                     result.tvalues[matched],
            'p_raw':                 result.pvalues[matched],
            'llf':                   result.llf,
            'aic':                   result.aic,
            'nobs':                  int(result.nobs),
        })
    return pd.DataFrame(rows)


def apply_fdr_within_model(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    _, df['p_fdr'] = fdrcorrection(df['p_raw'].values)
    return df


# ============================================================================
# TEXT FORMATTING
# ============================================================================

def sig_stars(p: float) -> str:
    return ('***' if p < 0.001 else
            '**'  if p < 0.01  else
            '*'   if p < 0.05  else
            '†'   if p < 0.10  else '')


def format_block(title: str, rows_df: pd.DataFrame) -> str:
    sep  = '=' * 92
    sep2 = '-' * 92
    hdr  = (f"{'Term':<40} {'β':>8} {'SE':>8} {'z':>8} "
            f"{'p_raw':>10} {'p_fdr':>10} {'AIC':>10} {'N':>8} {'sig':>5}")
    lines = [sep, title, sep2, hdr, sep2]
    for _, row in rows_df.iterrows():
        aic_s = (f"{row['aic']:>10.2f}"
                 if pd.notna(row.get('aic')) else '       NaN')
        p_fdr = row.get('p_fdr', np.nan)
        lines.append(
            f"{row['term']:<40} {row['beta']:>8.4f} {row['se']:>8.4f} "
            f"{row['z']:>8.3f} {row['p_raw']:>10.4f} "
            f"{p_fdr:>10.4f} {aic_s} {int(row['nobs']):>8,} "
            f"{sig_stars(p_fdr):>5}"
        )
    lines += [sep2,
              'FDR: BH correction within each model across terms.',
              '† p<.10  * p<.05  ** p<.01  *** p<.001',
              f'Outcome = RSA_r.  Phase = {PHASE}.  Experiment = {EXPERIMENT}.',
              'All continuous predictors on raw scale.',
              sep]
    return '\n'.join(lines)


# ============================================================================
# PLOTTING
# ============================================================================

def _style_ax(ax):
    ax.set_facecolor(AX_COLOR)
    ax.tick_params(colors=TEXT_COLOR, labelsize=8.5)
    ax.xaxis.label.set_color(TEXT_COLOR)
    ax.yaxis.label.set_color(TEXT_COLOR)
    ax.title.set_color(TEXT_COLOR)
    ax.grid(True, color=GRID_COLOR, lw=0.6, zorder=0)
    for spine in ax.spines.values():
        spine.set_edgecolor(SPINE_COLOR)
        spine.set_linewidth(0.8)


def plot_forest(all_results: pd.DataFrame,
                region: str,
                predictor: str,
                band: str,
                save_path: Path):
    """
    Forest plot: beta of the predictor of interest ± 95%CI across
    models (Model1 / Model2) for one region × band.
    """
    df = all_results[
        (all_results['region']                == region) &
        (all_results['predictor_of_interest'] == predictor) &
        (all_results['band']                  == band) &
        (all_results['col']                   == predictor)
    ].copy()

    if df.empty:
        print(f"    No rows for forest plot {region} {predictor} {band}")
        return

    df['ci_lo'] = df['beta'] - 1.96 * df['se']
    df['ci_hi'] = df['beta'] + 1.96 * df['se']

    models = [m for m in ['Model1', 'Model2'] if m in df['model'].values]
    color  = REGION_COLORS.get(region, '#444444')

    fig, ax = plt.subplots(figsize=(9, max(3, len(models) * 1.4 + 1)))
    fig.patch.set_facecolor(BG_COLOR)
    _style_ax(ax)

    y_pos, yticks, ylabels = 0, [], []

    for model in models:
        row = df[df['model'] == model]
        if row.empty:
            continue
        row  = row.iloc[0]
        xerr = [[row['beta'] - row['ci_lo']], [row['ci_hi'] - row['beta']]]
        ax.errorbar(row['beta'], y_pos, xerr=xerr,
                    fmt='o', color=color, ecolor=color,
                    elinewidth=1.5, capsize=4, capthick=1.5,
                    markersize=7, zorder=3)
        p_show = row.get('p_fdr', row['p_raw'])
        stars  = sig_stars(p_show)
        if stars:
            offset = abs(row['ci_hi'] - row['beta']) * 0.15 + 0.001
            ax.text(row['ci_hi'] + offset, y_pos, stars,
                    color=color, va='center', fontsize=9, fontweight='bold')
        yticks.append(y_pos)
        ylabels.append(MODEL_LABELS.get(model, model))
        y_pos -= 1

    ax.axvline(0, color=SPINE_COLOR, lw=1.2, ls='--', zorder=1)
    ax.set_yticks(yticks)
    ax.set_yticklabels(ylabels, fontsize=9)
    ax.set_xlabel(f'β  ({PRED_LABELS[predictor]})', fontsize=10)
    ax.set_title(
        f"{region}  |  RSA_r ~ {PRED_LABELS[predictor]}  |  "
        f"{band.capitalize()} band  [{EXPERIMENT} / {PHASE}]",
        fontsize=11, fontweight='bold', pad=8)

    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"    ✓ Forest → {save_path.name}")


def _subject_slopes(hd: pd.DataFrame, predictor: str) -> pd.DataFrame:
    rows = []
    for subj, sg in hd.groupby('subject'):
        sg = sg.dropna(subset=[predictor, OUTCOME])
        if len(sg) < 3:
            continue
        x = sg[predictor].values.astype(float)
        y = sg[OUTCOME].values.astype(float)
        if x.std() == 0:
            continue
        m, b = np.polyfit(x, y, 1)
        rows.append({'subject': subj, 'slope': m,
                     'intercept': b, 'n_pairs': len(sg)})
    return pd.DataFrame(rows)


def _subject_zscore_rsa(hd: pd.DataFrame) -> pd.DataFrame:
    hd = hd.copy()
    def zscore(x):
        s = x.std()
        return (x - x.mean()) / s if s > 0 else x - x.mean()
    hd['RSA_r_z'] = hd.groupby('subject')[OUTCOME].transform(zscore)
    return hd


def plot_interaction(region: str,
                     predictor: str,
                     band: str,
                     raw_df: pd.DataFrame,
                     save_path: Path):
    """
    Two-panel figure for one region:
    TOP    — spaghetti + group mean (z-scored RSA_r within subject)
    BOTTOM — subject slope dot plot (OLS slope per subject)
    """
    sub_df = raw_df[
        raw_df['band'] == band
    ].copy().dropna(subset=[predictor, OUTCOME])

    if sub_df.empty:
        return

    color       = REGION_COLORS.get(region, '#444444')
    is_discrete = (predictor == 'T_lag')

    fig, (ax_sp, ax_sl) = plt.subplots(2, 1, figsize=(10, 10))
    fig.patch.set_facecolor(BG_COLOR)
    _style_ax(ax_sp)
    _style_ax(ax_sl)

    n_subj  = sub_df['subject'].nunique()
    n_pairs = len(sub_df)

    x_all = sub_df[predictor].values.astype(float)
    y_all = sub_df[OUTCOME].values.astype(float)
    r_val, p_val = pearsonr(x_all, y_all)
    p_str = f'p={p_val:.3f}' if p_val >= 0.001 else 'p<0.001'

    hd = _subject_zscore_rsa(sub_df)

    if is_discrete:
        MAX_VALS = 20
        top_vals = sorted(
            hd[predictor].value_counts().nlargest(MAX_VALS).index.tolist())
        x_grid = np.array(top_vals, dtype=float)
    else:
        N_BINS     = 12
        hd['_bin'] = pd.cut(hd[predictor], bins=N_BINS)
        bin_mids   = (hd.groupby('_bin', observed=True)[predictor]
                       .mean().dropna())
        x_grid     = bin_mids.values

    # ── TOP: Spaghetti ────────────────────────────────────────────────────────
    subj_lines = []
    for subj, sg in hd.groupby('subject'):
        sg = sg.dropna(subset=[predictor, 'RSA_r_z'])
        if len(sg) < 3:
            continue
        if is_discrete:
            pts = (sg.groupby(predictor)['RSA_r_z']
                     .mean().reindex(top_vals))
        else:
            pts = (sg.groupby('_bin', observed=True)['RSA_r_z'].mean())
            pts.index = pts.index.map(
                lambda b: b.mid if hasattr(b, 'mid') else np.nan)
            pts = pts.dropna()
        pts = pts.dropna()
        if len(pts) < 2:
            continue
        subj_lines.append(pts.values)
        ax_sp.plot(x_grid[:len(pts.values)], pts.values,
                   color=color, alpha=0.15, lw=0.9, zorder=2)

    if subj_lines:
        max_len  = max(len(s) for s in subj_lines)
        padded   = np.array([
            np.pad(s.astype(float), (0, max_len - len(s)),
                   constant_values=np.nan)
            for s in subj_lines
        ])
        grp_mean = np.nanmean(padded, axis=0)
        grp_sem  = (np.nanstd(padded, axis=0)
                    / np.sqrt((~np.isnan(padded)).sum(axis=0)))
        xg = x_grid[:max_len]

        ax_sp.fill_between(xg, grp_mean - grp_sem, grp_mean + grp_sem,
                           color=color, alpha=0.20, zorder=3)
        ax_sp.plot(xg, grp_mean, color=color, lw=2.5, zorder=4,
                   label=f'Group mean ± SEM  (n={n_subj} subj)')

        valid = np.isfinite(grp_mean) & np.isfinite(xg)
        if valid.sum() >= 2:
            m_z, b_z = np.polyfit(xg[valid], grp_mean[valid], 1)
            ax_sp.plot(xg[valid], m_z * xg[valid] + b_z,
                       color=TEXT_COLOR, lw=1.5, ls='--', alpha=0.6,
                       zorder=5, label=f'OLS  r={r_val:.3f} {p_str}')

    ax_sp.axhline(0, color=SPINE_COLOR, lw=1, ls=':', zorder=1)
    if is_discrete:
        ax_sp.set_xticks(x_grid)
        ax_sp.set_xticklabels(
            [str(int(v)) for v in x_grid], fontsize=7,
            rotation=45 if len(x_grid) > 12 else 0)
    ax_sp.set_xlabel(PRED_LABELS[predictor], fontsize=9)
    ax_sp.set_ylabel('RSA_r  (z-scored within subject)', fontsize=9)
    ax_sp.set_title(
        f'{region}  |  {band.capitalize()} band  |  '
        f'{EXPERIMENT} / {PHASE}\n'
        f'Spaghetti: each line = 1 subject  (n={n_subj}, {n_pairs:,} pairs)',
        fontsize=10, fontweight='bold')
    ax_sp.legend(facecolor='white', edgecolor=SPINE_COLOR, fontsize=8)

    # ── BOTTOM: Subject slope dot plot ────────────────────────────────────────
    slopes_df = _subject_slopes(hd, predictor)

    if slopes_df.empty:
        ax_sl.set_visible(False)
    else:
        slopes     = slopes_df['slope'].values
        n_sl       = len(slopes)
        mean_slope = slopes.mean()
        sem_slope  = slopes.std() / np.sqrt(n_sl)
        ci95       = 1.96 * sem_slope
        n_neg      = (slopes < 0).sum()
        n_pos      = (slopes > 0).sum()

        order      = np.argsort(slopes)
        y_pos_sl   = np.arange(n_sl)
        dot_colors = [REGION_COLORS.get(region, '#444444')
                      if s >= 0 else '#888888'
                      for s in slopes[order]]

        ax_sl.scatter(slopes[order], y_pos_sl,
                      c=dot_colors, s=40, zorder=3,
                      edgecolors='white', linewidths=0.4)
        for yi, si, dc in zip(y_pos_sl, slopes[order], dot_colors):
            ax_sl.plot([0, si], [yi, yi],
                       color=dc, alpha=0.35, lw=0.8, zorder=2)

        ax_sl.axvline(0, color=SPINE_COLOR, lw=1.2, ls='--', zorder=1)
        ax_sl.axvspan(mean_slope - ci95, mean_slope + ci95,
                      color=color, alpha=0.12, zorder=0)
        ax_sl.axvline(mean_slope, color=color, lw=2.5, zorder=4,
                      label=f'Mean β={mean_slope:.4f} ± {ci95:.4f} (95%CI)')

        t_stat, p_1samp = ttest_1samp(slopes, 0)
        p1_str = f'p={p_1samp:.3f}' if p_1samp >= 0.001 else 'p<0.001'
        stars  = sig_stars(p_1samp)
        ax_sl.text(mean_slope, n_sl + 0.3,
                   f'{stars}  {p1_str}' if stars else p1_str,
                   ha='center', va='bottom',
                   color=color, fontsize=9, fontweight='bold')

        ax_sl.set_yticks([])
        ax_sl.set_xlabel(
            f'Subject-level OLS slope  (RSA_r ~ {predictor})', fontsize=9)
        ax_sl.set_title(
            f'{region}  |  {band.capitalize()} band\n'
            f'Each dot = 1 subject slope  '
            f'({n_neg}/{n_sl} negative, {n_pos}/{n_sl} positive)',
            fontsize=10, fontweight='bold')
        ax_sl.legend(facecolor='white', edgecolor=SPINE_COLOR, fontsize=8)
        pct_neg = n_neg / n_sl * 100
        ax_sl.text(0.02, 0.04,
                   f'{pct_neg:.0f}% of subjects show negative slope',
                   transform=ax_sl.transAxes,
                   fontsize=8, color=TEXT_COLOR, style='italic')

    fig.suptitle(
        f"RSA_r ~ {PRED_LABELS[predictor]}  |  {region}  |  "
        f"{band.capitalize()} band\n"
        f"Top: z-scored spaghetti + group mean  |  "
        f"Bottom: per-subject slopes",
        fontsize=12, fontweight='bold', y=1.01, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"    ✓ Interaction plot → {save_path.name}")


def plot_summary_heatmap(all_results: pd.DataFrame,
                         region: str,
                         save_path: Path):
    """
    Heatmap of predictor beta (Model1) across predictor × band for one region.
    """
    pivot_rows = []
    for pred in PREDICTORS:
        for band in BANDS:
            sub = all_results[
                (all_results['region']                == region) &
                (all_results['predictor_of_interest'] == pred) &
                (all_results['band']                  == band) &
                (all_results['model']                 == 'Model1') &
                (all_results['col']                   == pred)
            ]
            if sub.empty:
                beta, stars = np.nan, ''
            else:
                r     = sub.iloc[0]
                beta  = r['beta']
                stars = sig_stars(r.get('p_fdr', r['p_raw']))
            pivot_rows.append({'predictor': PRED_LABELS[pred],
                                'band': band, 'beta': beta, 'stars': stars})

    piv       = pd.DataFrame(pivot_rows)
    beta_mat  = piv.pivot(index='predictor', columns='band', values='beta')[BANDS]
    stars_mat = piv.pivot(index='predictor', columns='band', values='stars')[BANDS]

    fig, ax = plt.subplots(figsize=(9, 3.0))
    fig.patch.set_facecolor(BG_COLOR)
    vmax = np.nanmax(np.abs(beta_mat.values)) or 1.0
    im   = ax.imshow(beta_mat.values.astype(float),
                     aspect='auto', cmap='RdBu_r',
                     vmin=-vmax, vmax=vmax)

    for i in range(beta_mat.shape[0]):
        for j in range(beta_mat.shape[1]):
            val  = beta_mat.values[i, j]
            star = stars_mat.values[i, j]
            if not np.isnan(val):
                cell_norm = (val + vmax) / (2 * vmax)
                txt_color = ('white'
                             if (cell_norm < 0.35 or cell_norm > 0.65)
                             else TEXT_COLOR)
                ax.text(j, i, f"{val:.3f}\n{star}",
                        ha='center', va='center',
                        color=txt_color, fontsize=9, fontweight='bold')

    ax.set_xticks(range(len(BANDS)))
    ax.set_xticklabels([b.capitalize() for b in BANDS], fontsize=10)
    ax.set_yticks(range(len(beta_mat.index)))
    ax.set_yticklabels(beta_mat.index.tolist(), fontsize=9)
    ax.tick_params(colors=TEXT_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(SPINE_COLOR)

    cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label('β  (predictor → RSA_r)', fontsize=9, color=TEXT_COLOR)
    cbar.ax.yaxis.set_tick_params(color=TEXT_COLOR)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COLOR)

    ax.set_title(
        f'{region}  |  RSA_r ~ predictor  |  β (Model 1)  '
        f'across Predictors × Bands  [{EXPERIMENT} / {PHASE}]',
        fontsize=10, fontweight='bold', pad=8, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"  ✓ Heatmap → {save_path.name}")


def plot_beta_bars(all_results: pd.DataFrame,
                   region: str,
                   save_path: Path):
    """
    Two-row bar chart for one region:
    Row 1 — Model 1 (bare): one bar per band × predictor
    Row 2 — Model 2 (controlled): one bar per band × predictor
    """
    color  = REGION_COLORS.get(region, '#444444')
    n_cols = len(PREDICTORS)
    fig, axes = plt.subplots(2, n_cols, figsize=(7 * n_cols, 8))
    fig.patch.set_facecolor(BG_COLOR)
    if n_cols == 1:
        axes = axes[:, np.newaxis]

    x     = np.arange(len(BANDS))
    width = 0.50

    for col_idx, pred in enumerate(PREDICTORS):
        for row_idx, model_key in enumerate(['Model1', 'Model2']):
            ax = axes[row_idx, col_idx]
            _style_ax(ax)

            betas, errors, pvals = [], [], []
            for band in BANDS:
                sub = all_results[
                    (all_results['region']                == region) &
                    (all_results['predictor_of_interest'] == pred) &
                    (all_results['model']                 == model_key) &
                    (all_results['band']                  == band) &
                    (all_results['col']                   == pred)
                ]
                if sub.empty:
                    betas.append(np.nan); errors.append(0); pvals.append(1.0)
                else:
                    r = sub.iloc[0]
                    betas.append(r['beta'])
                    errors.append(r['se'])
                    pvals.append(r.get('p_fdr', r['p_raw']))

            plot_betas  = [b if np.isfinite(b) else 0 for b in betas]
            plot_errors = [e if np.isfinite(betas[i]) else 0
                           for i, e in enumerate(errors)]

            bars = ax.bar(x, plot_betas, width,
                          color=[BAND_COLORS[b] for b in BANDS],
                          yerr=plot_errors,
                          error_kw=dict(ecolor=TEXT_COLOR, capsize=3,
                                        elinewidth=1),
                          alpha=0.82)

            for bar, beta, p in zip(bars, betas, pvals):
                if not np.isfinite(beta):
                    continue
                stars = sig_stars(p)
                if stars:
                    h    = bar.get_height()
                    sign = 1 if h >= 0 else -1
                    ax.text(bar.get_x() + bar.get_width() / 2,
                            h + sign * max(abs(h) * 0.05, 0.005),
                            stars, ha='center', va='bottom',
                            color=TEXT_COLOR, fontsize=9, fontweight='bold')

            ax.axhline(0, color=SPINE_COLOR, lw=1.0, ls='--')
            ax.set_xticks(x)
            ax.set_xticklabels([b.capitalize() for b in BANDS], fontsize=9)
            ax.set_xlabel('Frequency Band', fontsize=9)
            ax.set_ylabel('β', fontsize=9)
            ax.set_title(
                f"{region}  |  RSA_r ~ {PRED_LABELS[pred]}\n"
                f"[{MODEL_LABELS[model_key]}]",
                fontsize=9, fontweight='bold')

    fig.suptitle(
        f'{region}  |  RSA_r ~ T_lag / SP_lag  |  '
        f'β by Frequency Band  [{EXPERIMENT} / {PHASE}]',
        fontsize=12, fontweight='bold', y=1.01, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"  ✓ Beta bars → {save_path.name}")


# ============================================================================
# DATA LOADING
# ============================================================================

def load_data() -> Optional[pd.DataFrame]:
    fpath = (INPUT_DIR
             / f"ALL_SUBJECTS_{EXPERIMENT}_allregions_allbands_rsa_lag.csv")

    if not fpath.exists():
        print(f"  ✗ Master CSV not found: {fpath}")
        print("  Trying per-region fallback …")
        dfs = []
        for region in REGIONS:
            for band in BANDS:
                p = (INPUT_DIR /
                     f"ALL_SUBJECTS_{EXPERIMENT}_{region}_retrieval_{band}_rsa_lag.csv")
                if p.exists():
                    tmp = pd.read_csv(p)
                    dfs.append(tmp)
        if not dfs:
            print("  ✗ No fallback files found either.")
            return None
        df = pd.concat(dfs, ignore_index=True)
    else:
        print(f"  Loading {fpath.name} …")
        df = pd.read_csv(fpath)
        print(f"  Loaded {len(df):,} rows")

    # ── Filter to retrieval phase only ────────────────────────────────────────
    if 'phase' in df.columns:
        before = len(df)
        df     = df[df['phase'] == PHASE].copy()
        print(f"  Phase filter ({PHASE}): {before:,} → {len(df):,} rows")
    else:
        print("  WARNING: 'phase' column not found — assuming all rows are retrieval")

    # ── Filter to target regions ──────────────────────────────────────────────
    if 'region' in df.columns:
        df = df[df['region'].isin(REGIONS)].copy()
        print(f"  Region filter {REGIONS}: {len(df):,} rows")

    if df.empty:
        print("  ✗ No data after filtering.")
        return None

    # Prefix subjects with experiment to future-proof merges
    df['subject'] = (EXPERIMENT + '_'
                     + df['subject'].astype(str))

    print(f"\n  Total rows   : {len(df):,}")
    print(f"  Subjects     : {df['subject'].nunique()}")
    print(f"  Regions      : {sorted(df['region'].unique().tolist())}")
    print(f"  Bands        : {sorted(df['band'].unique().tolist())}")
    return df


# ============================================================================
# ANALYSIS LOOP  (one region × predictor × band)
# ============================================================================

def run_analysis_for_region_band(df: pd.DataFrame,
                                  region: str,
                                  predictor: str,
                                  band: str,
                                  has_semsim: bool
                                  ) -> Tuple[pd.DataFrame, str]:
    """
    Fits Model 1 & Model 2 for RSA_r ~ predictor within one
    region × frequency band (retrieval phase, single experiment).
    """
    sub = df[
        (df['region'] == region) &
        (df['band']   == band)
    ].copy()

    if sub.empty:
        return pd.DataFrame(), ""

    print(f"\n  ── {region}  |  RSA_r ~ {predictor}  |  {band.capitalize()} ──")
    print(f"     Rows: {len(sub):,}  |  Subjects: {sub['subject'].nunique()}")

    all_rows    = []
    text_blocks = [
        f"EXPERIMENT: {EXPERIMENT}  PHASE: {PHASE}  "
        f"REGION: {region}  PREDICTOR: {predictor}  BAND: {band}",
        f"N rows: {len(sub):,}  |  subjects: {sub['subject'].nunique()}\n",
    ]

    cross_cov   = CROSS_COVARIATE[predictor]
    cross_label = PRED_LABELS[cross_cov]

    def run_and_collect(real_cols, model_key, title, pred_display,
                        formula_rhs=None):
        res, _ = fit_lmm(sub, real_cols, model_key,
                         formula_rhs=formula_rhs)
        rows   = extract_rows(res, pred_display, model_key,
                              predictor, band, region)
        rows   = apply_fdr_within_model(rows)
        if not rows.empty:
            all_rows.append(rows)
            block = format_block(title, rows)
            text_blocks.append(block)
            print('\n' + block)
        return res

    # ── Model 1: bare — no covariates, no experiment dummy ───────────────────
    run_and_collect(
        [predictor],
        'Model1',
        f'Model 1 — RSA_r ~ {predictor}  [{region} / {band}]',
        {predictor: PRED_LABELS[predictor]},
    )

    # ── Model 2: controlled — cross-lag + output_lag [+ semantic_sim] ────────
    ctrl_cols = [predictor, cross_cov, 'output_lag']
    ctrl_disp = {
        predictor:   PRED_LABELS[predictor],
        cross_cov:   f'{cross_label}  [covariate]',
        'output_lag': 'output_lag',
    }
    if has_semsim:
        ctrl_cols = [predictor, cross_cov, 'output_lag', 'semantic_sim']
        ctrl_disp['semantic_sim'] = 'semantic_sim'

    run_and_collect(
        ctrl_cols,
        'Model2',
        f'Model 2 — RSA_r ~ {predictor} + {cross_cov} + controls  '
        f'[{region} / {band}]',
        ctrl_disp,
    )

    result_df = (pd.concat(all_rows, ignore_index=True)
                 if all_rows else pd.DataFrame())
    return result_df, '\n\n'.join(text_blocks)


# ============================================================================
# ENTRY POINT
# ============================================================================

def main():
    print(f"\n{'='*80}")
    print(f"RSA_r ~ T_lag / SP_lag  |  All 4 regions  |  "
          f"{EXPERIMENT}  |  {PHASE} phase")
    print(f"{'='*80}")

    df = load_data()
    if df is None or df.empty:
        print("No data loaded. Exiting.")
        return

    has_semsim = ('semantic_sim' in df.columns
                  and df['semantic_sim'].notna().any())
    print(f"\n  Outcome        : RSA_r")
    print(f"  Predictors     : {PREDICTORS}")
    print(f"  Bands          : {BANDS}")
    print(f"  Regions        : {REGIONS}")
    print(f"  semantic_sim   : {has_semsim}")

    ALL_RESULTS = []

    for region in REGIONS:
        region_results = []
        print(f"\n{'─'*80}")
        print(f"REGION: {region}")
        print(f"{'─'*80}")

        for predictor in PREDICTORS:
            for band in BANDS:
                res_df, text = run_analysis_for_region_band(
                    df, region, predictor, band, has_semsim)

                if not res_df.empty:
                    region_results.append(res_df)
                    ALL_RESULTS.append(res_df)

                tag      = f"{region}_{predictor}_{band}"
                csv_path = OUTPUT_DIR / f"LMM_{tag}_results.csv"
                txt_path = OUTPUT_DIR / f"LMM_{tag}_results.txt"
                if not res_df.empty:
                    res_df.to_csv(csv_path, index=False)
                    print(f"    ✓ {csv_path.name}")
                if text:
                    with open(txt_path, 'w') as f:
                        f.write(text)
                    print(f"    ✓ {txt_path.name}")

        # ── Per-region plots ──────────────────────────────────────────────────
        if region_results:
            reg_df = pd.concat(region_results, ignore_index=True)

            plot_summary_heatmap(
                reg_df, region,
                PLOT_DIR / f"summary_heatmap_{region}.png")
            plot_beta_bars(
                reg_df, region,
                PLOT_DIR / f"beta_bars_{region}.png")

            for predictor in PREDICTORS:
                for band in BANDS:
                    region_sub = df[
                        (df['region'] == region) &
                        (df['band']   == band)
                    ].copy()

                    plot_forest(
                        reg_df, region, predictor, band,
                        PLOT_DIR / f"forest_{region}_{predictor}_{band}.png")
                    plot_interaction(
                        region, predictor, band, region_sub,
                        PLOT_DIR / f"interaction_{region}_{predictor}_{band}.png")

    # ── Master CSV ────────────────────────────────────────────────────────────
    if ALL_RESULTS:
        master_df = pd.concat(ALL_RESULTS, ignore_index=True)
        master_path = OUTPUT_DIR / 'LMM_ALL_results.csv'
        master_df.to_csv(master_path, index=False)
        print(f"\n  ✓ Master results → {master_path.name}  "
              f"({len(master_df):,} rows)")

    print(f"\n{'='*80}")
    print(f"✓ ANALYSIS COMPLETE  [{EXPERIMENT} / {PHASE}]")
    print(f"  Results : {OUTPUT_DIR}")
    print(f"  Plots   : {PLOT_DIR}")
    print(f"{'='*80}")


if __name__ == '__main__':
    main()


RSA_r ~ T_lag / SP_lag  |  All 4 regions  |  DBOY1  |  encoding phase
  Loading ALL_SUBJECTS_DBOY1_allregions_allbands_rsa_lag.csv …
  Loaded 117,312 rows
  Phase filter (encoding): 117,312 → 29,328 rows
  Region filter ['LHP', 'RHP', 'LLTC', 'RLTC']: 29,328 rows

  Total rows   : 29,328
  Subjects     : 39
  Regions      : ['LHP', 'LLTC', 'RHP', 'RLTC']
  Bands        : ['alpha', 'beta', 'gamma', 'theta']

  Outcome        : RSA_r
  Predictors     : ['SP_lag']
  Bands          : ['theta', 'alpha', 'gamma']
  Regions        : ['LHP', 'RHP', 'LLTC', 'RLTC']
  semantic_sim   : True

────────────────────────────────────────────────────────────────────────────────
REGION: LHP
────────────────────────────────────────────────────────────────────────────────

  ── LHP  |  RSA_r ~ SP_lag  |  Theta ──
     Rows: 1,814  |  Subjects: 27
    [Model1] RSA_r ~ SP_lag  |  N=1,814
    [Model1] optimizer=lbfgs  converged=True  llf=-1014.8061  AIC=nan

Model 1 — RSA_r ~ SP_lag  [LHP / theta]
----------

In [12]:
#!/usr/bin/env python3
"""
RSA Lag — Sequential LMM Analysis (4 Steps)
============================================
Answers one question at a time:

  Step 1 — Does RSA_r track T_lag / SP_lag at all?
            RSA_r_z ~ predictor_z + experiment_dummy + (1|subj/sess)
            Pooled across all bands and both hemispheres.

  Step 2 — Is the effect theta-specific?
            RSA_r_z ~ T_lag_z + band_alpha + band_beta + band_gamma
                    + T_lag_z_x_alpha + T_lag_z_x_beta + T_lag_z_x_gamma
                    + experiment_dummy + (1|subj/sess)
            Pooled across both hemispheres.
            Run separately for T_lag and SP_lag.
            Key terms: T_lag_z_x_{band} — is that band's slope ≠ theta slope?

  Step 3 — Is the theta effect lateralized? (run only if Step 2 confirms theta)
            RSA_r_z ~ predictor_z + hemisphere_dummy + predictor_z_x_hemi
                    + experiment_dummy + (1|subj/sess)
            Theta band only.

  Step 4 — Do T_lag and SP_lag survive mutual control? (theta band only)
            RSA_r_z ~ T_lag_z + SP_lag_z + output_lag_z
                    + experiment_dummy + (1|subj/sess)

Z-scoring: RSA_r, T_lag, SP_lag, output_lag z-scored within (subject × band)
           before any analysis.  Betas are in SD units.

Input  : ./rsa_lag_hc_bands/ALL_SUBJECTS_{exp}_hc_allbands_rsa_lag.csv
Output : ./rsa_lag_hc_bands/SEQUENTIAL_LMM_results.csv
         ./rsa_lag_hc_bands/SEQUENTIAL_LMM_results.txt
"""

import warnings
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
from statsmodels.regression.mixed_linear_model import MixedLM
from statsmodels.stats.multitest import fdrcorrection

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

EXPERIMENTS = ['DBOY1', 'EFRCourierReadOnly', 'EFRCourierOpenLoop']
INPUT_DIR   = Path('./rsa_lag_hc_bands')
OUTPUT_DIR  = Path('./rsa_lag_hc_bands')

BANDS        = ['theta', 'alpha', 'beta', 'gamma']
NON_REF_BANDS = ['alpha', 'beta', 'gamma']   # theta = reference

OUTCOME = 'RSA_r'   # will be overwritten by RSA_r_z before fitting


# ============================================================================
# DATA LOADING
# ============================================================================

def load_data() -> Optional[pd.DataFrame]:
    dfs = []
    for exp in EXPERIMENTS:
        fpath = INPUT_DIR / f"ALL_SUBJECTS_{exp}_hc_allbands_rsa_lag.csv"
        if fpath.exists():
            tmp = pd.read_csv(fpath)
            tmp['experiment'] = exp
            dfs.append(tmp)
            print(f"  Loaded {fpath.name}  ({len(tmp):,} rows)")
        else:
            print(f"  ✗ No file for {exp} — skipping")

    if not dfs:
        return None

    df = pd.concat(dfs, ignore_index=True)

    # Ensure hemisphere column
    if 'hemisphere' not in df.columns:
        df['hemisphere'] = df['region'].map({'LHP': 'left', 'RHP': 'right'})

    # Prefix subjects to avoid cross-experiment collisions
    df['subject'] = df['experiment'].astype(str) + '_' + df['subject'].astype(str)

    # Dummies
    df['hemisphere_dummy'] = (df['hemisphere'] == 'right').astype(int)
    ref_exp = 'DBOY1' if 'DBOY1' in df['experiment'].unique() \
              else df['experiment'].unique()[0]
    df['experiment_dummy'] = (df['experiment'] != ref_exp).astype(int)

    print(f"\n  Total rows : {len(df):,}")
    print(f"  Subjects   : {df['subject'].nunique()}")
    print(f"  Bands      : {sorted(df['band'].unique().tolist())}")
    print(f"  Regions    : {sorted(df['region'].unique().tolist())}")
    return df


# ============================================================================
# Z-SCORING  (within subject × band)
# ============================================================================

def zscore(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    """Z-score each column within each (subject × band) group."""
    df = df.copy()
    def _z(x):
        s = x.std(ddof=1)
        return (x - x.mean()) / s if s > 0 else pd.Series(0.0, index=x.index)
    for col in cols:
        if col in df.columns:
            df[f'{col}_z'] = df.groupby(['subject', 'band'])[col].transform(_z)
    return df


# ============================================================================
# LMM FITTING
# ============================================================================

def fit_lmm(df: pd.DataFrame,
            formula_cols: List[str],
            formula_rhs: str,
            label: str) -> Optional[object]:
    """
    Fit RSA_r_z ~ formula_rhs.

    Random effects strategy (tried in order):
      1. Nested: (1|subject) + (1|subject:session)  via vc_formula
      2. Simple : (1|subject) only — fallback if nested hits a boundary
         (detected by subj_sess Var == 16 or NaN SE on variance components)

    formula_cols : real DataFrame column names used for dropna.
    """
    df = df.copy()
    df['subj_sess'] = df['subject'].astype(str) + '_' + df['session'].astype(str)

    keep = ['RSA_r_z'] + [c for c in formula_cols if c in df.columns] \
           + ['subject', 'subj_sess']
    df   = df[keep].dropna()

    n = len(df)
    if n < 20:
        print(f"  [{label}] Too few rows ({n}) — skipping")
        return None

    formula = f"RSA_r_z ~ {formula_rhs}"
    print(f"\n  [{label}] {formula}")
    print(f"  [{label}] N={n:,}  subjects={df['subject'].nunique()}")

    def _is_boundary(res) -> bool:
        """Return True if the nested random effect hit a numerical boundary."""
        if res is None or not np.isfinite(res.llf):
            return True
        # subj_sess Var == 16 or its SE is NaN are the tell-tale signs
        params = res.params
        bse    = res.bse
        for k in params.index:
            if 'subj_sess' in k.lower():
                if abs(params[k] - 16.0) < 1e-6:
                    return True
                if not np.isfinite(bse[k]):
                    return True
        return False

    # ── Attempt 1: nested random effects ─────────────────────────────────────
    result = None
    for method in ['lbfgs', 'nm', 'powell']:
        try:
            model = MixedLM.from_formula(
                formula,
                data       = df,
                groups     = df['subject'],
                vc_formula = {'subj_sess': '0 + C(subj_sess)'},
            )
            res = model.fit(reml=True, method=method)
            if np.isfinite(res.llf) and not _is_boundary(res):
                print(f"  [{label}] nested RE  optimizer={method}  "
                      f"converged={getattr(res, 'converged', None)}")
                result = res
                break
        except Exception as e:
            pass

    # ── Attempt 2: subject-only random effects (simpler, more stable) ─────────
    if result is None or _is_boundary(result):
        print(f"  [{label}] ⚠ Nested RE boundary/failed — "
              f"falling back to (1|subject) only")
        for method in ['lbfgs', 'nm', 'powell']:
            try:
                model = MixedLM.from_formula(
                    formula,
                    data   = df,
                    groups = df['subject'],
                )
                res = model.fit(reml=True, method=method)
                if np.isfinite(res.llf):
                    print(f"  [{label}] subject-only RE  optimizer={method}  "
                          f"converged={getattr(res, 'converged', None)}")
                    result = res
                    break
            except Exception as e:
                print(f"  [{label}] subject-only {method} failed: {e}")
                result = None

    if result is None:
        print(f"  [{label}] WARNING: all fits unsuccessful")
    return result


# ============================================================================
# RESULT FORMATTING
# ============================================================================

def sig_stars(p: float) -> str:
    return ('***' if p < 0.001 else '**' if p < 0.01 else
            '*'   if p < 0.05  else '†'  if p < 0.10  else '')


def result_to_df(result, step: str, model_desc: str,
                 hemisphere: str = 'combined') -> pd.DataFrame:
    """
    Extract all terms from a fitted result into a tidy DataFrame.
    FDR is applied only to fixed-effect terms with finite p-values.
    Random-effect variance rows (Intercept Var, subj_sess Var, etc.)
    are kept in the table but excluded from FDR correction.
    """
    if result is None:
        return pd.DataFrame()

    rows = []
    for term in result.params.index:
        p_val = result.pvalues[term]
        # Identify random-effect variance rows: SE or p is NaN
        is_re = (not np.isfinite(result.bse[term]) or
                 not np.isfinite(p_val) or
                 'var' in term.lower())
        rows.append({
            'step':       step,
            'model':      model_desc,
            'hemisphere': hemisphere,
            'term':       term,
            'beta':       result.params[term],
            'se':         result.bse[term],
            'z':          result.tvalues[term],
            'p_raw':      p_val,
            'p_fdr':      np.nan,        # filled below for fixed effects
            'is_re':      is_re,
            'nobs':       int(result.nobs),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # Apply FDR only to fixed-effect rows with finite p-values
    fe_mask = (~df['is_re']) & (df['p_raw'].notna()) & np.isfinite(df['p_raw'])
    if fe_mask.sum() > 0:
        _, fdr_vals = fdrcorrection(df.loc[fe_mask, 'p_raw'].values)
        df.loc[fe_mask, 'p_fdr'] = fdr_vals

    return df


def print_table(df: pd.DataFrame, title: str):
    if df.empty:
        print(f"\n  [no results for: {title}]")
        return

    sep  = '=' * 95
    sep2 = '-' * 95
    hdr  = (f"{'Term':<45} {'β':>8} {'SE':>8} {'z':>8} "
            f"{'p_raw':>10} {'p_fdr':>10} {'N':>8} {'':>5}")

    print(f"\n{sep}")
    print(title)
    print(sep2)
    print(hdr)
    print(sep2)

    # Fixed effects first
    fe = df[~df['is_re']] if 'is_re' in df.columns else df
    for _, row in fe.iterrows():
        p_fdr = row.get('p_fdr', np.nan)
        p_fdr_str = f"{p_fdr:>10.4f}" if np.isfinite(p_fdr) else f"{'—':>10}"
        print(f"  {row['term']:<43} {row['beta']:>8.4f} {row['se']:>8.4f} "
              f"{row['z']:>8.3f} {row['p_raw']:>10.4f} "
              f"{p_fdr_str} {int(row['nobs']):>8,} "
              f"{sig_stars(p_fdr) if np.isfinite(p_fdr) else '':>5}")

    # Random effects below separator (no FDR, just variance estimate)
    re = df[df['is_re']] if 'is_re' in df.columns else pd.DataFrame()
    if not re.empty:
        print(f"  {'— random effects —':<43}")
        for _, row in re.iterrows():
            beta_str = f"{row['beta']:>8.4f}" if np.isfinite(row['beta']) else f"{'—':>8}"
            print(f"  {row['term']:<43} {beta_str}")

    print(sep2)
    print("  FDR: BH within fixed effects only.  "
          "† p<.10  * p<.05  ** p<.01  *** p<.001")
    print(f"  Reference: band=theta, hemisphere=LHP, experiment=DBOY1")
    print(sep)


# ============================================================================
# STEP 1 — Does the effect exist?
# ============================================================================

def step1(df: pd.DataFrame) -> List[pd.DataFrame]:
    """
    RSA_r_z ~ predictor_z + output_lag_z + experiment_dummy + (1|subj/sess)
    output_lag_z included as covariate to control for recall sequence position.
    Pooled across all bands and both hemispheres.
    Run for T_lag and SP_lag separately.
    """
    print(f"\n{'#'*70}")
    print("STEP 1 — Does RSA_r track T_lag / SP_lag at all?")
    print(f"{'#'*70}")
    print("  Data: all bands pooled, both hemispheres")
    print("  Covariate: output_lag_z (recall sequence position)")

    results = []
    for pred in ['T_lag', 'SP_lag']:
        pred_z = f'{pred}_z'
        res = fit_lmm(
            df,
            formula_cols = [pred_z, 'output_lag_z', 'experiment_dummy'],
            formula_rhs  = f'{pred_z} + output_lag_z + experiment_dummy',
            label        = f'Step1 {pred}',
        )
        rdf = result_to_df(res, step='Step1',
                           model_desc=f'RSA_r_z ~ {pred_z} + output_lag_z')
        print_table(rdf,
                    f"Step 1 | RSA_r_z ~ {pred_z} + output_lag_z + experiment"
                    f"  [all bands, combined]")
        results.append(rdf)

    return results


# ============================================================================
# STEP 2 — Is the effect theta-specific?
# ============================================================================

def step2(df: pd.DataFrame) -> List[pd.DataFrame]:
    """
    RSA_r_z ~ predictor_z
            + band_alpha + band_beta + band_gamma          (band main effects)
            + predictor_z_x_alpha + _x_beta + _x_gamma    (pred × band interactions)
            + experiment_dummy
            + (1|subj/sess)

    theta is the reference band.
    Interaction terms = "is this band's slope different from theta?"
    Run for T_lag and SP_lag separately, both hemispheres pooled.
    """
    print(f"\n{'#'*70}")
    print("STEP 2 — Is the effect theta-specific?")
    print(f"{'#'*70}")
    print("  Data: all bands pooled, both hemispheres")
    print("  Reference band: theta")
    print("  KEY TERMS: predictor_z_x_{{alpha/beta/gamma}}")
    print("             significant → that band's slope ≠ theta slope")

    # Pre-compute band dummies (theta omitted = reference)
    df = df.copy()
    for band in NON_REF_BANDS:
        df[f'band_{band}'] = (df['band'] == band).astype(float)

    results = []
    for pred in ['T_lag', 'SP_lag']:
        pred_z = f'{pred}_z'

        # Pre-compute predictor × band interaction columns
        for band in NON_REF_BANDS:
            df[f'{pred_z}_x_{band}'] = df[pred_z] * df[f'band_{band}']

        band_cols  = [f'band_{b}'       for b in NON_REF_BANDS]
        inter_cols = [f'{pred_z}_x_{b}' for b in NON_REF_BANDS]

        formula_cols = [pred_z] + band_cols + inter_cols \
                       + ['output_lag_z', 'experiment_dummy']
        formula_rhs  = (f'{pred_z} + '
                        + ' + '.join(band_cols) + ' + '
                        + ' + '.join(inter_cols)
                        + ' + output_lag_z + experiment_dummy')

        res = fit_lmm(df, formula_cols, formula_rhs,
                      label=f'Step2 {pred}')
        rdf = result_to_df(res, step='Step2',
                           model_desc=f'RSA_r_z ~ {pred_z} * band + output_lag_z')
        print_table(rdf,
                    f"Step 2 | RSA_r_z ~ {pred_z} * band + output_lag_z + experiment  "
                    f"[all bands, combined]\n"
                    f"  Interaction terms test: is band X slope ≠ theta slope?")
        results.append(rdf)

    return results


# ============================================================================
# STEP 3 — Is the theta effect lateralized?
# ============================================================================

def step3(df: pd.DataFrame) -> List[pd.DataFrame]:
    """
    Theta band only.
    RSA_r_z ~ predictor_z + hemisphere_dummy + predictor_z_x_hemi
            + experiment_dummy + (1|subj/sess)

    Run for T_lag and SP_lag separately.
    KEY TERM: predictor_z_x_hemi
              significant → LHP and RHP slopes are different
    Also run separately on LHP and RHP for simple slopes.
    """
    print(f"\n{'#'*70}")
    print("STEP 3 — Is the theta effect lateralized?  (theta band only)")
    print(f"{'#'*70}")

    theta_df = df[df['band'] == 'theta'].copy()
    print(f"  Theta rows: {len(theta_df):,}  "
          f"subjects: {theta_df['subject'].nunique()}")

    results = []
    for pred in ['T_lag', 'SP_lag']:
        pred_z     = f'{pred}_z'
        inter_col  = f'{pred_z}_x_hemi'
        theta_df[inter_col] = theta_df[pred_z] * theta_df['hemisphere_dummy']

        # Combined: interaction test
        res = fit_lmm(
            theta_df,
            formula_cols = [pred_z, 'hemisphere_dummy', inter_col,
                            'output_lag_z', 'experiment_dummy'],
            formula_rhs  = (f'{pred_z} + hemisphere_dummy + {inter_col}'
                            f' + output_lag_z + experiment_dummy'),
            label        = f'Step3 {pred} combined',
        )
        rdf = result_to_df(res, step='Step3',
                           model_desc=f'RSA_r_z ~ {pred_z} * hemisphere + output_lag_z',
                           hemisphere='combined')
        print_table(rdf,
                    f"Step 3 | RSA_r_z ~ {pred_z} * hemisphere + output_lag_z  "
                    f"[theta, combined]\n"
                    f"  KEY TERM: {inter_col}  "
                    f"(significant → LHP slope ≠ RHP slope)")
        results.append(rdf)

        # Simple slopes: LHP and RHP separately
        for hemi in ['left', 'right']:
            hemi_label = 'LHP' if hemi == 'left' else 'RHP'
            hd = theta_df[theta_df['hemisphere'] == hemi].copy()
            res_h = fit_lmm(
                hd,
                formula_cols = [pred_z, 'output_lag_z', 'experiment_dummy'],
                formula_rhs  = f'{pred_z} + output_lag_z + experiment_dummy',
                label        = f'Step3 {pred} {hemi_label}',
            )
            rdf_h = result_to_df(res_h, step='Step3',
                                 model_desc=f'RSA_r_z ~ {pred_z} + output_lag_z',
                                 hemisphere=hemi_label)
            print_table(rdf_h,
                        f"Step 3 | RSA_r_z ~ {pred_z} + output_lag_z + experiment  "
                        f"[theta, {hemi_label} only]")
            results.append(rdf_h)

    return results


# ============================================================================
# STEP 4 — Do T_lag and SP_lag survive mutual control?
# ============================================================================

def step4(df: pd.DataFrame) -> List[pd.DataFrame]:
    """
    Theta band only.
    RSA_r_z ~ T_lag_z + SP_lag_z + output_lag_z + experiment_dummy
            + (1|subj/sess)

    Both predictors in the same model — tests independence.
    Run on combined, LHP only, RHP only.
    """
    print(f"\n{'#'*70}")
    print("STEP 4 — Do T_lag and SP_lag survive mutual control?  (theta only)")
    print(f"{'#'*70}")

    theta_df = df[df['band'] == 'theta'].copy()
    print(f"  Theta rows: {len(theta_df):,}  "
          f"subjects: {theta_df['subject'].nunique()}")

    formula_cols = ['T_lag_z', 'SP_lag_z', 'output_lag_z', 'experiment_dummy']
    formula_rhs  = 'T_lag_z + SP_lag_z + output_lag_z + experiment_dummy'

    results = []
    for subset_label, subset_df in [('combined', theta_df),
                                     ('LHP', theta_df[theta_df['hemisphere'] == 'left']),
                                     ('RHP', theta_df[theta_df['hemisphere'] == 'right'])]:
        res = fit_lmm(
            subset_df,
            formula_cols = formula_cols,
            formula_rhs  = formula_rhs,
            label        = f'Step4 {subset_label}',
        )
        rdf = result_to_df(res, step='Step4',
                           model_desc='RSA_r_z ~ T_lag_z + SP_lag_z + output_lag_z',
                           hemisphere=subset_label)
        print_table(rdf,
                    f"Step 4 | RSA_r_z ~ T_lag_z + SP_lag_z + output_lag_z  "
                    f"[theta, {subset_label}]")
        results.append(rdf)

    return results


# ============================================================================
# ENTRY POINT
# ============================================================================

def main():
    print(f"\n{'='*70}")
    print("RSA LAG — SEQUENTIAL LMM ANALYSIS")
    print(f"{'='*70}")

    # ── Load ──────────────────────────────────────────────────────────────────
    df = load_data()
    if df is None or df.empty:
        print("No data loaded. Exiting.")
        return

    # ── Z-score within (subject × band) ──────────────────────────────────────
    df = zscore(df, ['RSA_r', 'T_lag', 'SP_lag', 'output_lag'])
    print(f"\n  Z-scored within (subject × band): RSA_r, T_lag, SP_lag, output_lag")

    has_semsim = ('semantic_sim' in df.columns and
                  df['semantic_sim'].notna().any())
    print(f"  semantic_sim available: {has_semsim}")

    # ── Run steps ─────────────────────────────────────────────────────────────
    all_results = []

    all_results += step1(df)
    all_results += step2(df)
    all_results += step3(df)
    all_results += step4(df)

    # ── Save ──────────────────────────────────────────────────────────────────
    if all_results:
        master = pd.concat(all_results, ignore_index=True)
        csv_path = OUTPUT_DIR / 'SEQUENTIAL_LMM_results.csv'
        master.to_csv(csv_path, index=False)
        print(f"\n  ✓ Results saved → {csv_path}")
        print(f"  Total result rows: {len(master):,}")

    print(f"\n{'='*70}")
    print("INTERPRETATION GUIDE")
    print(f"{'='*70}")
    print("""
  All models include output_lag_z as a covariate to control for recall
  sequence position (pairs recalled consecutively have small output_lag
  and tend to also have small T_lag/SP_lag — must be regressed out first).

  Step 1: If T_lag_z and/or SP_lag_z are significant after controlling for
          output_lag_z → a genuine clustering effect exists, proceed.
          If not → the effect was driven by output sequence, not spatial/temporal coding.

  Step 2: Look at the interaction terms (predictor_z_x_alpha/beta/gamma).
          If NOT significant → cannot claim theta specificity.
          If significant (negative) → theta slope is larger than other bands
          → theta specificity formally supported.

  Step 3: Look at predictor_z_x_hemi.
          If significant → LHP and RHP slopes are genuinely different.
          Simple slopes (LHP/RHP separately) show the direction.

  Step 4: Both T_lag_z and SP_lag_z in the same model with output_lag_z.
          If both survive → temporal and spatial clustering are INDEPENDENT.
          If one drops out → it was redundant with the other.
    """)


if __name__ == '__main__':
    main()


RSA LAG — SEQUENTIAL LMM ANALYSIS
  Loaded ALL_SUBJECTS_DBOY1_hc_allbands_rsa_lag.csv  (13,028 rows)
  Loaded ALL_SUBJECTS_EFRCourierReadOnly_hc_allbands_rsa_lag.csv  (1,712 rows)
  Loaded ALL_SUBJECTS_EFRCourierOpenLoop_hc_allbands_rsa_lag.csv  (1,276 rows)

  Total rows : 16,016
  Subjects   : 44
  Bands      : ['alpha', 'beta', 'gamma', 'theta']
  Regions    : ['LHP', 'RHP']

  Z-scored within (subject × band): RSA_r, T_lag, SP_lag, output_lag
  semantic_sim available: True

######################################################################
STEP 1 — Does RSA_r track T_lag / SP_lag at all?
######################################################################
  Data: all bands pooled, both hemispheres
  Covariate: output_lag_z (recall sequence position)

  [Step1 T_lag] RSA_r_z ~ T_lag_z + output_lag_z + experiment_dummy
  [Step1 T_lag] N=16,016  subjects=44
  [Step1 T_lag] nested RE  optimizer=nm  converged=True

Step 1 | RSA_r_z ~ T_lag_z + output_lag_z + experiment  [all band

In [20]:
#!/usr/bin/env python3
"""
RSA Lag — Sequential LMM Analysis (4 Steps) — RAW SCALE (no z-scoring)
========================================================================
Identical logic to rsa_lag_sequential_lmm.py but predictors and outcome
are kept on their original scales.  Between-subject baseline differences
are absorbed by the random intercept (1|subject).

Betas are in original units:
  RSA_r   : Pearson r  (roughly −1 to +1, typically −0.3 to +0.3)
  T_lag   : number of list positions separating the two items
  SP_lag  : Euclidean distance in store-coordinate units
  output_lag : number of recall output positions between the two items

Why run without z-scoring?
  - Betas are directly interpretable in substantive units
  - No assumption that within-subject SD is comparable across subjects/bands
  - Avoids the zero-variance group problem for output_lag (values 1–4)
  - Random intercept per subject already removes between-subject mean differences

Trade-off: betas are NOT comparable across predictors (T_lag vs SP_lag
have different scales) or across bands (if band means differ).

Steps
-----
  Step 1 — RSA_r ~ predictor + output_lag + experiment + (1|subj)
            All bands pooled, both hemispheres.

  Step 2 — RSA_r ~ predictor + band_alpha + band_beta + band_gamma
                 + predictor_x_alpha + predictor_x_beta + predictor_x_gamma
                 + output_lag + experiment + (1|subj)
            Tests whether predictor slope differs across bands
            (theta = reference level).

  Step 3 — RSA_r ~ predictor + hemisphere_dummy + predictor_x_hemi
                 + output_lag + experiment + (1|subj)
            Theta band only.  Tests LHP vs RHP asymmetry.
            Also run separately for LHP and RHP.

  Step 4 — RSA_r ~ T_lag + SP_lag + output_lag + experiment + (1|subj)
            Theta band only.  Tests independence of T_lag and SP_lag.

Input  : ./rsa_lag_hc_bands/ALL_SUBJECTS_{exp}_hc_allbands_rsa_lag.csv
Output : ./rsa_lag_hc_bands/SEQUENTIAL_LMM_RAW_results.csv
         ./rsa_lag_hc_bands/SEQUENTIAL_LMM_RAW_results.txt
"""

import warnings
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
from statsmodels.regression.mixed_linear_model import MixedLM
from statsmodels.stats.multitest import fdrcorrection

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

EXPERIMENTS   = ['DBOY1', 'EFRCourierReadOnly']
INPUT_DIR     = Path('./rsa_lag_hc_bands')
OUTPUT_DIR    = Path('./rsa_lag_hc_bands')

BANDS         = ['theta', 'alpha', 'beta', 'gamma']
NON_REF_BANDS = ['alpha', 'beta', 'gamma']   # theta = reference

OUTCOME = 'RSA_r'   # raw, no z-scoring


# ============================================================================
# DATA LOADING
# ============================================================================

def load_data() -> Optional[pd.DataFrame]:
    dfs = []
    for exp in EXPERIMENTS:
        fpath = INPUT_DIR / f"ALL_SUBJECTS_{exp}_hc_allbands_rsa_lag.csv"
        if fpath.exists():
            tmp = pd.read_csv(fpath)
            tmp['experiment'] = exp
            dfs.append(tmp)
            print(f"  Loaded {fpath.name}  ({len(tmp):,} rows)")
        else:
            print(f"  ✗ No file for {exp} — skipping")

    if not dfs:
        return None

    df = pd.concat(dfs, ignore_index=True)

    if 'hemisphere' not in df.columns:
        df['hemisphere'] = df['region'].map({'LHP': 'left', 'RHP': 'right'})

    df['subject'] = df['experiment'].astype(str) + '_' + df['subject'].astype(str)

    df['hemisphere_dummy'] = (df['hemisphere'] == 'right').astype(int)
    ref_exp = 'DBOY1' if 'DBOY1' in df['experiment'].unique() \
              else df['experiment'].unique()[0]
    df['experiment_dummy'] = (df['experiment'] != ref_exp).astype(int)

    print(f"\n  Total rows : {len(df):,}")
    print(f"  Subjects   : {df['subject'].nunique()}")
    print(f"  Bands      : {sorted(df['band'].unique().tolist())}")
    print(f"  Regions    : {sorted(df['region'].unique().tolist())}")

    # Descriptive stats so reader knows raw scales
    print(f"\n  Predictor scales (raw):")
    for col in ['RSA_r', 'T_lag', 'SP_lag', 'output_lag']:
        if col in df.columns:
            s = df[col].dropna()
            print(f"    {col:<12} mean={s.mean():.3f}  "
                  f"sd={s.std():.3f}  "
                  f"range=[{s.min():.2f}, {s.max():.2f}]")

    return df


# ============================================================================
# LMM FITTING
# ============================================================================

def fit_lmm(df: pd.DataFrame,
            formula_cols: List[str],
            formula_rhs: str,
            label: str) -> Optional[object]:
    """
    Fit RSA_r ~ formula_rhs.

    Random effects strategy (tried in order):
      1. Nested: (1|subject) + (1|subject:session)  via vc_formula
      2. Simple : (1|subject) only — fallback if nested hits a boundary
    """
    df = df.copy()
    df['subj_sess'] = df['subject'].astype(str) + '_' + df['session'].astype(str)

    keep = [OUTCOME] + [c for c in formula_cols if c in df.columns] \
           + ['subject', 'subj_sess']
    df   = df[keep].dropna()

    n = len(df)
    if n < 20:
        print(f"  [{label}] Too few rows ({n}) — skipping")
        return None

    formula = f"{OUTCOME} ~ {formula_rhs}"
    print(f"\n  [{label}] {formula}")
    print(f"  [{label}] N={n:,}  subjects={df['subject'].nunique()}")

    def _is_boundary(res) -> bool:
        if res is None or not np.isfinite(res.llf):
            return True
        for k in res.params.index:
            if 'subj_sess' in k.lower():
                if abs(res.params[k] - 16.0) < 1e-6:
                    return True
                if not np.isfinite(res.bse[k]):
                    return True
        return False

    # Attempt 1: nested random effects
    result = None
    for method in ['lbfgs', 'nm', 'powell']:
        try:
            model = MixedLM.from_formula(
                formula,
                data       = df,
                groups     = df['subject'],
                vc_formula = {'subj_sess': '0 + C(subj_sess)'},
            )
            res = model.fit(reml=True, method=method)
            if np.isfinite(res.llf) and not _is_boundary(res):
                print(f"  [{label}] nested RE  optimizer={method}  "
                      f"converged={getattr(res, 'converged', None)}")
                result = res
                break
        except Exception:
            pass

    # Attempt 2: subject-only random effects
    if result is None or _is_boundary(result):
        print(f"  [{label}] ⚠ Nested RE boundary — "
              f"falling back to (1|subject) only")
        for method in ['lbfgs', 'nm', 'powell']:
            try:
                model = MixedLM.from_formula(
                    formula,
                    data   = df,
                    groups = df['subject'],
                )
                res = model.fit(reml=True, method=method)
                if np.isfinite(res.llf):
                    print(f"  [{label}] subject-only RE  optimizer={method}  "
                          f"converged={getattr(res, 'converged', None)}")
                    result = res
                    break
            except Exception as e:
                print(f"  [{label}] subject-only {method} failed: {e}")
                result = None

    if result is None:
        print(f"  [{label}] WARNING: all fits unsuccessful")
    return result


# ============================================================================
# RESULT FORMATTING
# ============================================================================

def sig_stars(p: float) -> str:
    return ('***' if p < 0.001 else '**' if p < 0.01 else
            '*'   if p < 0.05  else '†'  if p < 0.10  else '')


def result_to_df(result, step: str, model_desc: str,
                 hemisphere: str = 'combined') -> pd.DataFrame:
    """
    Extract all terms.  FDR applied only to fixed-effect rows
    with finite p-values.  Random-effect rows tagged with is_re=True.
    """
    if result is None:
        return pd.DataFrame()

    rows = []
    for term in result.params.index:
        p_val = result.pvalues[term]
        is_re = (not np.isfinite(result.bse[term]) or
                 not np.isfinite(p_val) or
                 'var' in term.lower())
        rows.append({
            'step':       step,
            'model':      model_desc,
            'hemisphere': hemisphere,
            'term':       term,
            'beta':       result.params[term],
            'se':         result.bse[term],
            'z':          result.tvalues[term],
            'p_raw':      p_val,
            'p_fdr':      np.nan,
            'is_re':      is_re,
            'nobs':       int(result.nobs),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    fe_mask = (~df['is_re']) & df['p_raw'].notna() & np.isfinite(df['p_raw'])
    if fe_mask.sum() > 0:
        _, fdr_vals = fdrcorrection(df.loc[fe_mask, 'p_raw'].values)
        df.loc[fe_mask, 'p_fdr'] = fdr_vals

    return df


def print_table(df: pd.DataFrame, title: str):
    if df.empty:
        print(f"\n  [no results for: {title}]")
        return

    sep  = '=' * 95
    sep2 = '-' * 95
    hdr  = (f"{'Term':<45} {'β':>10} {'SE':>10} {'z':>8} "
            f"{'p_raw':>10} {'p_fdr':>10} {'N':>8}")

    print(f"\n{sep}")
    print(title)
    print(sep2)
    print(hdr)
    print(sep2)

    fe = df[~df['is_re']] if 'is_re' in df.columns else df
    for _, row in fe.iterrows():
        p_fdr = row.get('p_fdr', np.nan)
        p_fdr_str = f"{p_fdr:>10.4f}" if np.isfinite(p_fdr) else f"{'—':>10}"
        stars = sig_stars(p_fdr) if np.isfinite(p_fdr) else ''
        print(f"  {row['term']:<43} {row['beta']:>10.5f} {row['se']:>10.5f} "
              f"{row['z']:>8.3f} {row['p_raw']:>10.4f} "
              f"{p_fdr_str} {int(row['nobs']):>8,}  {stars}")

    re = df[df['is_re']] if 'is_re' in df.columns else pd.DataFrame()
    if not re.empty:
        print(f"  {'— random effects —'}")
        for _, row in re.iterrows():
            beta_str = f"{row['beta']:>10.5f}" if np.isfinite(row['beta']) else f"{'—':>10}"
            print(f"  {row['term']:<43} {beta_str}")

    print(sep2)
    print("  Outcome = RSA_r (raw Pearson r).  "
          "Predictors on raw scale (T_lag: list positions, "
          "SP_lag: distance units, output_lag: recall positions).")
    print("  FDR: BH within fixed effects only.  "
          "† p<.10  * p<.05  ** p<.01  *** p<.001")
    print(f"  Reference: band=theta, hemisphere=LHP, experiment=DBOY1")
    print(sep)


# ============================================================================
# STEP 1 — Does the effect exist?
# ============================================================================

def step1(df: pd.DataFrame) -> List[pd.DataFrame]:
    """
    RSA_r ~ predictor + output_lag + experiment_dummy + (1|subj)
    All bands pooled, both hemispheres.
    """
    print(f"\n{'#'*70}")
    print("STEP 1 — Does RSA_r track T_lag / SP_lag at all?")
    print(f"{'#'*70}")
    print("  Data: all bands pooled, both hemispheres")
    print("  Covariate: output_lag (raw, not z-scored)")

    results = []
    for pred in ['T_lag', 'SP_lag']:
        res = fit_lmm(
            df,
            formula_cols = [pred, 'output_lag', 'experiment_dummy'],
            formula_rhs  = f'{pred} + output_lag + experiment_dummy',
            label        = f'Step1 {pred}',
        )
        rdf = result_to_df(res, step='Step1',
                           model_desc=f'RSA_r ~ {pred} + output_lag')
        print_table(rdf,
                    f"Step 1 | RSA_r ~ {pred} + output_lag + experiment"
                    f"  [all bands, combined]")
        results.append(rdf)

    return results


# ============================================================================
# STEP 2 — Is the effect theta-specific?
# ============================================================================

def step2(df: pd.DataFrame) -> List[pd.DataFrame]:
    """
    RSA_r ~ predictor
           + band_alpha + band_beta + band_gamma
           + predictor_x_alpha + predictor_x_beta + predictor_x_gamma
           + output_lag + experiment_dummy + (1|subj)

    Interaction terms = "is this band's slope ≠ theta slope?"
    """
    print(f"\n{'#'*70}")
    print("STEP 2 — Is the effect theta-specific?")
    print(f"{'#'*70}")
    print("  Data: all bands pooled, both hemispheres")
    print("  Reference band: theta")
    print("  KEY TERMS: predictor_x_{{alpha/beta/gamma}}")
    print("             significant → that band's slope ≠ theta slope")

    df = df.copy()
    for band in NON_REF_BANDS:
        df[f'band_{band}'] = (df['band'] == band).astype(float)

    results = []
    for pred in ['T_lag', 'SP_lag']:

        for band in NON_REF_BANDS:
            df[f'{pred}_x_{band}'] = df[pred] * df[f'band_{band}']

        band_cols  = [f'band_{b}'     for b in NON_REF_BANDS]
        inter_cols = [f'{pred}_x_{b}' for b in NON_REF_BANDS]

        formula_cols = [pred] + band_cols + inter_cols \
                       + ['output_lag', 'experiment_dummy']
        formula_rhs  = (f'{pred} + '
                        + ' + '.join(band_cols) + ' + '
                        + ' + '.join(inter_cols)
                        + ' + output_lag + experiment_dummy')

        res = fit_lmm(df, formula_cols, formula_rhs,
                      label=f'Step2 {pred}')
        rdf = result_to_df(res, step='Step2',
                           model_desc=f'RSA_r ~ {pred} * band + output_lag')
        print_table(rdf,
                    f"Step 2 | RSA_r ~ {pred} * band + output_lag + experiment"
                    f"  [all bands, combined]\n"
                    f"  KEY: {pred}_x_{{band}} significant → slope ≠ theta")
        results.append(rdf)

    return results


# ============================================================================
# STEP 3 — Is the theta effect lateralized?
# ============================================================================

def step3(df: pd.DataFrame) -> List[pd.DataFrame]:
    """
    Theta band only.
    RSA_r ~ predictor + hemisphere_dummy + predictor_x_hemi
           + output_lag + experiment_dummy + (1|subj)
    Also simple slopes for LHP and RHP separately.
    """
    print(f"\n{'#'*70}")
    print("STEP 3 — Is the theta effect lateralized?  (theta band only)")
    print(f"{'#'*70}")

    theta_df = df[df['band'] == 'theta'].copy()
    print(f"  Theta rows: {len(theta_df):,}  "
          f"subjects: {theta_df['subject'].nunique()}")

    results = []
    for pred in ['T_lag', 'SP_lag']:
        inter_col = f'{pred}_x_hemi'
        theta_df[inter_col] = theta_df[pred] * theta_df['hemisphere_dummy']

        # Combined: interaction test
        res = fit_lmm(
            theta_df,
            formula_cols = [pred, 'hemisphere_dummy', inter_col,
                            'output_lag', 'experiment_dummy'],
            formula_rhs  = (f'{pred} + hemisphere_dummy + {inter_col}'
                            f' + output_lag + experiment_dummy'),
            label        = f'Step3 {pred} combined',
        )
        rdf = result_to_df(res, step='Step3',
                           model_desc=f'RSA_r ~ {pred} * hemisphere + output_lag',
                           hemisphere='combined')
        print_table(rdf,
                    f"Step 3 | RSA_r ~ {pred} * hemisphere + output_lag  "
                    f"[theta, combined]\n"
                    f"  KEY TERM: {inter_col}  "
                    f"(significant → LHP slope ≠ RHP slope)")
        results.append(rdf)

        # Simple slopes: LHP and RHP separately
        for hemi, hemi_label in [('left', 'LHP'), ('right', 'RHP')]:
            hd = theta_df[theta_df['hemisphere'] == hemi].copy()
            res_h = fit_lmm(
                hd,
                formula_cols = [pred, 'output_lag', 'experiment_dummy'],
                formula_rhs  = f'{pred} + output_lag + experiment_dummy',
                label        = f'Step3 {pred} {hemi_label}',
            )
            rdf_h = result_to_df(res_h, step='Step3',
                                 model_desc=f'RSA_r ~ {pred} + output_lag',
                                 hemisphere=hemi_label)
            print_table(rdf_h,
                        f"Step 3 | RSA_r ~ {pred} + output_lag + experiment  "
                        f"[theta, {hemi_label} only]")
            results.append(rdf_h)

    return results


# ============================================================================
# STEP 4 — Do T_lag and SP_lag survive mutual control?
# ============================================================================

def step4(df: pd.DataFrame) -> List[pd.DataFrame]:
    """
    Theta band only.
    RSA_r ~ T_lag + SP_lag + output_lag + experiment_dummy + (1|subj)
    Combined, LHP, RHP.
    """
    print(f"\n{'#'*70}")
    print("STEP 4 — Do T_lag and SP_lag survive mutual control?  (theta only)")
    print(f"{'#'*70}")

    theta_df = df[df['band'] == 'theta'].copy()
    print(f"  Theta rows: {len(theta_df):,}  "
          f"subjects: {theta_df['subject'].nunique()}")

    formula_cols = ['T_lag', 'SP_lag', 'output_lag', 'experiment_dummy']
    formula_rhs  = 'T_lag + SP_lag + output_lag + experiment_dummy'

    results = []
    for subset_label, subset_df in [
            ('combined', theta_df),
            ('LHP',      theta_df[theta_df['hemisphere'] == 'left']),
            ('RHP',      theta_df[theta_df['hemisphere'] == 'right'])]:

        res = fit_lmm(
            subset_df,
            formula_cols = formula_cols,
            formula_rhs  = formula_rhs,
            label        = f'Step4 {subset_label}',
        )
        rdf = result_to_df(res, step='Step4',
                           model_desc='RSA_r ~ T_lag + SP_lag + output_lag',
                           hemisphere=subset_label)
        print_table(rdf,
                    f"Step 4 | RSA_r ~ T_lag + SP_lag + output_lag  "
                    f"[theta, {subset_label}]")
        results.append(rdf)

    return results


# ============================================================================
# ENTRY POINT
# ============================================================================

def main():
    print(f"\n{'='*70}")
    print("RSA LAG — SEQUENTIAL LMM ANALYSIS  (RAW SCALE, no z-scoring)")
    print(f"{'='*70}")

    df = load_data()
    if df is None or df.empty:
        print("No data loaded. Exiting.")
        return

    print(f"\n  Note: no z-scoring applied. Betas are in raw units.")
    print(f"  Between-subject differences absorbed by (1|subject) random intercept.")

    all_results = []
    all_results += step1(df)
    all_results += step2(df)
    all_results += step3(df)
    all_results += step4(df)

    if all_results:
        master = pd.concat(all_results, ignore_index=True)
        csv_path = OUTPUT_DIR / 'SEQUENTIAL_LMM_RAW_results.csv'
        master.to_csv(csv_path, index=False)
        print(f"\n  ✓ Results saved → {csv_path}")
        print(f"  Total result rows: {len(master):,}")

    print(f"\n{'='*70}")
    print("INTERPRETATION GUIDE")
    print(f"{'='*70}")
    print("""
  Betas are in raw units — not SD units.  Interpretation:
    T_lag   β : change in RSA_r per additional list-position gap
    SP_lag  β : change in RSA_r per additional store-distance unit
    output_lag β : change in RSA_r per additional recall-position gap

  Step 1: T_lag/SP_lag significant after controlling for output_lag?
          → a genuine clustering effect exists, not just recall order.

  Step 2: predictor_x_alpha/beta/gamma significant?
          → theta slope differs from that band (formal specificity test).
          NOT significant → effect is not theta-specific.

  Step 3: predictor_x_hemi significant?
          → LHP and RHP slopes are genuinely different.
          Simple slopes show which hemisphere drives the effect.

  Step 4: Both T_lag and SP_lag survive together?
          → temporal and spatial clustering are INDEPENDENT contributions.
    """)

    # ── Save text output ──────────────────────────────────────────────────────
    txt_path = OUTPUT_DIR / 'SEQUENTIAL_LMM_RAW_results.txt'
    if all_results:
        lines = []
        for rdf in all_results:
            if rdf.empty:
                continue
            step = rdf['step'].iloc[0]
            model = rdf['model'].iloc[0]
            hemi  = rdf['hemisphere'].iloc[0]
            lines.append(f"\n{'='*80}")
            lines.append(f"{step}  |  {model}  |  {hemi}")
            lines.append(f"{'='*80}")
            fe = rdf[~rdf['is_re']]
            for _, row in fe.iterrows():
                p_fdr = row['p_fdr']
                stars = sig_stars(p_fdr) if np.isfinite(p_fdr) else ''
                lines.append(
                    f"  {row['term']:<40} β={row['beta']:>10.5f}  "
                    f"SE={row['se']:>9.5f}  z={row['z']:>7.3f}  "
                    f"p={row['p_raw']:>8.4f}  "
                    f"p_fdr={p_fdr:>8.4f}  {stars}"
                    if np.isfinite(p_fdr) else
                    f"  {row['term']:<40} β={row['beta']:>10.5f}  "
                    f"SE={row['se']:>9.5f}  z={row['z']:>7.3f}  "
                    f"p={row['p_raw']:>8.4f}"
                )
        with open(txt_path, 'w') as f:
            f.write('\n'.join(lines))
        print(f"  ✓ Text output   → {txt_path}")


if __name__ == '__main__':
    main()


RSA LAG — SEQUENTIAL LMM ANALYSIS  (RAW SCALE, no z-scoring)
  Loaded ALL_SUBJECTS_DBOY1_hc_allbands_rsa_lag.csv  (13,028 rows)
  Loaded ALL_SUBJECTS_EFRCourierReadOnly_hc_allbands_rsa_lag.csv  (1,712 rows)

  Total rows : 14,740
  Subjects   : 39
  Bands      : ['alpha', 'beta', 'gamma', 'theta']
  Regions    : ['LHP', 'RHP']

  Predictor scales (raw):
    RSA_r        mean=0.088  sd=0.416  range=[-1.00, 1.00]
    T_lag        mean=4.489  sd=2.894  range=[1.00, 11.00]
    SP_lag       mean=70.740  sd=30.713  range=[12.95, 142.72]
    output_lag   mean=2.416  sd=1.535  range=[1.00, 10.00]

  Note: no z-scoring applied. Betas are in raw units.
  Between-subject differences absorbed by (1|subject) random intercept.

######################################################################
STEP 1 — Does RSA_r track T_lag / SP_lag at all?
######################################################################
  Data: all bands pooled, both hemispheres
  Covariate: output_lag (raw, not z-score

In [21]:
#!/usr/bin/env python3
"""
RSA Lag — Sequential LMM Analysis (4 Steps) — RAW SCALE, DBOY1 only
========================================================================
Identical logic to rsa_lag_sequential_lmm_raw.py but restricted to the
DBOY1 experiment only, so no experiment covariate is needed.

Betas are in original units:
  RSA_r      : Pearson r
  T_lag      : number of list positions separating the two items
  SP_lag     : Euclidean distance in store-coordinate units
  output_lag : number of recall output positions between the two items

Steps
-----
  Step 1 — RSA_r ~ predictor + output_lag + (1|subj)
            All bands pooled, both hemispheres.

  Step 2 — RSA_r ~ predictor * band (explicit dummies, theta = reference)
                 + output_lag + (1|subj)
            Tests whether predictor slope differs across bands.

  Step 3 — RSA_r ~ predictor * hemisphere + output_lag + (1|subj)
            Theta band only.  Tests LHP vs RHP asymmetry.
            Also run separately for LHP and RHP.

  Step 4 — RSA_r ~ T_lag + SP_lag + output_lag + (1|subj)
            Theta band only.  Tests independence of T_lag and SP_lag.

Input  : ./rsa_lag_hc_bands/ALL_SUBJECTS_DBOY1_hc_allbands_rsa_lag.csv
Output : ./rsa_lag_hc_bands/SEQUENTIAL_LMM_RAW_DBOY1_results.csv
         ./rsa_lag_hc_bands/SEQUENTIAL_LMM_RAW_DBOY1_results.txt
"""

import warnings
from pathlib import Path
from typing import List, Optional

import numpy as np
import pandas as pd
from statsmodels.regression.mixed_linear_model import MixedLM
from statsmodels.stats.multitest import fdrcorrection

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

INPUT_DIR     = Path('./rsa_lag_hc_bands')
OUTPUT_DIR    = Path('./rsa_lag_hc_bands')

BANDS         = ['theta', 'alpha', 'beta', 'gamma']
NON_REF_BANDS = ['alpha', 'beta', 'gamma']   # theta = reference

OUTCOME = 'RSA_r'


# ============================================================================
# DATA LOADING
# ============================================================================

def load_data() -> Optional[pd.DataFrame]:
    fpath = INPUT_DIR / "ALL_SUBJECTS_DBOY1_hc_allbands_rsa_lag.csv"
    if not fpath.exists():
        print(f"  ✗ File not found: {fpath}")
        return None

    df = pd.read_csv(fpath)
    print(f"  Loaded {fpath.name}  ({len(df):,} rows)")

    if 'hemisphere' not in df.columns:
        df['hemisphere'] = df['region'].map({'LHP': 'left', 'RHP': 'right'})

    df['hemisphere_dummy'] = (df['hemisphere'] == 'right').astype(int)

    print(f"\n  Total rows : {len(df):,}")
    print(f"  Subjects   : {df['subject'].nunique()}")
    print(f"  Sessions   : {df['session'].nunique()}")
    print(f"  Bands      : {sorted(df['band'].unique().tolist())}")
    print(f"  Regions    : {sorted(df['region'].unique().tolist())}")

    print(f"\n  Predictor scales (raw):")
    for col in ['RSA_r', 'T_lag', 'SP_lag', 'output_lag']:
        if col in df.columns:
            s = df[col].dropna()
            print(f"    {col:<12} mean={s.mean():.3f}  "
                  f"sd={s.std():.3f}  "
                  f"range=[{s.min():.2f}, {s.max():.2f}]")

    return df


# ============================================================================
# LMM FITTING
# ============================================================================

def fit_lmm(df: pd.DataFrame,
            formula_cols: List[str],
            formula_rhs: str,
            label: str) -> Optional[object]:
    """
    Fit RSA_r ~ formula_rhs.

    Random effects strategy (tried in order):
      1. Nested: (1|subject) + (1|subject:session)  via vc_formula
      2. Simple : (1|subject) only — fallback if nested hits a boundary
    """
    df = df.copy()
    df['subj_sess'] = df['subject'].astype(str) + '_' + df['session'].astype(str)

    keep = [OUTCOME] + [c for c in formula_cols if c in df.columns] \
           + ['subject', 'subj_sess']
    df   = df[keep].dropna()

    n = len(df)
    if n < 20:
        print(f"  [{label}] Too few rows ({n}) — skipping")
        return None

    formula = f"{OUTCOME} ~ {formula_rhs}"
    print(f"\n  [{label}] {formula}")
    print(f"  [{label}] N={n:,}  subjects={df['subject'].nunique()}")

    def _is_boundary(res) -> bool:
        if res is None or not np.isfinite(res.llf):
            return True
        for k in res.params.index:
            if 'subj_sess' in k.lower():
                if abs(res.params[k] - 16.0) < 1e-6:
                    return True
                if not np.isfinite(res.bse[k]):
                    return True
        return False

    # Attempt 1: nested random effects
    result = None
    for method in ['lbfgs', 'nm', 'powell']:
        try:
            model = MixedLM.from_formula(
                formula,
                data       = df,
                groups     = df['subject'],
                vc_formula = {'subj_sess': '0 + C(subj_sess)'},
            )
            res = model.fit(reml=True, method=method)
            if np.isfinite(res.llf) and not _is_boundary(res):
                print(f"  [{label}] nested RE  optimizer={method}  "
                      f"converged={getattr(res, 'converged', None)}")
                result = res
                break
        except Exception:
            pass

    # Attempt 2: subject-only random effects
    if result is None or _is_boundary(result):
        print(f"  [{label}] ⚠ Nested RE boundary — "
              f"falling back to (1|subject) only")
        for method in ['lbfgs', 'nm', 'powell']:
            try:
                model = MixedLM.from_formula(
                    formula,
                    data   = df,
                    groups = df['subject'],
                )
                res = model.fit(reml=True, method=method)
                if np.isfinite(res.llf):
                    print(f"  [{label}] subject-only RE  optimizer={method}  "
                          f"converged={getattr(res, 'converged', None)}")
                    result = res
                    break
            except Exception as e:
                print(f"  [{label}] subject-only {method} failed: {e}")
                result = None

    if result is None:
        print(f"  [{label}] WARNING: all fits unsuccessful")
    return result


# ============================================================================
# RESULT FORMATTING
# ============================================================================

def sig_stars(p: float) -> str:
    return ('***' if p < 0.001 else '**' if p < 0.01 else
            '*'   if p < 0.05  else '†'  if p < 0.10  else '')


def result_to_df(result, step: str, model_desc: str,
                 hemisphere: str = 'combined') -> pd.DataFrame:
    if result is None:
        return pd.DataFrame()

    rows = []
    for term in result.params.index:
        p_val = result.pvalues[term]
        is_re = (not np.isfinite(result.bse[term]) or
                 not np.isfinite(p_val) or
                 'var' in term.lower())
        rows.append({
            'step':       step,
            'model':      model_desc,
            'hemisphere': hemisphere,
            'term':       term,
            'beta':       result.params[term],
            'se':         result.bse[term],
            'z':          result.tvalues[term],
            'p_raw':      p_val,
            'p_fdr':      np.nan,
            'is_re':      is_re,
            'nobs':       int(result.nobs),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    fe_mask = (~df['is_re']) & df['p_raw'].notna() & np.isfinite(df['p_raw'])
    if fe_mask.sum() > 0:
        _, fdr_vals = fdrcorrection(df.loc[fe_mask, 'p_raw'].values)
        df.loc[fe_mask, 'p_fdr'] = fdr_vals

    return df


def print_table(df: pd.DataFrame, title: str):
    if df.empty:
        print(f"\n  [no results for: {title}]")
        return

    sep  = '=' * 95
    sep2 = '-' * 95
    hdr  = (f"{'Term':<45} {'β':>10} {'SE':>10} {'z':>8} "
            f"{'p_raw':>10} {'p_fdr':>10} {'N':>8}")

    print(f"\n{sep}")
    print(title)
    print(sep2)
    print(hdr)
    print(sep2)

    fe = df[~df['is_re']] if 'is_re' in df.columns else df
    for _, row in fe.iterrows():
        p_fdr = row.get('p_fdr', np.nan)
        p_fdr_str = f"{p_fdr:>10.4f}" if np.isfinite(p_fdr) else f"{'—':>10}"
        stars = sig_stars(p_fdr) if np.isfinite(p_fdr) else ''
        print(f"  {row['term']:<43} {row['beta']:>10.5f} {row['se']:>10.5f} "
              f"{row['z']:>8.3f} {row['p_raw']:>10.4f} "
              f"{p_fdr_str} {int(row['nobs']):>8,}  {stars}")

    re = df[df['is_re']] if 'is_re' in df.columns else pd.DataFrame()
    if not re.empty:
        print(f"  {'— random effects —'}")
        for _, row in re.iterrows():
            beta_str = f"{row['beta']:>10.5f}" if np.isfinite(row['beta']) \
                       else f"{'—':>10}"
            print(f"  {row['term']:<43} {beta_str}")

    print(sep2)
    print("  Outcome = RSA_r (raw Pearson r).  DBOY1 only — no experiment covariate.")
    print("  FDR: BH within fixed effects only.  "
          "† p<.10  * p<.05  ** p<.01  *** p<.001")
    print(f"  Reference: band=theta, hemisphere=LHP")
    print(sep)


# ============================================================================
# STEP 1
# ============================================================================

def step1(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 1 — Does RSA_r track T_lag / SP_lag at all?")
    print(f"{'#'*70}")
    print("  Data: all bands pooled, both hemispheres")

    results = []
    for pred in ['T_lag', 'SP_lag']:
        res = fit_lmm(
            df,
            formula_cols = [pred, 'output_lag'],
            formula_rhs  = f'{pred} + output_lag',
            label        = f'Step1 {pred}',
        )
        rdf = result_to_df(res, step='Step1',
                           model_desc=f'RSA_r ~ {pred} + output_lag')
        print_table(rdf,
                    f"Step 1 | RSA_r ~ {pred} + output_lag  [all bands, combined]")
        results.append(rdf)

    return results


# ============================================================================
# STEP 2
# ============================================================================

def step2(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 2 — Is the effect theta-specific?")
    print(f"{'#'*70}")
    print("  Data: all bands pooled, both hemispheres")
    print("  Reference band: theta")

    df = df.copy()
    for band in NON_REF_BANDS:
        df[f'band_{band}'] = (df['band'] == band).astype(float)

    results = []
    for pred in ['T_lag', 'SP_lag']:

        for band in NON_REF_BANDS:
            df[f'{pred}_x_{band}'] = df[pred] * df[f'band_{band}']

        band_cols  = [f'band_{b}'     for b in NON_REF_BANDS]
        inter_cols = [f'{pred}_x_{b}' for b in NON_REF_BANDS]

        formula_cols = [pred] + band_cols + inter_cols + ['output_lag']
        formula_rhs  = (f'{pred} + '
                        + ' + '.join(band_cols) + ' + '
                        + ' + '.join(inter_cols)
                        + ' + output_lag')

        res = fit_lmm(df, formula_cols, formula_rhs, label=f'Step2 {pred}')
        rdf = result_to_df(res, step='Step2',
                           model_desc=f'RSA_r ~ {pred} * band + output_lag')
        print_table(rdf,
                    f"Step 2 | RSA_r ~ {pred} * band + output_lag  "
                    f"[all bands, combined]\n"
                    f"  KEY: {pred}_x_{{band}} significant → slope ≠ theta")
        results.append(rdf)

    return results


# ============================================================================
# STEP 3
# ============================================================================

def step3(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 3 — Is the theta effect lateralized?  (theta band only)")
    print(f"{'#'*70}")

    theta_df = df[df['band'] == 'theta'].copy()
    print(f"  Theta rows: {len(theta_df):,}  "
          f"subjects: {theta_df['subject'].nunique()}")

    results = []
    for pred in ['T_lag', 'SP_lag']:
        inter_col = f'{pred}_x_hemi'
        theta_df[inter_col] = theta_df[pred] * theta_df['hemisphere_dummy']

        # Combined: interaction test
        res = fit_lmm(
            theta_df,
            formula_cols = [pred, 'hemisphere_dummy', inter_col, 'output_lag'],
            formula_rhs  = (f'{pred} + hemisphere_dummy + {inter_col}'
                            f' + output_lag'),
            label        = f'Step3 {pred} combined',
        )
        rdf = result_to_df(res, step='Step3',
                           model_desc=f'RSA_r ~ {pred} * hemisphere + output_lag',
                           hemisphere='combined')
        print_table(rdf,
                    f"Step 3 | RSA_r ~ {pred} * hemisphere + output_lag  "
                    f"[theta, combined]\n"
                    f"  KEY TERM: {inter_col}  "
                    f"(significant → LHP slope ≠ RHP slope)")
        results.append(rdf)

        # Simple slopes per hemisphere
        for hemi, hemi_label in [('left', 'LHP'), ('right', 'RHP')]:
            hd = theta_df[theta_df['hemisphere'] == hemi].copy()
            res_h = fit_lmm(
                hd,
                formula_cols = [pred, 'output_lag'],
                formula_rhs  = f'{pred} + output_lag',
                label        = f'Step3 {pred} {hemi_label}',
            )
            rdf_h = result_to_df(res_h, step='Step3',
                                 model_desc=f'RSA_r ~ {pred} + output_lag',
                                 hemisphere=hemi_label)
            print_table(rdf_h,
                        f"Step 3 | RSA_r ~ {pred} + output_lag  "
                        f"[theta, {hemi_label} only]")
            results.append(rdf_h)

    return results


# ============================================================================
# STEP 4
# ============================================================================

def step4(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 4 — Do T_lag and SP_lag survive mutual control?  (theta only)")
    print(f"{'#'*70}")

    theta_df = df[df['band'] == 'theta'].copy()
    print(f"  Theta rows: {len(theta_df):,}  "
          f"subjects: {theta_df['subject'].nunique()}")

    formula_cols = ['T_lag', 'SP_lag', 'output_lag']
    formula_rhs  = 'T_lag + SP_lag + output_lag'

    results = []
    for subset_label, subset_df in [
            ('combined', theta_df),
            ('LHP',      theta_df[theta_df['hemisphere'] == 'left']),
            ('RHP',      theta_df[theta_df['hemisphere'] == 'right'])]:

        res = fit_lmm(
            subset_df,
            formula_cols = formula_cols,
            formula_rhs  = formula_rhs,
            label        = f'Step4 {subset_label}',
        )
        rdf = result_to_df(res, step='Step4',
                           model_desc='RSA_r ~ T_lag + SP_lag + output_lag',
                           hemisphere=subset_label)
        print_table(rdf,
                    f"Step 4 | RSA_r ~ T_lag + SP_lag + output_lag  "
                    f"[theta, {subset_label}]")
        results.append(rdf)

    return results


# ============================================================================
# ENTRY POINT
# ============================================================================

def main():
    print(f"\n{'='*70}")
    print("RSA LAG — SEQUENTIAL LMM  |  DBOY1 only  |  RAW SCALE")
    print(f"{'='*70}")

    df = load_data()
    if df is None or df.empty:
        print("No data loaded. Exiting.")
        return

    all_results = []
    all_results += step1(df)
    all_results += step2(df)
    all_results += step3(df)
    all_results += step4(df)

    # Save CSV
    if all_results:
        master = pd.concat(all_results, ignore_index=True)
        csv_path = OUTPUT_DIR / 'SEQUENTIAL_LMM_RAW_DBOY1_results.csv'
        master.to_csv(csv_path, index=False)
        print(f"\n  ✓ Results → {csv_path}  ({len(master):,} rows)")

        # Save text
        txt_path = OUTPUT_DIR / 'SEQUENTIAL_LMM_RAW_DBOY1_results.txt'
        lines = []
        for rdf in all_results:
            if rdf.empty:
                continue
            lines.append(f"\n{'='*80}")
            lines.append(f"{rdf['step'].iloc[0]}  |  "
                         f"{rdf['model'].iloc[0]}  |  "
                         f"{rdf['hemisphere'].iloc[0]}")
            lines.append(f"{'='*80}")
            for _, row in rdf[~rdf['is_re']].iterrows():
                p_fdr = row['p_fdr']
                stars = sig_stars(p_fdr) if np.isfinite(p_fdr) else ''
                p_fdr_s = f"{p_fdr:.4f}" if np.isfinite(p_fdr) else '—'
                lines.append(
                    f"  {row['term']:<40} β={row['beta']:>10.5f}  "
                    f"SE={row['se']:>9.5f}  z={row['z']:>7.3f}  "
                    f"p={row['p_raw']:>8.4f}  p_fdr={p_fdr_s:>8}  {stars}"
                )
        with open(txt_path, 'w') as f:
            f.write('\n'.join(lines))
        print(f"  ✓ Text    → {txt_path}")

    print(f"\n{'='*70}")
    print("INTERPRETATION GUIDE")
    print(f"{'='*70}")
    print("""
  Step 1: T_lag/SP_lag significant controlling for output_lag?
          → genuine clustering effect, not recall-order artifact.

  Step 2: predictor_x_alpha/beta/gamma significant?
          → theta slope differs from that band → theta specificity confirmed.
          NOT significant → effect is broadband, not theta-specific.

  Step 3: predictor_x_hemi significant?
          → LHP and RHP slopes are genuinely different.
          Simple slopes show which hemisphere drives the effect.

  Step 4: Both T_lag and SP_lag survive together + output_lag?
          → temporal and spatial clustering are INDEPENDENT.
    """)


if __name__ == '__main__':
    main()


RSA LAG — SEQUENTIAL LMM  |  DBOY1 only  |  RAW SCALE
  Loaded ALL_SUBJECTS_DBOY1_hc_allbands_rsa_lag.csv  (13,028 rows)

  Total rows : 13,028
  Subjects   : 34
  Sessions   : 6
  Bands      : ['alpha', 'beta', 'gamma', 'theta']
  Regions    : ['LHP', 'RHP']

  Predictor scales (raw):
    RSA_r        mean=0.070  sd=0.394  range=[-1.00, 1.00]
    T_lag        mean=4.523  sd=2.914  range=[1.00, 11.00]
    SP_lag       mean=70.149  sd=30.696  range=[12.96, 142.72]
    output_lag   mean=2.450  sd=1.560  range=[1.00, 10.00]

######################################################################
STEP 1 — Does RSA_r track T_lag / SP_lag at all?
######################################################################
  Data: all bands pooled, both hemispheres

  [Step1 T_lag] RSA_r ~ T_lag + output_lag
  [Step1 T_lag] N=13,028  subjects=34
  [Step1 T_lag] nested RE  optimizer=nm  converged=True

Step 1 | RSA_r ~ T_lag + output_lag  [all bands, combined]
---------------------------------------

In [23]:
#!/usr/bin/env python3
"""
Visualization: RSA Lag Sequential LMM Results (DBOY1, raw scale)
=================================================================
Produces four figures, one per step:

  Fig 1 (Step 1) — Coefficient plot: T_lag and SP_lag slopes across all bands
  Fig 2 (Step 2) — Band factor plot: theta slope + interaction terms (Δ from theta)
  Fig 3 (Step 3) — Hemisphere plot: LHP vs RHP slopes in theta band
  Fig 4 (Step 4) — Independence plot: T_lag and SP_lag side-by-side in theta band

All figures use a clean, publication-ready style.
Input : ./rsa_lag_hc_bands/SEQUENTIAL_LMM_RAW_DBOY1_results.csv
Output: ./rsa_lag_hc_bands/plots/
"""

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

warnings.filterwarnings('ignore')

# ============================================================================
# PATHS
# ============================================================================

RESULTS_CSV = Path('./rsa_lag_hc_bands/SEQUENTIAL_LMM_RAW_DBOY1_results.csv')
PLOT_DIR    = Path('./rsa_lag_hc_bands/plots')
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================================
# STYLE
# ============================================================================

# Publication palette
C_THETA  = '#2166AC'   # blue
C_ALPHA  = '#74ADD1'   # light blue
C_BETA   = '#F46D43'   # orange
C_GAMMA  = '#D73027'   # red
C_LHP    = '#1A3A6B'   # navy
C_RHP    = '#8B1A1A'   # crimson
C_TLAG   = '#2B4B8C'   # dark blue
C_SPLAG  = '#7B2D8B'   # purple
C_GRAY   = '#888888'
C_LIGHT  = '#F5F5F5'
C_GRID   = '#E0E0E0'

BAND_COLORS  = {'theta': C_THETA, 'alpha': C_ALPHA,
                'beta': C_BETA,   'gamma': C_GAMMA}
PRED_COLORS  = {'T_lag': C_TLAG, 'SP_lag': C_SPLAG}
PRED_LABELS  = {'T_lag': 'Temporal lag (T_lag)',
                'SP_lag': 'Spatial lag (SP_lag)'}

plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         10,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.facecolor':    C_LIGHT,
    'axes.grid':         True,
    'axes.grid.axis':    'x',
    'grid.color':        C_GRID,
    'grid.linewidth':    0.8,
    'figure.facecolor':  'white',
    'figure.dpi':        150,
})


def sig_stars(p):
    if not np.isfinite(p):
        return ''
    return ('***' if p < 0.001 else '**' if p < 0.01 else
            '*'   if p < 0.05  else '†'  if p < 0.10  else '')


def _get(df, step, model_kw, hemi, term_kw):
    """Fetch a single row by keyword matching."""
    mask = (df['step'] == step) & \
           (df['model'].str.contains(model_kw, regex=False)) & \
           (df['hemisphere'] == hemi) & \
           (df['term'].str.contains(term_kw, regex=False)) & \
           (~df['is_re'])
    sub = df[mask]
    if sub.empty:
        return None
    return sub.iloc[0]


def error_bar(ax, y, row, color, marker='o', size=8, zorder=3, label=None):
    """Draw one errorbar (beta ± 1.96*SE) on a horizontal axes."""
    if row is None or not np.isfinite(row['beta']):
        return
    b, se = row['beta'], row['se']
    ci = 1.96 * se
    ax.errorbar(b, y, xerr=[[ci], [ci]],
                fmt=marker, color=color, ecolor=color,
                elinewidth=1.4, capsize=3.5, capthick=1.4,
                markersize=size, zorder=zorder,
                markerfacecolor=color if np.isfinite(row['p_fdr'])
                                and row['p_fdr'] < 0.05 else 'white',
                markeredgecolor=color, markeredgewidth=1.5,
                label=label)
    # significance annotation
    stars = sig_stars(row.get('p_fdr', np.nan))
    if stars:
        ax.text(b + ci + abs(b) * 0.05 + 0.001, y, stars,
                color=color, va='center', fontsize=9, fontweight='bold')


def style_coef_ax(ax, title, xlabel='β  (RSA_r units)'):
    ax.axvline(0, color='#555555', lw=1.0, ls='--', zorder=1)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.tick_params(axis='y', length=0)
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
        spine.set_edgecolor('#AAAAAA')


# ============================================================================
# FIGURE 1 — Step 1: overall T_lag and SP_lag slopes
# ============================================================================

def fig_step1(df):
    fig, ax = plt.subplots(figsize=(7, 3.2))

    rows_info = [
        ('T_lag',  'T_lag', 'combined', 'T_lag',  C_TLAG,  'o'),
        ('SP_lag', 'SP_lag','combined', 'SP_lag',  C_SPLAG, 's'),
    ]

    yticks, ylabels = [], []
    for y, (step_kw, model_kw, hemi, term_kw, color, marker) in \
            enumerate(rows_info[::-1]):
        row = _get(df, 'Step1', model_kw, hemi, term_kw)
        error_bar(ax, y, row, color, marker=marker, size=9)
        ylabels.append(PRED_LABELS[term_kw])
        yticks.append(y)

    ax.set_yticks(yticks)
    ax.set_yticklabels(ylabels, fontsize=10)
    style_coef_ax(ax,
        'Step 1 — Does RSA_r track clustering?\n'
        'All bands pooled, both hemispheres, controlling for output_lag')

    legend_handles = [
        Line2D([0],[0], marker='o', color='w',
               markerfacecolor=C_TLAG,  markersize=8, label='T_lag  (sig.)'),
        Line2D([0],[0], marker='o', color='w',
               markerfacecolor='white', markeredgecolor=C_TLAG,
               markersize=8, label='T_lag  (n.s.)'),
        Line2D([0],[0], marker='s', color='w',
               markerfacecolor='white', markeredgecolor=C_SPLAG,
               markersize=8, label='SP_lag  (n.s.)'),
    ]
    ax.legend(handles=legend_handles, loc='lower right',
              fontsize=8.5, framealpha=0.9)

    # Annotation
    ax.text(0.02, 0.08,
            'Filled = p_fdr < 0.05  |  Open = n.s.\n'
            'Error bars = 95% CI',
            transform=ax.transAxes, fontsize=7.5,
            color=C_GRAY, style='italic')

    fig.tight_layout()
    path = PLOT_DIR / 'fig1_step1_overall.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ Fig 1 → {path.name}")


# ============================================================================
# FIGURE 2 — Step 2: T_lag theta slope + band interactions
# ============================================================================

def fig_step2(df):
    """
    Two panels (T_lag / SP_lag).
    Left section of each panel: theta slope (main effect of predictor).
    Right section: interaction terms — difference of each band's slope from theta.
    """
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    BANDS    = ['theta', 'alpha', 'beta', 'gamma']
    NON_REF  = ['alpha', 'beta', 'gamma']

    for ax, pred in zip(axes, ['T_lag', 'SP_lag']):
        color  = PRED_COLORS[pred]
        yticks, ylabels = [], []
        y = 0

        # ── Theta slope (main effect) ─────────────────────────────────────────
        row = _get(df, 'Step2', f'{pred} * band', 'combined', f'^{pred}$')
        if row is None:
            # fallback: match just pred in term
            row = _get(df, 'Step2', f'{pred} * band', 'combined', pred)
        error_bar(ax, y, row, BAND_COLORS['theta'], marker='o', size=9)
        yticks.append(y);  ylabels.append('Theta slope\n(reference)')
        y -= 1.4

        # Separator
        ax.axhline(y + 0.7, color='#CCCCCC', lw=0.8, ls=':')

        # ── Interaction terms (Δ from theta) ──────────────────────────────────
        for band in NON_REF:
            inter_term = f'{pred}_x_{band}'
            row_int = _get(df, 'Step2', f'{pred} * band', 'combined', inter_term)
            error_bar(ax, y, row_int,
                      BAND_COLORS[band], marker='s', size=8)
            yticks.append(y)
            ylabels.append(f'{band.capitalize()} − Theta\n(interaction)')
            y -= 1.2

        ax.set_yticks(yticks)
        ax.set_yticklabels(ylabels, fontsize=9)
        style_coef_ax(ax,
            f'Step 2 — Is the {pred} effect theta-specific?\n'
            f'All bands pooled  |  theta = reference')

        # Band color legend
        handles = [
            Line2D([0],[0], marker='o', color='w',
                   markerfacecolor=BAND_COLORS[b],
                   markersize=8, label=b.capitalize())
            for b in BANDS
        ]
        ax.legend(handles=handles, loc='lower right',
                  fontsize=8, framealpha=0.9)

        # Label sections
        ax.text(0.02, 0.96, 'Reference slope', transform=ax.transAxes,
                fontsize=7.5, color='#555555', style='italic', va='top')
        ax.text(0.02, 0.68, 'Δ from theta\n(not sig. → broadband)',
                transform=ax.transAxes,
                fontsize=7.5, color='#555555', style='italic', va='top')

    fig.suptitle('Step 2 — Band specificity test\n'
                 'Interaction terms: is that band\'s slope ≠ theta slope?',
                 fontsize=11, fontweight='bold', y=1.02)
    fig.tight_layout()
    path = PLOT_DIR / 'fig2_step2_band_specificity.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ Fig 2 → {path.name}")


# ============================================================================
# FIGURE 3 — Step 3: lateralization (theta band)
# ============================================================================

def fig_step3(df):
    """
    Two panels (T_lag / SP_lag).
    Each panel shows three rows: combined + interaction, LHP simple slope, RHP simple slope.
    """
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    for ax, pred in zip(axes, ['T_lag', 'SP_lag']):
        yticks, ylabels = [], []
        y = 0

        # Combined: main predictor slope
        row_comb = _get(df, 'Step3',
                        f'{pred} * hemisphere', 'combined', f'{pred}$')
        if row_comb is None:
            row_comb = _get(df, 'Step3',
                            f'{pred} * hemisphere', 'combined', pred)
        error_bar(ax, y, row_comb, '#444444', marker='D', size=9,
                  label='Combined (both hemi)')
        yticks.append(y);  ylabels.append('Combined\n(LHP slope)')
        y -= 1.0

        # Combined: interaction term
        inter_term = f'{pred}_x_hemi'
        row_int = _get(df, 'Step3',
                       f'{pred} * hemisphere', 'combined', inter_term)
        error_bar(ax, y, row_int, '#999999', marker='D', size=7)
        yticks.append(y);  ylabels.append('Interaction\n(RHP − LHP)')
        y -= 1.4

        # Separator
        ax.axhline(y + 0.7, color='#CCCCCC', lw=0.8, ls=':')

        # LHP simple slope
        row_lhp = _get(df, 'Step3', f'{pred} + output_lag', 'LHP', pred)
        error_bar(ax, y, row_lhp, C_LHP, marker='o', size=9,
                  label='LHP')
        yticks.append(y);  ylabels.append('LHP\n(simple slope)')
        y -= 1.2

        # RHP simple slope
        row_rhp = _get(df, 'Step3', f'{pred} + output_lag', 'RHP', pred)
        error_bar(ax, y, row_rhp, C_RHP, marker='s', size=9,
                  label='RHP')
        yticks.append(y);  ylabels.append('RHP\n(simple slope)')

        ax.set_yticks(yticks)
        ax.set_yticklabels(ylabels, fontsize=9)
        style_coef_ax(ax,
            f'Step 3 — Is the {pred} effect lateralized?\n'
            f'Theta band only  |  interaction tests LHP ≠ RHP')

        handles = [
            Line2D([0],[0], marker='D', color='w',
                   markerfacecolor='#444444', markersize=8, label='Combined'),
            Line2D([0],[0], marker='o', color='w',
                   markerfacecolor=C_LHP, markersize=8, label='LHP'),
            Line2D([0],[0], marker='s', color='w',
                   markerfacecolor=C_RHP, markersize=8, label='RHP'),
        ]
        ax.legend(handles=handles, loc='lower right',
                  fontsize=8, framealpha=0.9)

    fig.suptitle('Step 3 — Lateralization test (theta band)\n'
                 'Interaction term tests LHP slope ≠ RHP slope',
                 fontsize=11, fontweight='bold', y=1.02)
    fig.tight_layout()
    path = PLOT_DIR / 'fig3_step3_lateralization.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ Fig 3 → {path.name}")


# ============================================================================
# FIGURE 4 — Step 4: T_lag and SP_lag mutual control (theta)
# ============================================================================

def fig_step4(df):
    """
    Three panels: combined, LHP, RHP.
    Each panel: T_lag and SP_lag side-by-side in the same model.
    """
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
    subsets = ['combined', 'LHP', 'RHP']
    titles  = ['Combined\n(both hemispheres)',
               'LHP only\n(left hippocampus)',
               'RHP only\n(right hippocampus)']

    for ax, subset, title in zip(axes, subsets, titles):
        yticks, ylabels = [], []

        for y, (pred, color, marker) in enumerate([
                ('SP_lag', C_SPLAG, 's'),
                ('T_lag',  C_TLAG,  'o')]):
            row = _get(df, 'Step4', 'T_lag + SP_lag', subset, pred)
            error_bar(ax, y, row, color, marker=marker, size=9)
            yticks.append(y)
            ylabels.append(PRED_LABELS[pred])

        ax.set_yticks(yticks)
        if ax == axes[0]:
            ax.set_yticklabels(ylabels, fontsize=10)
        style_coef_ax(ax,
            f'Step 4 — {title}')

        # Add beta annotation next to each point
        for y, pred in enumerate(['SP_lag', 'T_lag']):
            row = _get(df, 'Step4', 'T_lag + SP_lag', subset, pred)
            if row is not None and np.isfinite(row['beta']):
                p_fdr = row.get('p_fdr', np.nan)
                stars = sig_stars(p_fdr)
                label = f"β={row['beta']:.4f} {stars}".strip()
                ax.text(row['beta'] - 1.96 * row['se'] - 0.002, y,
                        label, ha='right', va='center',
                        fontsize=7.5, color=PRED_COLORS[pred])

    handles = [
        Line2D([0],[0], marker='o', color='w',
               markerfacecolor=C_TLAG,  markersize=8, label='T_lag'),
        Line2D([0],[0], marker='s', color='w',
               markerfacecolor=C_SPLAG, markersize=8, label='SP_lag'),
    ]
    axes[-1].legend(handles=handles, loc='lower right',
                    fontsize=9, framealpha=0.9)

    fig.suptitle('Step 4 — Do T_lag and SP_lag survive mutual control?\n'
                 'Theta band only  |  both in the same model + output_lag',
                 fontsize=11, fontweight='bold', y=1.02)
    fig.tight_layout()
    path = PLOT_DIR / 'fig4_step4_independence.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ Fig 4 → {path.name}")


# ============================================================================
# FIGURE 5 — Summary: all key effects in one panel
# ============================================================================

def fig_summary(df):
    """
    Single compact figure showing the key T_lag slopes across all contexts:
      - All bands combined (Step 1)
      - Theta band: combined, LHP, RHP (Step 3 simple slopes)
      - Theta band: T_lag controlling for SP_lag (Step 4 combined)
    Color = hemisphere; shape = model context.
    """
    fig, ax = plt.subplots(figsize=(8, 5))

    entries = [
        # (label, step, model_kw, hemi, term, color, marker, y)
        ('All bands, both hemi\n(Step 1)',
         'Step1', 'T_lag + output_lag', 'combined', 'T_lag',
         '#444444', 'D', 6),
        ('All bands, both hemi\n+ SP_lag control (Step 4→1)',
         'Step4', 'T_lag + SP_lag', 'combined', 'T_lag',
         '#666666', 'D', 5),
        ('Theta, both hemi\n(Step 3 combined)',
         'Step3', 'T_lag * hemisphere', 'combined', 'T_lag',
         '#444444', 'o', 3.5),
        ('Theta, LHP\n(Step 3 simple slope)',
         'Step3', 'T_lag + output_lag', 'LHP', 'T_lag',
         C_LHP, 'o', 2.5),
        ('Theta, RHP\n(Step 3 simple slope)',
         'Step3', 'T_lag + output_lag', 'RHP', 'T_lag',
         C_RHP, 's', 1.5),
        ('Theta, LHP\n+ SP_lag control (Step 4)',
         'Step4', 'T_lag + SP_lag', 'LHP', 'T_lag',
         C_LHP, '^', 0.5),
    ]

    yticks, ylabels = [], []
    for label, step, model_kw, hemi, term, color, marker, y in entries:
        row = _get(df, step, model_kw, hemi, term)
        error_bar(ax, y, row, color, marker=marker, size=9)
        yticks.append(y)
        ylabels.append(label)

    # Separators
    ax.axhline(4.8, color='#CCCCCC', lw=0.8, ls=':')
    ax.axhline(3.0, color='#CCCCCC', lw=0.8, ls=':')
    ax.text(0.01, 0.97, 'All bands', transform=ax.transAxes,
            fontsize=8, color='#777777', style='italic', va='top')
    ax.text(0.01, 0.67, 'Theta band', transform=ax.transAxes,
            fontsize=8, color='#777777', style='italic', va='top')

    ax.set_yticks(yticks)
    ax.set_yticklabels(ylabels, fontsize=9)
    style_coef_ax(ax,
        'Summary — T_lag → RSA_r across analysis contexts\n'
        'DBOY1  |  raw scale  |  filled = p_fdr < 0.05')

    handles = [
        Line2D([0],[0], marker='D', color='w',
               markerfacecolor='#444444', markersize=8, label='Combined'),
        Line2D([0],[0], marker='o', color='w',
               markerfacecolor=C_LHP, markersize=8, label='LHP'),
        Line2D([0],[0], marker='s', color='w',
               markerfacecolor=C_RHP, markersize=8, label='RHP'),
        Line2D([0],[0], marker='o', color='w',
               markerfacecolor='white', markeredgecolor='#444444',
               markersize=8, label='n.s.  (open)'),
        Line2D([0],[0], marker='o', color='w',
               markerfacecolor='#444444',
               markersize=8, label='p_fdr < 0.05  (filled)'),
    ]
    ax.legend(handles=handles, loc='lower right',
              fontsize=8, framealpha=0.9)

    fig.tight_layout()
    path = PLOT_DIR / 'fig5_summary_T_lag.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ Fig 5 → {path.name}")


# ============================================================================
# ENTRY POINT
# ============================================================================

def main():
    print(f"\n{'='*60}")
    print("RSA LAG — VISUALIZATION")
    print(f"{'='*60}")

    if not RESULTS_CSV.exists():
        print(f"  ✗ Results file not found: {RESULTS_CSV}")
        print("  Run rsa_lag_sequential_lmm_raw.py first.")
        return

    df = pd.read_csv(RESULTS_CSV)
    # is_re may be stored as object — ensure bool
    df['is_re'] = df['is_re'].astype(bool)

    print(f"  Loaded {len(df):,} result rows")
    print(f"  Steps: {df['step'].unique().tolist()}")
    print(f"\n  Generating figures …")

    fig_step1(df)
    fig_step2(df)
    fig_step3(df)
    fig_step4(df)
    fig_summary(df)

    print(f"\n  ✓ All figures saved to {PLOT_DIR}/")
    print(f"{'='*60}")


if __name__ == '__main__':
    main()


RSA LAG — VISUALIZATION
  Loaded 71 result rows
  Steps: ['Step1', 'Step2', 'Step3', 'Step4']

  Generating figures …
  ✓ Fig 1 → fig1_step1_overall.png
  ✓ Fig 2 → fig2_step2_band_specificity.png
  ✓ Fig 3 → fig3_step3_lateralization.png
  ✓ Fig 4 → fig4_step4_independence.png
  ✓ Fig 5 → fig5_summary_T_lag.png

  ✓ All figures saved to rsa_lag_hc_bands/plots/
